# Figure1 

## Overlay 2D keypoint trajectories on the first frame of a time interval


In [ ]:
"""
Data loading for 2D keypoint overlay visualization.
Run this cell once, then iterate on the visualization cell below.
"""

from pathlib import Path
import csv
import json
from typing import List, Dict, Tuple, Optional
import numpy as np
import cv2

# =====================================================================
# DATA CONFIG
# =====================================================================

# Directories
# =============================================
# # walking videos 
# VIDEO_DIR = Path("/mnt/lemebel/happyhouse_102025/session6/2025_10_12_15_06_46/videos/")
# CSV_DIR   = Path("/mnt/lemebel/happyhouse_102025/session6/2025_10_12_15_06_46/videos/predictions/2025_12_12_15_05_35/")
# FIG_DIR   = Path("/mnt/lemebel/happyhouse_102025/session6/2025_10_12_15_06_46/figures")
# FRAME_START = 137600
# FRAME_END   = 138400

# =============================================
# # courtship videos
# VIDEO_DIR = Path("/home/user/red_data/courtship_videos/2025_10_20_13_20_04/videos/")
# CSV_DIR   = Path("/home/user/red_data/courtship_videos/2025_10_20_13_20_04/videos/labeled_data/2026_02_13_10_49_27/")
# FIG_DIR   = Path("/home/user/red_data/courtship_videos/2025_10_20_13_20_04/videos/figures")
# FRAME_START = 149573
# FRAME_END   = 149588

# # Staircase videos
# VIDEO_DIR = Path("/home/user/red_data/stairs_videos/2026_01_12_14_07_16/")
# CSV_DIR   = Path("/home/user/red_data/stairs_videos/2026_01_12_14_07_16/labeled_data/2026_02_15_18_03_13/")
# FIG_DIR   = Path("/home/user/red_data/stairs_videos/2026_01_12_14_07_16/figures")
# FRAME_START = 24209
# FRAME_END   = 24321

# headless BDN2 videos 
VIDEO_DIR = Path("/media/user/LaCie2/Session7/2025_10_13_18_22_50/")
CSV_DIR   = Path("/media/user/LaCie2/Session7/2025_10_13_18_22_50/labeled_data/2026_04_15_12_11_09/")
FIG_DIR   = Path("/home/user/3D_tracking_paper/output/BDN2 headless/")
FRAME_START = 350

FRAME_END   = 3700

# Camera selection — set to a camera stem (e.g. "Cam2012862") or None for first found
CAMERA_NAME = 'Cam2012861'

# Keypoints to plot (for trajectory overlay)
KP_NAMES = ["T1L_TaTip"]

# =====================================================================
# LOADING FUNCTIONS
# =====================================================================

def load_skeleton_from_csv_header(csv_path: Path):
    with csv_path.open("r", encoding="utf-8") as f:
        first_line = f.readline().strip()
    skel_path = Path(first_line).expanduser()
    if not skel_path.is_file():
        skel_path = (csv_path.parent / first_line).resolve()
        if not skel_path.is_file():
            print(f"[WARNING] Skeleton JSON not found: '{first_line}'")
            return None, None, None
    try:
        with skel_path.open("r", encoding="utf-8") as f:
            skel = json.load(f)
        names = skel.get("node_names", None)
        edges = skel.get("edges", None)
        return (skel_path,
                names if isinstance(names, list) else None,
                edges if isinstance(edges, list) else None)
    except Exception as e:
        print(f"[WARNING] Failed to load skeleton: {e}")
        return skel_path, None, None


def load_keypoints2d(csv_path: Path, not_assigned_threshold: float = 1e6):
    skel_path, node_names, edges = load_skeleton_from_csv_header(csv_path)
    with csv_path.open("r", encoding="utf-8") as f:
        lines = f.read().splitlines()
    if len(lines) < 2:
        raise ValueError(f"{csv_path} has no data rows.")
    first_data_row = next(csv.reader([lines[1]]))
    n_fields = len(first_data_row)
    if (n_fields - 1) % 3 != 0:
        raise ValueError(f"Unexpected column count ({n_fields})")
    n_kp = (n_fields - 1) // 3

    frames, all_pts = [], []
    for raw in lines[1:]:
        if not raw.strip():
            continue
        row = next(csv.reader([raw]))
        fid = int(row[0])
        frames.append(fid)
        kp = np.full((n_kp, 2), np.nan)
        for i in range(n_kp):
            base = 1 + 3 * i
            kid, x, y = int(row[base]), float(row[base+1]), float(row[base+2])
            if abs(x) >= not_assigned_threshold or abs(y) >= not_assigned_threshold:
                x, y = np.nan, np.nan
            if 0 <= kid < n_kp:
                kp[kid] = [x, y]
        all_pts.append(kp)

    return np.array(frames), np.stack(all_pts), skel_path, node_names, edges


def resolve_keypoint_indices(node_names, kp_names):
    idxs, labels = [], []
    for name in (kp_names or []):
        idx = node_names.index(name)
        if idx not in idxs:
            idxs.append(idx)
            labels.append(name)
    return idxs, labels


def get_frame_mask(frames, frame_start, frame_end):
    mask = (frames >= frame_start) & (frames <= frame_end)
    if not np.any(mask):
        raise ValueError(
            f"No frames in [{frame_start}, {frame_end}]. "
            f"Available: {frames.min()}–{frames.max()}"
        )
    return mask


def read_frame(video_path: Path, frame_index: int):
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise RuntimeError(f"Could not open video: {video_path}")
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_index)
    ok, bgr = cap.read()
    cap.release()
    if not ok or bgr is None:
        raise RuntimeError(f"Could not read frame {frame_index}")
    return cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)


def find_camera_pairs(video_dir: Path, csv_dir: Path):
    vexts = {".avi", ".mp4", ".mov", ".mkv"}
    vmap = {p.stem: p for p in video_dir.iterdir() if p.is_file() and p.suffix.lower() in vexts}
    cmap = {p.stem: p for p in csv_dir.iterdir() if p.is_file() and p.suffix.lower() == ".csv"}
    return [(s, vmap[s], cmap[s]) for s in sorted(set(vmap) & set(cmap))]


# =====================================================================
# LOAD DATA
# =====================================================================

pairs = find_camera_pairs(VIDEO_DIR, CSV_DIR)
if not pairs:
    raise RuntimeError("No camera/video pairs found — check VIDEO_DIR and CSV_DIR.")

if CAMERA_NAME:
    match = [(n, v, c) for n, v, c in pairs if n == CAMERA_NAME]
    if not match:
        raise ValueError(f"'{CAMERA_NAME}' not found. Available: {[n for n,_,_ in pairs]}")
    cam_name, video_path, csv_path = match[0]
else:
    cam_name, video_path, csv_path = pairs[0]
print(f"Camera: {cam_name}")

frames, points, _, node_names, edges = load_keypoints2d(csv_path)
kp_idxs, kp_labels = resolve_keypoint_indices(node_names, KP_NAMES)
mask = get_frame_mask(frames, FRAME_START, FRAME_END)
n_masked = int(np.sum(mask))
print(f"  Frames {FRAME_START}–{FRAME_END}  ({n_masked} frames)")

frame_rgb = read_frame(video_path, FRAME_START)
img_h, img_w = frame_rgb.shape[:2]
print(f"  Image: {img_w}×{img_h}")

In [ ]:
"""
Visualization: overlay 2D trajectories + skeleton on camera frame.
Re-run this cell to iterate — data is loaded in the cell above.
Saves SVG with each skeleton as a grouped object for Inkscape editing.
"""

from matplotlib.collections import LineCollection
from matplotlib.colors import LinearSegmentedColormap, Normalize
import matplotlib as mpl
import matplotlib.cm as cm
import matplotlib.pyplot as plt
import xml.etree.ElementTree as ET

# =====================================================================
# VISUALIZATION CONFIG 
# =====================================================================

USE_TRIPOD_COLORS = True
POINTS_ONLY = False           # True = scatter dots, False = connected lines

# Skeleton overlay
SHOW_SKELETON_OVERLAY = False
SKELETON_EVERY_N_FRAMES =500    # draw skeleton every N frames within bout
SKELETON_JOINT_SIZE = 2.5        # joint marker diameter (points)
SKELETON_LINE_WIDTH = 1.2        # edge line width
SKELETON_ALPHA_START = 0.9       # transparency of oldest skeleton pose
SKELETON_ALPHA_END   = 0.9       # transparency of newest skeleton pose

# Body-part colors — easily editable dict
SKELETON_COLORS = {
    "body":  "#510455ED",  # gray
    "WingL": "#58B0FD",  # light blue
    "WingR": "#6B79D4",  # salmon
    "T1L":   "#0CE2DB",  # blue
    "T1R":   "#08CE0CC8",  # purple
    "T2L":   "#6FE60D",  # red
    "T2R":   "#E2BA1B",  # green
    "T3L":   "#F94B06",  # orange
    "T3R":   "#D10E0E",  # brown
}

# =====================================================================
# DRAWING FUNCTIONS
# =====================================================================

CMAP_tri1 = LinearSegmentedColormap.from_list("left",  ["#2a6d447a", "#2a6d447a"])
CMAP_tri2 = LinearSegmentedColormap.from_list("right", ["#E29911B6", "#E29911B6"])

def get_side_cmap(kp_name: str):
    if "T1L" in kp_name or "T3L" in kp_name or "T2R" in kp_name:
        return CMAP_tri1
    else:
        return CMAP_tri2


def classify_body_part(name: str) -> str:
    """Map a node name to a body-part key for coloring."""
    for prefix in ("T1L", "T1R", "T2L", "T2R", "T3L", "T3R"):
        if name.startswith(prefix):
            return prefix
    if name.startswith("WingL"):
        return "WingL"
    if name.startswith("WingR"):
        return "WingR"
    return "body"


def draw_skeleton_2d(ax, points_2d, edges, node_names, colors_dict,
                     alpha, lw, joint_size, img_h, gid_prefix=None):
    """Draw one skeleton frame using LineCollection + scatter.

    Each skeleton gets two artists (edges + joints) tagged with gid_prefix
    so they can be grouped in the SVG for Inkscape.
    """
    pts = points_2d.copy().astype(float)
    pts[:, 1] = img_h - 1 - pts[:, 1]

    # Edges as LineCollection (single SVG element)
    segments, edge_colors = [], []
    for i, j in edges:
        if i >= len(pts) or j >= len(pts):
            continue
        x0, y0 = pts[i]
        x1, y1 = pts[j]
        if np.isnan(x0) or np.isnan(y0) or np.isnan(x1) or np.isnan(y1):
            continue
        segments.append([(x0, y0), (x1, y1)])
        edge_colors.append(colors_dict.get(classify_body_part(node_names[i]), "#888888"))

    if segments:
        lc = LineCollection(segments, colors=edge_colors, linewidths=lw,
                            alpha=alpha, capstyle="round")
        if gid_prefix:
            lc.set_gid(f"{gid_prefix}_edges")
        ax.add_collection(lc)

    # Joints as scatter (single SVG element)
    jx, jy, jcolors = [], [], []
    for k in range(len(pts)):
        xk, yk = pts[k]
        if np.isnan(xk) or np.isnan(yk):
            continue
        jx.append(xk)
        jy.append(yk)
        jcolors.append(colors_dict.get(classify_body_part(node_names[k]), "#888888"))

    if jx:
        sc = ax.scatter(jx, jy, s=joint_size**2, c=jcolors, alpha=alpha, zorder=5, edgecolors='none')
        if gid_prefix:
            sc.set_gid(f"{gid_prefix}_joints")


def plot_trajectory(ax, x, y, *, label, cmap, lw=0.5, marker_size=2.5,
                    points_only=False):
    valid = ~np.isnan(x) & ~np.isnan(y)
    x, y = np.asarray(x)[valid], np.asarray(y)[valid]

    if len(x) < 2:
        return

    color = cmap(0.5)

    if points_only:
        ax.scatter(x, y, s=marker_size**2, c=[color] * len(x), edgecolors='none',
                   label=label, zorder=4)
    else:
        ax.plot(x, y, color=color, lw=lw, solid_capstyle='round',
                label=label, zorder=4)


def group_skeletons_in_svg(svg_path, frame_ids):
    """Post-process SVG to wrap each skeleton's elements in a <g> group.

    Each skeleton pose becomes a single selectable/deletable group in Inkscape,
    labeled 'Skeleton frame {fid}' in the Objects panel.
    """
    SVG_NS = 'http://www.w3.org/2000/svg'
    INKSCAPE_NS = 'http://www.inkscape.org/namespaces/inkscape'
    ET.register_namespace('', SVG_NS)
    ET.register_namespace('xlink', 'http://www.w3.org/1999/xlink')
    ET.register_namespace('inkscape', INKSCAPE_NS)

    tree = ET.parse(svg_path)
    root = tree.getroot()

    # Build parent map
    parent_map = {c: p for p in root.iter() for c in p}

    for fid in frame_ids:
        prefix = f"skel_{fid}_"
        targets = [el for el in root.iter() if (el.get('id') or '').startswith(prefix)]
        if not targets:
            continue

        # Create wrapping <g> with Inkscape label
        group = ET.Element(f'{{{SVG_NS}}}g')
        group.set('id', f'skeleton_frame_{fid}')
        group.set(f'{{{INKSCAPE_NS}}}label', f'Skeleton frame {fid}')

        # Insert group where first target lives
        first_parent = parent_map[targets[0]]
        idx = list(first_parent).index(targets[0])
        first_parent.insert(idx, group)

        # Move targets into group
        for t in targets:
            parent_map[t].remove(t)
            group.append(t)

    tree.write(str(svg_path), xml_declaration=True)


# =====================================================================
# FIGURE
# =====================================================================

# Recompute mask against the current frames array (guards against other
# cells overwriting `frames`/`points` between runs).
mask = (frames >= FRAME_START) & (frames <= FRAME_END)

fig, ax = plt.subplots(figsize=(10, 10))
ax.imshow(frame_rgb, origin="upper")
ax.axis("off")

# Trajectories
for idx, label in zip(kp_idxs, kp_labels):
    xy = points[mask, idx, :]
    x, y_raw = xy[:, 0], xy[:, 1]
    y_img = img_h - 1 - y_raw

    cmap_kp = get_side_cmap(label) if USE_TRIPOD_COLORS else mpl.cm.viridis

    plot_trajectory(ax, x, y_img, label=label, cmap=cmap_kp,
                    lw=1.5, points_only=POINTS_ONLY)

# Skeleton overlay
skeleton_frame_ids = []
if SHOW_SKELETON_OVERLAY and edges is not None and node_names is not None:
    interval_frames = frames[mask]
    skeleton_frame_ids = interval_frames[::SKELETON_EVERY_N_FRAMES]
    print(f"  Drawing {len(skeleton_frame_ids)} skeleton poses "
          f"(every {SKELETON_EVERY_N_FRAMES} frames)")

    for i, fid in enumerate(skeleton_frame_ids):
        row_idx = np.where(frames == fid)[0][0]
        pts_2d = points[row_idx]
        t = i / max(len(skeleton_frame_ids) - 1, 1)
        alpha = SKELETON_ALPHA_START + (SKELETON_ALPHA_END - SKELETON_ALPHA_START) * t
        draw_skeleton_2d(ax, pts_2d, edges, node_names, SKELETON_COLORS,
                         alpha, SKELETON_LINE_WIDTH, SKELETON_JOINT_SIZE, img_h,
                         gid_prefix=f"skel_{fid}")

ax.set_xlim(0, img_w)
ax.set_ylim(img_h, 0)

# Colorbars
norm = Normalize(vmin=FRAME_START, vmax=FRAME_END)
cbar_kw = dict(fraction=0.022, pad=0.015, aspect=30)

sides = set()
for name in kp_labels:
    pfx = name.split("_")[0]
    if "L" in pfx:
        sides.add("left")
    elif "R" in pfx:
        sides.add("right")

if USE_TRIPOD_COLORS:
    if "left" in sides:
        sm = cm.ScalarMappable(cmap=CMAP_tri1, norm=norm)
        cb = fig.colorbar(sm, ax=ax, location="left", **cbar_kw)
        cb.set_label("Left leg — Frame", fontsize=9)
        cb.ax.tick_params(labelsize=8)
    if "right" in sides:
        sm = cm.ScalarMappable(cmap=CMAP_tri2, norm=norm)
        cb = fig.colorbar(sm, ax=ax, location="right", **cbar_kw)
        cb.set_label("Right leg — Frame", fontsize=9)
        cb.ax.tick_params(labelsize=8)
else:
    sm = cm.ScalarMappable(cmap="viridis", norm=norm)
    cb = fig.colorbar(sm, ax=ax, **cbar_kw)
    cb.set_label("Frame", fontsize=9)
    cb.ax.tick_params(labelsize=8)

# Title & legend
ax.set_title(f"{cam_name}  |  Frames {FRAME_START}–{FRAME_END}",
             fontsize=12, fontweight="bold", pad=10)
ax.legend(title="Keypoints", loc="upper left", fontsize=9,
          title_fontsize=10, facecolor="white", edgecolor="gray", framealpha=0.85)

fig.tight_layout()
FIG_DIR.mkdir(parents=True, exist_ok=True)
svg_path = FIG_DIR / f"{cam_name}.svg"
fig.savefig(svg_path, dpi=300, bbox_inches="tight")

# Group skeletons in SVG for Inkscape
if SHOW_SKELETON_OVERLAY and len(skeleton_frame_ids) > 0:
    group_skeletons_in_svg(svg_path, skeleton_frame_ids)
    print(f"  Grouped {len(skeleton_frame_ids)} skeletons in {svg_path.name}")

plt.show()

In [ ]:
"""
Standalone XY trajectory plot from data3D.csv — colored by time.
Optional confidence subplot.
"""

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
import matplotlib.cm as cm

# =====================================================================
# CONFIG — edit these
# =====================================================================

DATA3D_PATH = Path(
    "/home/user/src/JARVIS-HybridNet/projects/fly50_V5/predictions/"
    "predictions3D/Predictions_3D_20260415-120633/data3D.csv"
)

FRAME_START = 0
FRAME_END   = 3800 

KP_NAMES = ["T1R_TaTip"]

MARKER_SIZE = 5.0        # scatter dot size (points)
CMAP_NAME   = "viridis"  # any matplotlib colormap
SHOW_CONFIDENCE = True   # add a confidence subplot below

# =====================================================================
# LOAD
# =====================================================================

def _load_data3d(path, frame_start, frame_end):
    """Load x, y, z, confidence for all keypoints in the frame range.

    data3D.csv has two header rows:
      row 0: keypoint names repeated 4× each
      row 1: x, y, z, confidence (repeated)
    Data rows are 0-indexed (row 0 = frame 0).
    """
    header = pd.read_csv(path, nrows=0, header=[0, 1])
    node_names = list(dict.fromkeys(header.columns.get_level_values(0)))
    n_kp = len(node_names)

    # Read only the rows we need
    # skiprows: skip header row 0,1 then skip frames before frame_start
    skip = list(range(2, 2 + frame_start))  # keep rows 0-1 (headers)
    n_frames = frame_end - frame_start + 1
    df = pd.read_csv(path, header=[0, 1], skiprows=skip, nrows=n_frames)

    frames = np.arange(frame_start, frame_start + len(df))
    points = np.full((len(df), n_kp, 3), np.nan)
    confidence = np.full((len(df), n_kp), np.nan)

    for i, name in enumerate(node_names):
        sub = df[name]
        points[:, i, 0] = pd.to_numeric(sub["x"], errors="coerce").values
        points[:, i, 1] = pd.to_numeric(sub["y"], errors="coerce").values
        points[:, i, 2] = pd.to_numeric(sub["z"], errors="coerce").values
        confidence[:, i] = pd.to_numeric(sub["confidence"], errors="coerce").values

    return frames, points, confidence, node_names

frames, points, confidence, node_names = _load_data3d(DATA3D_PATH, FRAME_START, FRAME_END)
print(f"Loaded {DATA3D_PATH.name}: {len(frames)} frames "
      f"[{FRAME_START}–{FRAME_END}], {len(node_names)} keypoints")

# =====================================================================
# PLOT
# =====================================================================

n_rows = 2 if SHOW_CONFIDENCE else 1
height_ratios = [3, 1] if SHOW_CONFIDENCE else [1]
fig, axes = plt.subplots(n_rows, 1, figsize=(8, 5 + 2 * SHOW_CONFIDENCE),
                         height_ratios=height_ratios, squeeze=False)
ax_xy = axes[0, 0]

cmap = plt.get_cmap(CMAP_NAME)
norm = Normalize(vmin=FRAME_START, vmax=FRAME_END)

for name in KP_NAMES:
    idx = node_names.index(name)
    x = points[:, idx, 0]
    y = points[:, idx, 2]
    valid = ~np.isnan(x) & ~np.isnan(y)

    ax_xy.scatter(x[valid], y[valid], s=MARKER_SIZE**2,
                  c=frames[valid], cmap=cmap, norm=norm,
                  edgecolors="none", label=name, zorder=3)

ax_xy.set_xlabel("x (mm)")
ax_xy.set_ylabel("y (mm)")
ax_xy.set_title(f"{DATA3D_PATH.stem}  |  Frames {FRAME_START}–{FRAME_END}")
ax_xy.legend(fontsize=9)
ax_xy.set_aspect("equal")

sm = cm.ScalarMappable(cmap=cmap, norm=norm)
fig.colorbar(sm, ax=ax_xy, fraction=0.03, pad=0.02, label="Frame")

# Confidence subplot
if SHOW_CONFIDENCE:
    ax_c = axes[1, 0]
    for name in KP_NAMES:
        idx = node_names.index(name)
        conf = confidence[:, idx]
        valid = ~np.isnan(conf)
        ax_c.plot(frames[valid], conf[valid], lw=1, alpha=0.8, label=name)
    ax_c.set_xlabel("Frame")
    ax_c.set_ylabel("Confidence")
    ax_c.set_xlim(FRAME_START, FRAME_END)
    ax_c.set_ylim(0, 1.05)
    ax_c.legend(fontsize=8, loc="lower left")

fig.tight_layout()
plt.show()


## Extract same frame from different cameras


In [ ]:
import cv2
import numpy as np
from pathlib import Path
from PIL import Image


def read_frame(video_path, frame_index):
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise RuntimeError(f"Could not open video: {video_path}")
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_index)
    ok, frame_bgr = cap.read()
    cap.release()
    if not ok or frame_bgr is None:
        raise RuntimeError(f"Could not read frame {frame_index} from {video_path}")
    return cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)


def get_video_files(video_dir):
    exts = {".avi", ".mp4", ".mov", ".mkv"}
    files = [p for p in video_dir.iterdir()
             if p.is_file() and p.suffix.lower() in exts]
    files.sort(key=lambda p: p.name)
    return files


def export_frames_per_camera(frame_index, video_dir, output_dir, crop_half="left"):
    # Save one lossless TIFF per camera, cropped to half width.
    # crop_half: "left" or "right"
    video_files = get_video_files(video_dir)
    if not video_files:
        raise RuntimeError(f"No videos found in: {video_dir}")

    print(f"Found {len(video_files)} videos. Saving to: {output_dir}")
    output_dir.mkdir(exist_ok=True, parents=True)

    for vpath in video_files:
        frame = read_frame(vpath, frame_index)
        h, w  = frame.shape[:2]
        half  = w // 2
        frame = frame[:, :half, :] if crop_half == "left" else frame[:, half:, :]

        out_tif = output_dir / f"{vpath.stem}_frame{frame_index}.tiff"
        Image.fromarray(frame).save(out_tif, compression="tiff_lzw")
        print(f"  saved: {out_tif.name}  ({frame.shape[1]}x{frame.shape[0]} px)")

    print("Done.")


# ── User inputs ───────────────────────────────────────────────────────────────
FRAME_INDEX = 107615
VIDEO_DIR   = Path("/mnt/lemebel/happyhouse_102025/session6/2025_10_12_15_06_46/videos")
OUTPUT_DIR  = Path("/home/user/3D_tracking_paper/output/free_walking/figure1/")
CROP_HALF   = "left"   # "left" or "right"

# ── Run ───────────────────────────────────────────────────────────────────────
export_frames_per_camera(FRAME_INDEX, VIDEO_DIR, OUTPUT_DIR, crop_half=CROP_HALF)


## Get frames in between 

In [ ]:
import cv2
import csv
import numpy as np
from pathlib import Path
from PIL import Image
from rembg import remove
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection as MplLineCollection
from collections import defaultdict

# ======================
# USER INPUTS
# =================T=====
# # WALKING VIDEO
# VIDEO_FOLDER = Path("/mnt/lemebel/happyhouse_102025/session6/2025_10_12_15_06_46/videos/")
# CAMERA_FILENAME = "Cam2012631.mp4"
# PREDICTION_CSV = Path("/mnt/lemebel/happyhouse_102025/session6/2025_10_12_15_06_46/labeled_data_grooming_V3/fly50_grooming_V3_output/2026_04_14_08_48_22/Cam2012631.csv")

# START_FRAME = 558153
# END_FRAME = 559860
# FRAME_INTERVAL = 100
# CROP_MARGIN = 10
# MIN_CROP_SIZE = 60
# OUTPUT_ROOT = Path("/home/user/3D_tracking_paper/output/grooming/trajectory_exports")

# COURSHIP VIDEO
VIDEO_FOLDER = Path("/home/user/red_data/courtship_videos/2025_10_20_13_20_04/videos/")
CAMERA_FILENAME = "Cam2012630.mp4"
PREDICTION_CSV = Path("/home/user/red_data/courtship_videos/2025_10_20_13_20_04/videos/labeled_data/2026_02_13_10_49_27/Cam2012630.csv")    

START_FRAME = 446306
END_FRAME = 448312
FRAME_INTERVAL = 1
CROP_MARGIN = 500
MIN_CROP_SIZE = 500

# OUTPUT_ROOT = Path("/home/user/red_data/courtship_videos/2025_10_20_13_20_04/videos/figures/trajectory_exports")


# # Staircase videos
# VIDEO_FOLDER = Path("/home/user/red_data/stairs_videos/2026_01_12_14_07_16/")
# CAMERA_FILENAME = "Cam2012631.mp4"
# PREDICTION_CSV   = Path("/home/user/red_data/stairs_videos/2026_01_12_14_07_16/labeled_data/2026_02_15_18_03_13/Cam2012861.csv")
# START_FRAME = 23141
# END_FRAME   = 25000
# FRAME_INTERVAL = 5
# CROP_MARGIN = 60
# MIN_CROP_SIZE = 160

# OUTPUT_ROOT = Path("/home/user/red_data/stairs_videos/2026_01_12_14_07_16/figures/trajectory_exports")

# # headless BDN2 videos 
# VIDEO_FOLDER = Path("/media/user/LaCie2/Session7/2025_10_13_18_22_50/")
# CAMERA_FILENAME = "Cam2012631.mp4"
# PREDICTION_CSV   = Path("/media/user/LaCie2/Session7/2025_10_13_18_22_50/labeled_data/2026_04_15_12_11_09/Cam2012831.csv")
# START_FRAME = 0
# END_FRAME   = 3999
# FRAME_INTERVAL = 25
# CROP_MARGIN = 60
# MIN_CROP_SIZE = 160
# OUTPUT_ROOT = Path("/home/user/3D_tracking_paper/output/BDN2 headless/trajectory_exports")


# ======================
# LOAD PREDICTIONS
# ======================

def load_predictions(csv_path):
    preds = {}
    with csv_path.open() as f:
        reader = csv.reader(f)
        next(reader)  # skeleton path
        for row in reader:
            frame = int(row[0])
            n_kp = (len(row)-1)//3
            pts = np.full((n_kp,2), np.nan)
            for i in range(n_kp):
                b = 1+3*i
                kid = int(row[b])
                x, y = float(row[b+1]), float(row[b+2])
                if abs(x)>1e6 or abs(y)>1e6:
                    continue
                if 0 <= kid < n_kp:
                    pts[kid] = [x, y]
            preds[frame] = pts
    return preds

preds = load_predictions(PREDICTION_CSV)


def load_skeleton(csv_path: Path):
    """Return (node_names, edges) from the skeleton JSON referenced in the CSV header."""
    with csv_path.open("r", encoding="utf-8") as f:
        first_line = f.readline().strip()
    skel_path = Path(first_line).expanduser()
    if not skel_path.is_file():
        skel_path = (csv_path.parent / first_line).resolve()
    if not skel_path.is_file():
        print(f"[WARNING] Skeleton not found: '{first_line}'")
        return None, None
    with skel_path.open("r", encoding="utf-8") as f:
        skel = json.load(f)
    names = skel.get("node_names")
    edges = skel.get("edges")
    return (names if isinstance(names, list) else None,
            edges if isinstance(edges, list) else None)


def classify_body_part(name: str) -> str:
    for prefix in ("T1L", "T1R", "T2L", "T2R", "T3L", "T3R"):
        if name.startswith(prefix):
            return prefix
    if name.startswith("WingL"):
        return "WingL"
    if name.startswith("WingR"):
        return "WingR"
    if name in ("EyeL", "EyeR"):
        return name
    return "body"


def draw_skeleton_cv2(frame_bgr, pts, edges, node_names, lw=2, joint_r=3):
    """Draw skeleton edges and joints onto a BGR frame in-place."""
    n = len(pts)
    for i, j in edges:
        if i >= n or j >= n:
            continue
        pi = classify_body_part(node_names[i])
        pj = classify_body_part(node_names[j])
        if pi not in PARTS_TO_SHOW or pj not in PARTS_TO_SHOW:
            continue
        xi, yi = pts[i]
        xj, yj = pts[j]
        if np.isnan(xi) or np.isnan(yi) or np.isnan(xj) or np.isnan(yj):
            continue
        color = SKELETON_COLORS_BGR.get(pi, (128, 128, 128))
        cv2.line(frame_bgr, (int(xi), int(yi)), (int(xj), int(yj)), color, lw, cv2.LINE_AA)
    for k in range(n):
        pk = classify_body_part(node_names[k])
        if pk not in PARTS_TO_SHOW:
            continue
        xk, yk = pts[k]
        if np.isnan(xk) or np.isnan(yk):
            continue
        color = SKELETON_COLORS_BGR.get(pk, (128, 128, 128))
        cv2.circle(frame_bgr, (int(xk), int(yk)), joint_r, color, -1, cv2.LINE_AA)



def save_frame_svg(frame_bgr, pts_math, edges, node_names, svg_path,
                   colors_hex, lw, joint_size, alpha):
    """Save a camera frame + skeleton as an SVG.

    Each body part is a separate named group so it can be selected,
    recolored, or hidden individually in Inkscape.
    """
    frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    img_h, img_w = frame_rgb.shape[:2]

    fig, ax = plt.subplots(figsize=(img_w / 100, img_h / 100), dpi=100)
    ax.imshow(frame_rgb, origin="upper")
    ax.axis("off")
    ax.set_xlim(0, img_w)
    ax.set_ylim(img_h, 0)

    # Flip to display convention (math y → image y for matplotlib axes)
    pts = pts_math.copy().astype(float)
    pts[:, 1] = img_h - 1 - pts[:, 1]

    segs_by_part   = defaultdict(list)
    joints_by_part = defaultdict(lambda: ([], []))

    for i, j in edges:
        if i >= len(pts) or j >= len(pts):
            continue
        pi = classify_body_part(node_names[i])
        pj = classify_body_part(node_names[j])
        if pi not in PARTS_TO_SHOW or pj not in PARTS_TO_SHOW:
            continue
        x0, y0 = pts[i]; x1, y1 = pts[j]
        if any(np.isnan(v) for v in (x0, y0, x1, y1)):
            continue
        segs_by_part[pi].append([(x0, y0), (x1, y1)])

    for k in range(len(pts)):
        xk, yk = pts[k]
        if np.isnan(xk) or np.isnan(yk):
            continue
        p = classify_body_part(node_names[k])
        if p not in PARTS_TO_SHOW:
            continue
        joints_by_part[p][0].append(xk)
        joints_by_part[p][1].append(yk)

    for part, segs in segs_by_part.items():
        color = colors_hex.get(part, "#888888")
        lc = MplLineCollection(segs, colors=[color] * len(segs),
                               linewidths=lw, alpha=alpha, capstyle="round")
        lc.set_gid(f"skel_{part}_edges")
        ax.add_collection(lc)

    for part, (jx, jy) in joints_by_part.items():
        color = colors_hex.get(part, "#888888")
        sc = ax.scatter(jx, jy, s=joint_size ** 2, c=[color] * len(jx),
                        alpha=alpha, zorder=5, edgecolors="none")
        sc.set_gid(f"skel_{part}_joints")

    fig.tight_layout(pad=0)
    fig.savefig(str(svg_path), format="svg", bbox_inches="tight")
    plt.close(fig)


node_names, edges = load_skeleton(PREDICTION_CSV)
if node_names:
    print(f"  Skeleton: {len(node_names)} nodes, {len(edges)} edges")

# ======================
# SETUP OUTPUT DIRS
# ======================

camera_name = Path(CAMERA_FILENAME).stem

# ======================
# SAVE MODES (set True/False)
# ======================
SAVE_FULL_FRAMES = True
SAVE_CROPPED_SQUARE_FRAMES = False
SAVE_BG_REMOVED_FRAMES = False

# Skeleton overlay
SHOW_SKELETON_OVERLAY = False
SKELETON_LINE_WIDTH   = 2       # edge thickness (px)
SKELETON_JOINT_RADIUS = 3       # joint circle radius (px)
SKELETON_COLORS_BGR = {         # BGR tuples matching cell-3 hex palette
    "body":  ( 85,   4,  81),
    "WingL": (253, 176,  88),
    "WingR": (212, 121, 107),
    "T1L":   (219, 226,  12),
    "T1R":   ( 12, 206,   8),
    "T2L":   ( 13, 230, 111),
    "T2R":   ( 27, 186, 226),
    "T3L":   (  6,  75, 249),
    "T3R":   ( 14,  14, 209),
    "EyeL": (  0, 255, 255),
    "EyeR": (  0, 200, 255),
}

# SVG export (matplotlib — skeleton editable per body part)
SAVE_SVG_FRAMES    = False
SVG_LINE_WIDTH     = 1.5
SVG_JOINT_SIZE     = 3.0
SVG_ALPHA          = 0.9
SKELETON_COLORS_HEX = {
    "body":  "#510455",
    "WingL": "#58B0FD",
    "WingR": "#6B79D4",
    "T1L":   "#0CE2DB",
    "T1R":   "#08CE0C",
    "T2L":   "#6FE60D",
    "T2R":   "#E2BA1B",
    "T3L":   "#F94B06",
    "T3R":   "#D10E0E",
    "EyeL": "#FFFF00",
    "EyeR": "#FFD700",
}

# Which body parts to draw (legs + eyes). Remove entries to hide them.
PARTS_TO_SHOW = {"T1L", "T1R", "T2L", "T2R", "T3L", "T3R", "EyeL", "EyeR"}

# For square crops: final output is always this size (px)
SQUARE_CROP_SIZE = 1024

# Where to run background removal from: "full" or "crop"
BG_REMOVE_SOURCE = "crop"

assert BG_REMOVE_SOURCE in {"full", "crop"}, "BG_REMOVE_SOURCE must be 'full' or 'crop'"
assert SAVE_FULL_FRAMES or SAVE_CROPPED_SQUARE_FRAMES or SAVE_BG_REMOVED_FRAMES, "Enable at least one save mode"

mode_tags = []
if SAVE_FULL_FRAMES:
    mode_tags.append("full")
if SAVE_CROPPED_SQUARE_FRAMES:
    mode_tags.append(f"crop{SQUARE_CROP_SIZE}")
if SAVE_BG_REMOVED_FRAMES:
    mode_tags.append(f"nobg_{BG_REMOVE_SOURCE}")

export_name = f"{camera_name}_frames_{START_FRAME}_{END_FRAME}_every_{FRAME_INTERVAL}_{'_'.join(mode_tags)}"

full_dir      = OUTPUT_ROOT / export_name / "full_raw"
full_skel_dir = OUTPUT_ROOT / export_name / "full_skel"
svg_dir       = OUTPUT_ROOT / export_name / "svg_skel"
crop_dir      = OUTPUT_ROOT / export_name / f"crop_square_{SQUARE_CROP_SIZE}_raw"
crop_skel_dir = OUTPUT_ROOT / export_name / f"crop_square_{SQUARE_CROP_SIZE}_skel"
nobg_dir      = OUTPUT_ROOT / export_name / f"nobg_{BG_REMOVE_SOURCE}_rgba"

if SAVE_FULL_FRAMES:
    full_dir.mkdir(parents=True, exist_ok=True)
    if SHOW_SKELETON_OVERLAY:
        full_skel_dir.mkdir(parents=True, exist_ok=True)
if SAVE_SVG_FRAMES and SHOW_SKELETON_OVERLAY:
    svg_dir.mkdir(parents=True, exist_ok=True)
if SAVE_CROPPED_SQUARE_FRAMES:
    crop_dir.mkdir(parents=True, exist_ok=True)
    if SHOW_SKELETON_OVERLAY:
        crop_skel_dir.mkdir(parents=True, exist_ok=True)
if SAVE_BG_REMOVED_FRAMES:
    nobg_dir.mkdir(parents=True, exist_ok=True)

video_path = VIDEO_FOLDER / CAMERA_FILENAME
assert video_path.exists(), f"Video not found: {video_path}"

def extract_square_with_padding(frame_bgr, cx, cy, side):
    h, w = frame_bgr.shape[:2]
    half = side // 2
    x0 = cx - half
    y0 = cy - half
    x1 = x0 + side
    y1 = y0 + side

    sx0 = max(0, x0)
    sy0 = max(0, y0)
    sx1 = min(w, x1)
    sy1 = min(h, y1)

    crop = np.zeros((side, side, 3), dtype=frame_bgr.dtype)
    dx0 = sx0 - x0
    dy0 = sy0 - y0
    crop[dy0:dy0 + (sy1 - sy0), dx0:dx0 + (sx1 - sx0)] = frame_bgr[sy0:sy1, sx0:sx1]
    return crop

def remove_bg_preserve_color(frame_bgr):
    pil_img = Image.fromarray(cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB))
    rgba = remove(pil_img)
    alpha = np.array(rgba.split()[-1])
    rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    return np.dstack([rgb, alpha])

# ======================
# EXTRACT / CROP / BG REMOVE
# ======================

cap = cv2.VideoCapture(str(video_path))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
print(f"Video frames: {total_frames}")
print(
    f"Processing frames with modes: full={SAVE_FULL_FRAMES}, crop={SAVE_CROPPED_SQUARE_FRAMES}, "
    f"nobg={SAVE_BG_REMOVED_FRAMES} ({BG_REMOVE_SOURCE})"
)

current_frame = START_FRAME
saved_full = 0
saved_crop = 0
saved_nobg = 0
saved_full_skel = 0
saved_crop_skel = 0
saved_svg = 0

while current_frame < END_FRAME:
    cap.set(cv2.CAP_PROP_POS_FRAMES, current_frame)
    ret, frame = cap.read()
    if not ret:
        print(f"Failed to read frame {current_frame}")
        break

    if current_frame not in preds:
        print(f"No prediction for frame {current_frame}, skipping.")
        current_frame += FRAME_INTERVAL
        continue

    pts_orig = preds[current_frame]               # math convention (for SVG)
    pts = pts_orig.copy()
    pts[:, 1] = frame.shape[0] - 1 - pts[:, 1]  # → image convention (for OpenCV)
    valid = ~np.isnan(pts[:, 0])
    if not np.any(valid):
        print(f"No valid keypoints for frame {current_frame}, skipping.")
        current_frame += FRAME_INTERVAL
        continue

    xs = pts[valid, 0]
    ys = pts[valid, 1]
    cx = int(np.round((xs.min() + xs.max()) / 2.0))
    cy = int(np.round((ys.min() + ys.max()) / 2.0))

    required_side = int(np.ceil(max(xs.max() - xs.min(), ys.max() - ys.min()) + 2 * CROP_MARGIN))
    required_side = max(required_side, MIN_CROP_SIZE)
    sample_side = max(required_side, SQUARE_CROP_SIZE)

    square_crop = extract_square_with_padding(frame, cx, cy, sample_side)
    if sample_side != SQUARE_CROP_SIZE:
        square_crop = cv2.resize(square_crop, (SQUARE_CROP_SIZE, SQUARE_CROP_SIZE), interpolation=cv2.INTER_AREA)

    fname = f"{camera_name}_frame_{current_frame:06d}.png"

    if SAVE_FULL_FRAMES:
        cv2.imwrite(str(full_dir / fname), frame)
        saved_full += 1
        if SHOW_SKELETON_OVERLAY and edges is not None:
            out_skel = frame.copy()
            draw_skeleton_cv2(out_skel, pts, edges, node_names,
                              lw=SKELETON_LINE_WIDTH, joint_r=SKELETON_JOINT_RADIUS)
            cv2.imwrite(str(full_skel_dir / fname), out_skel)
            saved_full_skel += 1

    if SAVE_CROPPED_SQUARE_FRAMES:
        cv2.imwrite(str(crop_dir / fname), square_crop)
        saved_crop += 1
        if SHOW_SKELETON_OVERLAY and edges is not None:
            scale = SQUARE_CROP_SIZE / sample_side
            x0_crop = cx - sample_side // 2
            y0_crop = cy - sample_side // 2
            pts_crop = (pts - np.array([x0_crop, y0_crop])) * scale
            out_skel = square_crop.copy()
            draw_skeleton_cv2(out_skel, pts_crop, edges, node_names,
                              lw=SKELETON_LINE_WIDTH, joint_r=SKELETON_JOINT_RADIUS)
            cv2.imwrite(str(crop_skel_dir / fname), out_skel)
            saved_crop_skel += 1

    if SAVE_SVG_FRAMES and SHOW_SKELETON_OVERLAY and edges is not None:
        svg_path = svg_dir / (Path(fname).stem + ".svg")
        save_frame_svg(frame, pts_orig, edges, node_names, svg_path,
                       SKELETON_COLORS_HEX, SVG_LINE_WIDTH,
                       SVG_JOINT_SIZE, SVG_ALPHA)
        saved_svg += 1

    if SAVE_BG_REMOVED_FRAMES:
        bg_input = frame if BG_REMOVE_SOURCE == "full" else square_crop
        rgba_out = remove_bg_preserve_color(bg_input)
        Image.fromarray(rgba_out).save(nobg_dir / fname)
        saved_nobg += 1

    current_frame += FRAME_INTERVAL

cap.release()

print("\nSave summary:")
if SAVE_FULL_FRAMES:
    print(f"  Full raw:  {saved_full} -> {full_dir}")
    if SHOW_SKELETON_OVERLAY:
        print(f"  Full skel: {saved_full_skel} -> {full_skel_dir}")
if SAVE_CROPPED_SQUARE_FRAMES:
    print(f"  Crop raw:  {saved_crop} -> {crop_dir}")
    if SHOW_SKELETON_OVERLAY:
        print(f"  Crop skel: {saved_crop_skel} -> {crop_skel_dir}")
if SAVE_SVG_FRAMES and SHOW_SKELETON_OVERLAY:
    print(f"  SVG skel:  {saved_svg} -> {svg_dir}")
if SAVE_BG_REMOVED_FRAMES:
    print(f"  BG removed ({BG_REMOVE_SOURCE}): {saved_nobg} -> {nobg_dir}")


## 3D Egocentric Tarsus Tip Trajectories (mm)

In [ ]:
"""
3D Egocentric Tarsus Tip Trajectories
======================================
Visualize step cycles of all 6 legs in body-centered coordinates.
Outputs vector PDF for editing in Inkscape.
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
from mpl_toolkits.mplot3d.art3d import Line3DCollection
import matplotlib as mpl
from pathlib import Path
import cv2

# ============================================================
# USER CONFIGURATION - EDIT THESE VALUES
# ============================================================

# Path to 3D predictions CSV
DATA_3D_PATH = Path("/home/user/src/JARVIS-HybridNet/projects/fly50_V4/predictions/predictions3D/Predictions_3D_20260114-145343/data3D.csv")

# Frame interval to visualize (inclusive)
FRAME_START = 107610
FRAME_END = 108410
JARVIS_SCALE = 10.0  # JARVIS outputs are 10x actual mm
# Video for background frame (optional - set to None to skip)
VIDEO_PATH = Path("/mnt/lemebel/happyhouse_102025/session6/2025_10_12_15_06_46/videos/Cam2012630.mp4")
BACKGROUND_FRAME = FRAME_START  # Which frame to use as background

# Output path for PDF
OUTPUT_DIR = Path("/mnt/lemebel/happyhouse_102025/session6/2025_10_12_15_06_46/figures")
OUTPUT_FILENAME = "egocentric_tarsus_trajectories.pdf"

# Visualization options
SHOW_BACKGROUND_IMAGE = True  # Set False for cleaner vector output
LINE_WIDTH = 1.5
COLORMAP = "viridis"  # Options: "viridis", "plasma", "coolwarm", "rainbow"
FADE_TAIL = True  # Fade older parts of trajectory

print("Configuration loaded.")

In [ ]:
# ============================================================
# DATA LOADING AND PROCESSING FUNCTIONS
# ============================================================

def load_3d_data(csv_path: Path, frame_start: int, frame_end: int):
    """
    Load 3D keypoint data from JARVIS CSV format.
    """
    print(f"Loading 3D data from {csv_path}...")
    print(f"  Frame range: {frame_start} to {frame_end}")
    
    # Read header to get keypoint names
    header_df = pd.read_csv(csv_path, nrows=0)
    all_cols = header_df.columns.tolist()
    
    # Get unique keypoint names (each appears 4 times: x, y, z, confidence)
    keypoint_names = []
    for i in range(0, len(all_cols), 4):
        keypoint_names.append(all_cols[i])
    
    print(f"  Found {len(keypoint_names)} keypoints")
    
    # Load only the rows we need (skip header rows, then select frame range)
    df = pd.read_csv(csv_path, skiprows=[1], nrows=frame_end + 1)
    df = df.iloc[frame_start:frame_end + 1].reset_index(drop=True)
    
    print(f"  Loaded {len(df)} frames")
    return df, keypoint_names


def extract_keypoint_xyz(df, keypoint_name):
    """Extract x, y, z coordinates for a specific keypoint."""
    cols = df.columns.tolist()
    
    try:
        start_idx = cols.index(keypoint_name)
    except ValueError:
        raise ValueError(f"Keypoint '{keypoint_name}' not found in data")
    
    x = df.iloc[:, start_idx].values.astype(float)/JARVIS_SCALE
    y = df.iloc[:, start_idx + 1].values.astype(float)/JARVIS_SCALE
    z = df.iloc[:, start_idx + 2].values.astype(float)/JARVIS_SCALE

    return np.column_stack([x, y, z])


def compute_egocentric_transform(thorax_pos, head_pos, abd_pos):
    """
    Compute transformation to egocentric (body-centered) coordinates.
    """
    N = thorax_pos.shape[0]
    
    # Body axis: from abdomen to head (forward direction)
    forward = head_pos - abd_pos
    forward = forward / (np.linalg.norm(forward, axis=1, keepdims=True) + 1e-10)
    
    # Up direction (approximate - we'll use world Z and orthogonalize)
    world_up = np.array([0, 0, 1])
    
    # Right vector = forward × up
    right = np.cross(forward, world_up)
    right = right / (np.linalg.norm(right, axis=1, keepdims=True) + 1e-10)
    
    # Recompute up to be orthogonal
    up = np.cross(right, forward)
    up = up / (np.linalg.norm(up, axis=1, keepdims=True) + 1e-10)
    
    # Build rotation matrices (columns are right, forward, up in body frame)
    R = np.zeros((N, 3, 3))
    R[:, :, 0] = right      # X-axis (right)
    R[:, :, 1] = forward    # Y-axis (forward)  
    R[:, :, 2] = up         # Z-axis (up)
    
    return R, thorax_pos


def transform_to_egocentric(points, R, origin):
    """Transform points to egocentric coordinates."""
    centered = points - origin
    ego_points = np.einsum('nij,nj->ni', R.transpose(0, 2, 1), centered)
    return ego_points


def read_video_frame(video_path: Path, frame_index: int):
    """Read a single frame from video."""
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise RuntimeError(f"Could not open video: {video_path}")
    
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_index)
    ok, frame_bgr = cap.read()
    cap.release()
    
    if not ok or frame_bgr is None:
        raise RuntimeError(f"Could not read frame {frame_index}")
    
    return cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)


def compute_velocity(xyz, fps=800):
    """
    Compute instantaneous velocity from position data.
    
    Args:
        xyz: (N, 3) array of positions
        fps: frame rate in Hz
        
    Returns:
        velocity: (N, 3) velocity vectors (first frame = 0)
        speed: (N,) magnitude of velocity
    """
    velocity = np.zeros_like(xyz)
    velocity[1:] = (xyz[1:] - xyz[:-1]) * fps  # mm/s or BL/s depending on input
    speed = np.linalg.norm(velocity, axis=1)
    return velocity, speed


def plot_3d_trajectory_gradient(ax, xyz, label, cmap_name="viridis", lw=1.5, 
                                 fade_tail=True, color_override=None):
    """Plot a 3D trajectory with time-gradient coloring."""
    valid = ~np.any(np.isnan(xyz), axis=1)
    xyz = xyz[valid]
    
    if len(xyz) < 2:
        return
    
    points = xyz.reshape(-1, 1, 3)
    segments = np.concatenate([points[:-1], points[1:]], axis=1)
    
    t = np.linspace(0, 1, len(segments))
    
    if color_override is not None:
        colors = np.zeros((len(segments), 4))
        colors[:, :3] = mpl.colors.to_rgb(color_override)
        if fade_tail:
            colors[:, 3] = 0.3 + 0.7 * t
        else:
            colors[:, 3] = 1.0
    else:
        cmap = mpl.cm.get_cmap(cmap_name)
        colors = cmap(t)
        if fade_tail:
            colors[:, 3] = 0.3 + 0.7 * t
    
    lc = Line3DCollection(segments, colors=colors, linewidths=lw)
    ax.add_collection3d(lc)
    
    ax.scatter(*xyz[0], s=20, c=[colors[0]], marker='o', edgecolors='k', linewidths=0.5)
    ax.scatter(*xyz[-1], s=30, c=[colors[-1]], marker='o', edgecolors='k', linewidths=0.5, label=label)


def create_multi_view_figure(ego_trajectories, leg_names, leg_colors,
                             output_path, frame_rgb=None, 
                             title="Egocentric Tarsus Tip Trajectories",
                             cmap_name="viridis", lw=1.5, fade_tail=True):
    """Create a multi-panel figure with 3D view and 2D projections."""
    if frame_rgb is not None:
        fig = plt.figure(figsize=(18, 12))
        gs = fig.add_gridspec(2, 3, width_ratios=[1.2, 1, 1], height_ratios=[1, 1])
        
        ax_img = fig.add_subplot(gs[0, 0])
        ax_img.imshow(frame_rgb)
        ax_img.set_title("Reference Frame", fontsize=11)
        ax_img.axis('off')
        
        ax_3d = fig.add_subplot(gs[0, 1:], projection='3d')
        ax_xy = fig.add_subplot(gs[1, 0])
        ax_xz = fig.add_subplot(gs[1, 1])
        ax_yz = fig.add_subplot(gs[1, 2])
    else:
        fig = plt.figure(figsize=(16, 10))
        gs = fig.add_gridspec(2, 3, width_ratios=[1.2, 1, 1], height_ratios=[1.2, 1])
        
        ax_3d = fig.add_subplot(gs[0, :], projection='3d')
        ax_xy = fig.add_subplot(gs[1, 0])
        ax_xz = fig.add_subplot(gs[1, 1])
        ax_yz = fig.add_subplot(gs[1, 2])
    
    # Plot 3D trajectories
    for traj, name, color in zip(ego_trajectories, leg_names, leg_colors):
        plot_3d_trajectory_gradient(ax_3d, traj, name, cmap_name=cmap_name,
                                    lw=lw, fade_tail=fade_tail, color_override=color)
    
    ax_3d.set_xlabel('Right-Left (mm)')
    ax_3d.set_ylabel('Anterior-Posterior (mm)')
    ax_3d.set_zlabel('Dorsal-Ventral (mm)')
    ax_3d.set_title(title, fontsize=12, fontweight='bold')
    ax_3d.legend(loc='upper left', bbox_to_anchor=(1.02, 1), fontsize=9)
    
    # 2D projections
    for traj, name, color in zip(ego_trajectories, leg_names, leg_colors):
        valid = ~np.any(np.isnan(traj), axis=1)
        t = traj[valid]
        if len(t) < 2:
            continue
        
        for ax, xi, yi, xlabel, ylabel, proj_title in [
            (ax_xy, 0, 1, 'Right-Left', 'Anterior-Posterior', 'Top View (X-Y)'),
            (ax_xz, 0, 2, 'Right-Left', 'Dorsal-Ventral', 'Side View (X-Z)'),
            (ax_yz, 1, 2, 'Anterior-Posterior', 'Dorsal-Ventral', 'Front View (Y-Z)')
        ]:
            pts = np.column_stack([t[:, xi], t[:, yi]])
            segments = np.stack([pts[:-1], pts[1:]], axis=1)
            
            time_vals = np.linspace(0, 1, len(segments))
            colors = np.zeros((len(segments), 4))
            colors[:, :3] = mpl.colors.to_rgb(color)
            colors[:, 3] = 0.3 + 0.7 * time_vals if fade_tail else 1.0
            
            lc = LineCollection(segments, colors=colors, linewidths=lw)
            ax.add_collection(lc)
            ax.autoscale()
            ax.set_xlabel(xlabel, fontsize=10)
            ax.set_ylabel(ylabel, fontsize=10)
            ax.set_title(proj_title, fontsize=11)
            ax.set_aspect('equal')
            ax.grid(True, alpha=0.3)
    
    # Equal axis limits
    all_points = np.vstack([t[~np.any(np.isnan(t), axis=1)] for t in ego_trajectories])
    max_range = np.max(np.ptp(all_points, axis=0)) / 2 * 1.1
    mid = np.mean(all_points, axis=0)
    
    ax_3d.set_xlim(mid[0] - max_range, mid[0] + max_range)
    ax_3d.set_ylim(mid[1] - max_range, mid[1] + max_range)
    ax_3d.set_zlim(mid[2] - max_range, mid[2] + max_range)
    
    ax_xy.set_xlim(mid[0] - max_range, mid[0] + max_range)
    ax_xy.set_ylim(mid[1] - max_range, mid[1] + max_range)
    ax_xz.set_xlim(mid[0] - max_range, mid[0] + max_range)
    ax_xz.set_ylim(mid[2] - max_range, mid[2] + max_range)
    ax_yz.set_xlim(mid[1] - max_range, mid[1] + max_range)
    ax_yz.set_ylim(mid[2] - max_range, mid[2] + max_range)
    
    plt.tight_layout()
    return fig

print("Functions loaded.")

In [ ]:
# ============================================================
# MAIN EXECUTION - Generate the figure
# ============================================================

# Tarsus tip keypoint names
TARSUS_TIP_NAMES = [
    # "T1L_TaTip",  # Front left
    "T1R_TaTip",  # Front right
    "T2L_TaTip",  # Middle left
    "T2R_TaTip",  # Middle right
    "T3L_TaTip",  # Hind left
    "T3R_TaTip",  # Hind right
]

LEG_DISPLAY_NAMES = [
    # "Front Left (T1L)",
    "Front Right (T1R)", 
    "Middle Left (T2L)",
    "Middle Right (T2R)",
    "Hind Left (T3L)",
    "Hind Right (T3R)",
]

LEG_COLORS = [
    # "#E63946",  # T1L - red
    "#457B9D",  # T1R - blue
    "#F4A261",  # T2L - orange
    "#2A9D8F",  # T2R - teal
    "#9B2226",  # T3L - dark red
    "#1D3557",  # T3R - dark blue
]

THORAX_KP = "Scutellum"
HEAD_KP = "Antenna_Base"
ABDOMEN_KP = "Abd_tip"

# Load 3D data
df, keypoint_names = load_3d_data(DATA_3D_PATH, FRAME_START, FRAME_END)

print(f"\nAvailable keypoints: {keypoint_names[:10]}... ({len(keypoint_names)} total)")

In [ ]:
# Extract body reference positions
thorax_pos = extract_keypoint_xyz(df, THORAX_KP)
head_pos = extract_keypoint_xyz(df, HEAD_KP)
abd_pos = extract_keypoint_xyz(df, ABDOMEN_KP)

print(f"Computing egocentric transformation...")
R, origin = compute_egocentric_transform(thorax_pos, head_pos, abd_pos)

# Extract and transform tarsus tip trajectories
ego_trajectories = []
for kp_name in TARSUS_TIP_NAMES:
    world_pos = extract_keypoint_xyz(df, kp_name)
    ego_pos = transform_to_egocentric(world_pos, R, origin)
    ego_trajectories.append(ego_pos)
    print(f"  {kp_name}: {ego_pos.shape[0]} frames, range: "
          f"X[{ego_pos[:,0].min():.1f}, {ego_pos[:,0].max():.1f}] "
          f"Y[{ego_pos[:,1].min():.1f}, {ego_pos[:,1].max():.1f}] "
          f"Z[{ego_pos[:,2].min():.1f}, {ego_pos[:,2].max():.1f}]")

In [ ]:
# Load background frame if requested
frame_rgb = None
if SHOW_BACKGROUND_IMAGE and VIDEO_PATH is not None:
    try:
        frame_rgb = read_video_frame(VIDEO_PATH, BACKGROUND_FRAME)
        print(f"Loaded background frame {BACKGROUND_FRAME} from {VIDEO_PATH.name}")
    except Exception as e:
        print(f"Warning: Could not load video frame: {e}")
        frame_rgb = None

# Create the figure
print(f"Generating figure...")
title = f"Egocentric Tarsus Trajectories (Frames {FRAME_START}-{FRAME_END})"

fig = create_multi_view_figure(
    ego_trajectories, 
    LEG_DISPLAY_NAMES, 
    LEG_COLORS,
    OUTPUT_DIR / OUTPUT_FILENAME,
    frame_rgb=frame_rgb,
    title=title,
    cmap_name=COLORMAP,
    lw=LINE_WIDTH,
    fade_tail=FADE_TAIL
)

# Save as vector PDF
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
output_path = OUTPUT_DIR / OUTPUT_FILENAME

fig.savefig(
    output_path, 
    format='pdf',
    dpi=300,
    bbox_inches='tight',
    metadata={'Creator': 'Matplotlib', 'Title': title}
)

print(f"\n✅ Saved vector PDF to: {output_path}")
print("   Open in Inkscape to edit line styles, colors, and layout.")

plt.show()

## 3D Egocentric Tarsus Tip Trajectories (bl)

In [ ]:
# Calculate body length (distance from Abd_tip to Antenna_Base)
body_lengths = np.linalg.norm(head_pos - abd_pos, axis=1)
BODY_LENGTH = np.mean(body_lengths)
body_length_std = np.std(body_lengths)

print(f"Body length calculation:")
print(f"  Mean: {BODY_LENGTH:.2f} (raw units)")
print(f"  Std:  {body_length_std:.2f}")
print(f"  Range: [{body_lengths.min():.2f}, {body_lengths.max():.2f}]")

# Normalize trajectories by body length
ego_trajectories_bl = [traj / BODY_LENGTH for traj in ego_trajectories]

print(f"\nNormalized trajectory ranges (in BL):")
for kp_name, traj_bl in zip(TARSUS_TIP_NAMES, ego_trajectories_bl):
    print(f"  {kp_name}: "
          f"X[{traj_bl[:,0].min():.2f}, {traj_bl[:,0].max():.2f}] "
          f"Y[{traj_bl[:,1].min():.2f}, {traj_bl[:,1].max():.2f}] "
          f"Z[{traj_bl[:,2].min():.2f}, {traj_bl[:,2].max():.2f}] BL")

In [ ]:
# Create updated figure with BL units
def create_multi_view_figure_bl(ego_trajectories, leg_names, leg_colors,
                                frame_rgb=None, title="Egocentric Tarsus Tip Trajectories",
                                cmap_name="viridis", lw=1.5, fade_tail=True,
                                units="BL"):
    """
    Create a multi-panel figure with 3D view and 2D projections.
    Units parameter controls axis labels (default: BL for body length).
    """
    if frame_rgb is not None:
        fig = plt.figure(figsize=(18, 12))
        gs = fig.add_gridspec(2, 3, width_ratios=[1.2, 1, 1], height_ratios=[1, 1])
        
        ax_img = fig.add_subplot(gs[0, 0])
        ax_img.imshow(frame_rgb)
        ax_img.set_title("Reference Frame", fontsize=11)
        ax_img.axis('off')
        
        ax_3d = fig.add_subplot(gs[0, 1:], projection='3d')
        ax_xy = fig.add_subplot(gs[1, 0])
        ax_xz = fig.add_subplot(gs[1, 1])
        ax_yz = fig.add_subplot(gs[1, 2])
    else:
        fig = plt.figure(figsize=(16, 10))
        gs = fig.add_gridspec(2, 3, width_ratios=[1.2, 1, 1], height_ratios=[1.2, 1])
        
        ax_3d = fig.add_subplot(gs[0, :], projection='3d')
        ax_xy = fig.add_subplot(gs[1, 0])
        ax_xz = fig.add_subplot(gs[1, 1])
        ax_yz = fig.add_subplot(gs[1, 2])
    
    # Plot 3D trajectories
    for traj, name, color in zip(ego_trajectories, leg_names, leg_colors):
        plot_3d_trajectory_gradient(ax_3d, traj, name, cmap_name=cmap_name,
                                    lw=lw, fade_tail=fade_tail, color_override=color)
    
    # Updated axis labels with units parameter
    ax_3d.set_xlabel(f'Right-Left ({units})')
    ax_3d.set_ylabel(f'Anterior-Posterior ({units})')
    ax_3d.set_zlabel(f'Dorsal-Ventral ({units})')
    ax_3d.set_title(title, fontsize=12, fontweight='bold')
    ax_3d.legend(loc='upper left', bbox_to_anchor=(1.02, 1), fontsize=9)
    
    # 2D projections with time gradient
    for traj, name, color in zip(ego_trajectories, leg_names, leg_colors):
        valid = ~np.any(np.isnan(traj), axis=1)
        t = traj[valid]
        if len(t) < 2:
            continue
        
        for ax, xi, yi, xlabel, ylabel, proj_title in [
            (ax_xy, 0, 1, f'Right-Left ({units})', f'Anterior-Posterior ({units})', 'Top View (X-Y)'),
            (ax_xz, 0, 2, f'Right-Left ({units})', f'Dorsal-Ventral ({units})', 'Side View (X-Z)'),
            (ax_yz, 1, 2, f'Anterior-Posterior ({units})', f'Dorsal-Ventral ({units})', 'Front View (Y-Z)')
        ]:
            pts = np.column_stack([t[:, xi], t[:, yi]])
            segments = np.stack([pts[:-1], pts[1:]], axis=1)
            
            time_vals = np.linspace(0, 1, len(segments))
            colors = np.zeros((len(segments), 4))
            colors[:, :3] = mpl.colors.to_rgb(color)
            colors[:, 3] = 0.3 + 0.7 * time_vals if fade_tail else 1.0
            
            lc = LineCollection(segments, colors=colors, linewidths=lw)
            ax.add_collection(lc)
            ax.autoscale()
            ax.set_xlabel(xlabel, fontsize=10)
            ax.set_ylabel(ylabel, fontsize=10)
            ax.set_title(proj_title, fontsize=11)
            ax.set_aspect('equal')
            ax.grid(True, alpha=0.3)
    
    # Equal axis limits
    all_points = np.vstack([t[~np.any(np.isnan(t), axis=1)] for t in ego_trajectories])
    max_range = np.max(np.ptp(all_points, axis=0)) / 2 * 1.1
    mid = np.mean(all_points, axis=0)
    
    ax_3d.set_xlim(mid[0] - max_range, mid[0] + max_range)
    ax_3d.set_ylim(mid[1] - max_range, mid[1] + max_range)
    ax_3d.set_zlim(mid[2] - max_range, mid[2] + max_range)
    
    ax_xy.set_xlim(mid[0] - max_range, mid[0] + max_range)
    ax_xy.set_ylim(mid[1] - max_range, mid[1] + max_range)
    ax_xz.set_xlim(mid[0] - max_range, mid[0] + max_range)
    ax_xz.set_ylim(mid[2] - max_range, mid[2] + max_range)
    ax_yz.set_xlim(mid[1] - max_range, mid[1] + max_range)
    ax_yz.set_ylim(mid[2] - max_range, mid[2] + max_range)
    
    plt.tight_layout()
    return fig

# Generate the figure with BL-normalized data
title_bl = f"Egocentric Tarsus Trajectories (Frames {FRAME_START}-{FRAME_END}, BL = {BODY_LENGTH:.1f})"

fig_bl = create_multi_view_figure_bl(
    ego_trajectories_bl,  # Use normalized trajectories
    LEG_DISPLAY_NAMES, 
    LEG_COLORS,
    frame_rgb=frame_rgb,
    title=title_bl,
    cmap_name=COLORMAP,
    lw=LINE_WIDTH,
    fade_tail=FADE_TAIL,
    units="BL"
)

# Save as vector PDF
output_path_bl = OUTPUT_DIR / "egocentric_tarsus_trajectories_BL.pdf"

fig_bl.savefig(
    output_path_bl, 
    format='pdf',
    dpi=300,
    bbox_inches='tight',
    metadata={'Creator': 'Matplotlib', 'Title': title_bl}
)

print(f"\n✅ Saved normalized figure to: {output_path_bl}")
print(f"   Body length: {BODY_LENGTH:.2f} raw units")
print("   All axes now in body length (BL) units")

plt.show()

## Trajectory + vel visualization


In [ ]:
# ============================================================
# TRAJECTORY + VELOCITY VISUALIZATION
# ============================================================

def create_trajectory_velocity_figure(
    ego_trajectories,
    velocities,
    speeds,
    leg_names,
    leg_colors,
    fps=800,
    arrow_every=50,
    arrow_scale=0.1,
    speed_cmap='plasma',
    frame_rgb=None,
    title="Trajectory with Velocity",
    units="mm",
    lw=1.5
):
    """
    Create multi-panel figure with trajectories colored by speed and velocity arrows.
    
    Layout (2 rows x 4 cols):
    [ 3D Trajectory (speed colormap) ][ Top View (X-Y) ][ Side View (X-Z) ][ Speed vs Time ]
    [        Reference Frame         ][ Front View (Y-Z) with velocity arrows             ]
    
    Args:
        ego_trajectories: list of (N, 3) position arrays
        velocities: list of (N, 3) velocity arrays
        speeds: list of (N,) speed arrays
        leg_names: list of names for legend
        leg_colors: list of colors for each trajectory
        fps: frame rate in Hz
        arrow_every: show velocity arrow every N frames
        arrow_scale: scale factor for arrow length
        speed_cmap: colormap for speed coloring
        frame_rgb: optional reference frame image
        title: figure title
        units: units string for axis labels (e.g., "mm" or "BL")
        lw: line width
        
    Returns:
        fig: matplotlib Figure object
    """
    # Compute global speed range for consistent colormap
    all_speeds = np.concatenate([s[~np.isnan(s)] for s in speeds])
    vmin, vmax = np.percentile(all_speeds, [2, 98])  # Use percentiles to avoid outliers
    
    # Create figure with 2x4 layout
    fig = plt.figure(figsize=(20, 10))
    gs = fig.add_gridspec(2, 4, width_ratios=[1.2, 1, 1, 1], height_ratios=[1, 1],
                          hspace=0.25, wspace=0.3)
    
    # Row 1: 3D trajectory, Top view, Side view, Speed plot
    ax_3d = fig.add_subplot(gs[0, 0], projection='3d')
    ax_xy = fig.add_subplot(gs[0, 1])
    ax_xz = fig.add_subplot(gs[0, 2])
    ax_speed = fig.add_subplot(gs[0, 3])
    
    # Row 2: Reference frame (or empty), Front view with arrows spanning 3 cols
    if frame_rgb is not None:
        ax_img = fig.add_subplot(gs[1, 0])
        ax_img.imshow(frame_rgb)
        ax_img.set_title("Reference Frame", fontsize=11)
        ax_img.axis('off')
    else:
        ax_img = fig.add_subplot(gs[1, 0])
        ax_img.axis('off')
    
    ax_yz = fig.add_subplot(gs[1, 1:])  # Front view spans remaining columns
    
    cmap = plt.cm.get_cmap(speed_cmap)
    norm = plt.Normalize(vmin=vmin, vmax=vmax)
    
    # Plot each trajectory
    for traj, vel, spd, name, base_color in zip(ego_trajectories, velocities, speeds, 
                                                 leg_names, leg_colors):
        valid = ~np.any(np.isnan(traj), axis=1)
        traj_v = traj[valid]
        vel_v = vel[valid]
        spd_v = spd[valid]
        
        if len(traj_v) < 2:
            continue
        
        # --- 3D trajectory colored by speed ---
        points_3d = traj_v.reshape(-1, 1, 3)
        segments_3d = np.concatenate([points_3d[:-1], points_3d[1:]], axis=1)
        colors_3d = cmap(norm(spd_v[:-1]))
        lc_3d = Line3DCollection(segments_3d, colors=colors_3d, linewidths=lw)
        ax_3d.add_collection3d(lc_3d)
        
        # Start/end markers
        ax_3d.scatter(*traj_v[0], s=30, c='green', marker='o', edgecolors='k', 
                      linewidths=0.5, zorder=10)
        ax_3d.scatter(*traj_v[-1], s=30, c='red', marker='s', edgecolors='k', 
                      linewidths=0.5, zorder=10, label=name)
        
        # --- 2D projections colored by speed ---
        for ax, xi, yi in [(ax_xy, 0, 1), (ax_xz, 0, 2), (ax_yz, 1, 2)]:
            pts_2d = np.column_stack([traj_v[:, xi], traj_v[:, yi]])
            segs_2d = np.stack([pts_2d[:-1], pts_2d[1:]], axis=1)
            colors_2d = cmap(norm(spd_v[:-1]))
            lc_2d = LineCollection(segs_2d, colors=colors_2d, linewidths=lw)
            ax.add_collection(lc_2d)
        
        # --- Velocity arrows on Front view (Y-Z) ---
        arrow_indices = np.arange(0, len(traj_v), arrow_every)
        for idx in arrow_indices:
            if idx < len(vel_v):
                # Y-Z projection of velocity
                y, z = traj_v[idx, 1], traj_v[idx, 2]
                vy, vz = vel_v[idx, 1] * arrow_scale, vel_v[idx, 2] * arrow_scale
                if np.linalg.norm([vy, vz]) > 1e-6:  # Skip near-zero velocities
                    ax_yz.arrow(y, z, vy, vz, head_width=0.02, head_length=0.01,
                               fc=base_color, ec=base_color, alpha=0.7, zorder=5)
        
        # --- Speed vs Time plot ---
        time_s = np.arange(len(spd_v)) / fps
        ax_speed.plot(time_s, spd_v, color=base_color, lw=1.0, alpha=0.8, label=name)
    
    # Configure 3D axis
    ax_3d.set_xlabel(f'Right-Left ({units})', fontsize=9)
    ax_3d.set_ylabel(f'Anterior-Posterior ({units})', fontsize=9)
    ax_3d.set_zlabel(f'Dorsal-Ventral ({units})', fontsize=9)
    ax_3d.set_title("3D Trajectory (colored by speed)", fontsize=11, fontweight='bold')
    
    # Set equal axis limits for 3D
    all_points = np.vstack([t[~np.any(np.isnan(t), axis=1)] for t in ego_trajectories])
    max_range = np.max(np.ptp(all_points, axis=0)) / 2 * 1.1
    mid = np.mean(all_points, axis=0)
    ax_3d.set_xlim(mid[0] - max_range, mid[0] + max_range)
    ax_3d.set_ylim(mid[1] - max_range, mid[1] + max_range)
    ax_3d.set_zlim(mid[2] - max_range, mid[2] + max_range)
    
    # Configure 2D projection axes
    proj_configs = [
        (ax_xy, 0, 1, f'Right-Left ({units})', f'Anterior-Posterior ({units})', 'Top View (X-Y)'),
        (ax_xz, 0, 2, f'Right-Left ({units})', f'Dorsal-Ventral ({units})', 'Side View (X-Z)'),
        (ax_yz, 1, 2, f'Anterior-Posterior ({units})', f'Dorsal-Ventral ({units})', 
         'Front View (Y-Z) + Velocity Arrows')
    ]
    
    for ax, xi, yi, xlabel, ylabel, ax_title in proj_configs:
        ax.autoscale()
        ax.set_xlabel(xlabel, fontsize=9)
        ax.set_ylabel(ylabel, fontsize=9)
        ax.set_title(ax_title, fontsize=11)
        ax.set_aspect('equal')
        ax.grid(True, alpha=0.3)
    
    # Set equal limits for 2D projections
    ax_xy.set_xlim(mid[0] - max_range, mid[0] + max_range)
    ax_xy.set_ylim(mid[1] - max_range, mid[1] + max_range)
    ax_xz.set_xlim(mid[0] - max_range, mid[0] + max_range)
    ax_xz.set_ylim(mid[2] - max_range, mid[2] + max_range)
    ax_yz.set_xlim(mid[1] - max_range, mid[1] + max_range)
    ax_yz.set_ylim(mid[2] - max_range, mid[2] + max_range)
    
    # Configure speed plot
    ax_speed.set_xlabel('Time (s)', fontsize=9)
    speed_unit = f'{units}/s' if units != 'BL' else 'BL/s'
    ax_speed.set_ylabel(f'Speed ({speed_unit})', fontsize=9)
    ax_speed.set_title('Instantaneous Speed', fontsize=11)
    ax_speed.legend(fontsize=8, loc='upper right')
    ax_speed.grid(True, alpha=0.3)
    ax_speed.set_xlim(0, None)
    ax_speed.set_ylim(0, None)
    
    # Add colorbar for speed
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=[ax_3d, ax_xy, ax_xz], shrink=0.6, pad=0.02, 
                        location='right', aspect=30)
    cbar.set_label(f'Speed ({speed_unit})', fontsize=9)
    
    # Overall title
    fig.suptitle(title, fontsize=14, fontweight='bold', y=0.98)
    
    plt.tight_layout(rect=[0, 0, 1, 0.96])
    return fig


print("Trajectory + velocity visualization function loaded.")

In [ ]:
# --- USER CONFIGURATION ---
# Select leg pairs to plot (contralateral pairs)
leg_pairs = [
    ("T1L_TaTip", "T1R_TaTip"),  # Front legs
    ("T2L_TaTip", "T2R_TaTip"),  # Mid legs
    ("T3L_TaTip", "T3R_TaTip"),  # Hind legs
]

# Frame range to PLOT (relative to loaded data, or absolute frame numbers)
# Set to None to use full loaded range
PLOT_FRAME_START = None  # e.g., 107600 for absolute, or None for start of loaded data
PLOT_FRAME_END = None    # e.g., 108400 for absolute, or None for end of loaded data

# JARVIS scale factor (JARVIS outputs are 10x actual mm)
JARVIS_SCALE = 10.0

# Frame rate for velocity computation
FPS = 800

# Colors: left = warm, right = cool
COLORS = {
    # "T1L_TaTip": "#E63946",  "T1R_TaTip": "#457B9D",
    "T2L_TaTip": "#F4A261",  "T2R_TaTip": "#2A9D8F", 
    "T3L_TaTip": "#9B2226",  "T3R_TaTip": "#1D3557",
}

PAIR_LABELS = ["T1 (Front)", "T2 (Mid)", "T3 (Hind)"]
# --------------------------

# Extract data for all legs (full loaded range)
leg_data = {}
for left, right in leg_pairs:
    for kp in [left, right]:
        if kp not in keypoint_names:
            print(f"Warning: '{kp}' not found, skipping.")
            continue
        pos = extract_keypoint_xyz(df, kp) / JARVIS_SCALE
        vel, spd = compute_velocity(pos, fps=FPS)
        acc = np.zeros_like(vel)
        acc[1:] = (vel[1:] - vel[:-1]) * FPS  # mm/s^2
        acc_z = acc[:, 2]
        leg_data[kp] = {'pos': pos, 'vel': vel, 'spd': spd, 'acc': acc, 'acc_z': acc_z}

# Compute frame indices for plotting
n_frames_total = list(leg_data.values())[0]['pos'].shape[0]
all_frames = np.arange(FRAME_START, FRAME_START + n_frames_total)

# Determine plot range (convert absolute frame numbers to indices)
if PLOT_FRAME_START is None:
    plot_start_idx = 0
else:
    plot_start_idx = max(0, PLOT_FRAME_START - FRAME_START)

if PLOT_FRAME_END is None:
    plot_end_idx = n_frames_total
else:
    plot_end_idx = min(n_frames_total, PLOT_FRAME_END - FRAME_START)

# Slice for plotting
plot_slice = slice(plot_start_idx, plot_end_idx)
frames = all_frames[plot_slice]

print(f"Loaded data: frames {FRAME_START} to {FRAME_START + n_frames_total - 1}")
print(f"Plotting: frames {frames[0]} to {frames[-1]} ({len(frames)} frames)")

# ============================================================
# FIGURE: Z trajectories + Speed + Acceleration, both by pairs
# ============================================================
fig, axes = plt.subplots(9, 1, figsize=(14, 18), sharex=True)

# Plot Z, Speed, Acceleration for each leg pair
for i, ((left, right), label) in enumerate(zip(leg_pairs, PAIR_LABELS)):
    ax_z = axes[i]
    ax_spd = axes[i + 3]
    ax_acc = axes[i + 6]
    
    # Z trajectory (sliced to plot range)
    if left in leg_data:
        ax_z.plot(frames, leg_data[left]['pos'][plot_slice, 2], 
                  color=COLORS[left], lw=1.2, alpha=0.9, label='Left')
    if right in leg_data:
        ax_z.plot(frames, leg_data[right]['pos'][plot_slice, 2], 
                  color=COLORS[right], lw=1.2, alpha=0.9, label='Right')
    
    ax_z.set_ylabel(f'{label} Z (mm)', fontsize=10)
    ax_z.legend(loc='upper right', fontsize=9, ncol=2)
    ax_z.grid(True, alpha=0.3)
    
    # Speed (sliced to plot range)
    if left in leg_data:
        ax_spd.plot(frames, leg_data[left]['spd'][plot_slice], 
                    color=COLORS[left], lw=1.0, alpha=0.8, label='Left')
    if right in leg_data:
        ax_spd.plot(frames, leg_data[right]['spd'][plot_slice], 
                    color=COLORS[right], lw=1.0, alpha=0.8, label='Right')
    
    ax_spd.set_ylabel(f'{label} Speed (mm/s)', fontsize=10)
    ax_spd.legend(loc='upper right', fontsize=9, ncol=2)
    ax_spd.grid(True, alpha=0.3)
    ax_spd.set_ylim(0, None)

    # Acceleration (sliced to plot range)
    if left in leg_data:
        ax_acc.plot(frames, leg_data[left]['acc_z'][plot_slice], 
                    color=COLORS[left], lw=1.0, alpha=0.8, label='Left')
    if right in leg_data:
        ax_acc.plot(frames, leg_data[right]['acc_z'][plot_slice], 
                    color=COLORS[right], lw=1.0, alpha=0.8, label='Right')

    ax_acc.set_ylabel(f'{label} Accel (mm/s^2)', fontsize=10)
    ax_acc.legend(loc='upper right', fontsize=9, ncol=2)
    ax_acc.grid(True, alpha=0.3)
    ax_acc.set_ylim(0, None)

axes[0].set_title(f"Tarsus Tip Z Trajectories (Frames {frames[0]}-{frames[-1]})", 
                  fontsize=13, fontweight='bold')
axes[3].set_title('Instantaneous Speed by Leg Pair', fontsize=13, fontweight='bold')
axes[6].set_title('Instantaneous Acceleration by Leg Pair', fontsize=13, fontweight='bold')
axes[8].set_xlabel('Frame', fontsize=11)

plt.tight_layout()

# Save
output_path = OUTPUT_DIR / "z_trajectory_speed_by_pair.pdf"
fig.savefig(output_path, format='pdf', dpi=300, bbox_inches='tight')
print(f"Saved to: {output_path}")

plt.show()


## PHASE DETECTION: Velocity Threshold Method (Pratt et al. 2024)

In [ ]:
# ============================================================
# PHASE DETECTION: Velocity Threshold Method (Pratt et al. 2024)
# ============================================================
# Method: velocity > threshold = SWING, velocity < threshold = STANCE
# Reference: https://github.com/Prattbuw/Treadmill_Paper

from scipy.ndimage import gaussian_filter1d

# --- CONFIGURATION ---
VELOCITY_THRESHOLD = 25  # mm/s - adjust based on your data (Pratt uses 5-15)
GAUSSIAN_SIGMA = 1.5     # Smoothing parameter
# ---------------------

def detect_swing_stance(speed, threshold=25, sigma=1.5):
    """
    Detect swing/stance phases using velocity threshold.
    
    Args:
        speed: (N,) array of speed values (mm/s)
        threshold: velocity threshold - above = swing, below = stance
        sigma: Gaussian smoothing parameter
        
    Returns:
        phase: (N,) array - 1 = swing, 0 = stance
        smoothed_speed: (N,) smoothed speed values
    """
    # Smooth the speed signal
    smoothed = gaussian_filter1d(speed, sigma)
    
    # Threshold: swing = 1, stance = 0
    phase = (smoothed > threshold).astype(int)
    
    return phase, smoothed


def detect_phase_transitions(phase):
    """
    Find swing onset and stance onset frames.
    
    Returns:
        swing_onsets: frames where stance -> swing (foot up)
        stance_onsets: frames where swing -> stance (foot down)
    """
    diff = np.diff(phase)
    swing_onsets = np.where(diff == 1)[0] + 1   # 0->1 transition
    stance_onsets = np.where(diff == -1)[0] + 1  # 1->0 transition
    return swing_onsets, stance_onsets


# Compute phase for all legs
leg_phases = {}
leg_smoothed_speeds = {}

for kp, data in leg_data.items():
    phase, smoothed = detect_swing_stance(data['spd'], VELOCITY_THRESHOLD, GAUSSIAN_SIGMA)
    leg_phases[kp] = phase
    leg_smoothed_speeds[kp] = smoothed
    
    swing_on, stance_on = detect_phase_transitions(phase)
    print(f"{kp}: {len(swing_on)} swings, {len(stance_on)} stances detected")

print(f"\nUsing velocity threshold: {VELOCITY_THRESHOLD} mm/s")

In [ ]:
# ============================================================
# VISUALIZE PHASE DETECTION - Verify threshold is working
# ============================================================

fig, axes = plt.subplots(3, 1, figsize=(14, 8), sharex=True)

for i, ((left, right), label) in enumerate(zip(leg_pairs, PAIR_LABELS)):
    ax = axes[i]
    
    # Plot Z trajectory
    if left in leg_data:
        z_left = leg_data[left]['pos'][:, 2]
        phase_left = leg_phases[left]
        ax.plot(frames, z_left, color=COLORS[left], lw=1.2, alpha=0.9, label=f'{left[:-7]}L Z')
        # Shade swing periods
        ax.fill_between(frames, ax.get_ylim()[0], ax.get_ylim()[1], 
                        where=phase_left==1, alpha=0.2, color=COLORS[left], 
                        transform=ax.get_xaxis_transform())
    
    if right in leg_data:
        z_right = leg_data[right]['pos'][:, 2]
        phase_right = leg_phases[right]
        ax.plot(frames, z_right, color=COLORS[right], lw=1.2, alpha=0.9, label=f'{left[:-7]}R Z')
        ax.fill_between(frames, 0, 1, where=phase_right==1, alpha=0.2, color=COLORS[right],
                        transform=ax.get_xaxis_transform())
    
    ax.set_ylabel(f'{label}\nZ (mm)', fontsize=10)
    ax.legend(loc='upper right', fontsize=9, ncol=2)
    ax.grid(True, alpha=0.3)

axes[0].set_title(f'Phase Detection Verification (shaded = SWING, threshold={VELOCITY_THRESHOLD} mm/s)', 
                  fontsize=12, fontweight='bold')
axes[2].set_xlabel('Frame', fontsize=11)

plt.tight_layout()
plt.show()

print("Shaded regions = SWING phase (velocity > threshold)")
print("Adjust VELOCITY_THRESHOLD in previous cell if detection doesn't match Z peaks")

In [ ]:
# ============================================================
# INTER-LEG PHASE RELATIONSHIPS
# ============================================================
# Compute which legs are in phase (swing together) vs anti-phase (alternating)

def compute_phase_correlation(phase1, phase2):
    """
    Compute correlation between two phase signals.
    +1 = perfectly in phase (swing together)
    -1 = perfectly anti-phase (alternating)
     0 = no relationship
    """
    # Convert to -1/+1 for correlation
    p1 = 2 * phase1 - 1
    p2 = 2 * phase2 - 1
    return np.corrcoef(p1, p2)[0, 1]


def compute_overlap_fraction(phase1, phase2):
    """
    Compute fraction of time both legs are in same phase.
    """
    same = np.sum(phase1 == phase2) / len(phase1)
    return same


# Define leg pair comparisons
leg_comparisons = [
    # Contralateral pairs (left vs right, same segment)
    ("T1L_TaTip", "T1R_TaTip", "T1 L-R (contralateral)"),
    ("T2L_TaTip", "T2R_TaTip", "T2 L-R (contralateral)"),
    ("T3L_TaTip", "T3R_TaTip", "T3 L-R (contralateral)"),
    # Ipsilateral pairs (same side, different segments)
    ("T1L_TaTip", "T2L_TaTip", "T1-T2 Left (ipsilateral)"),
    ("T2L_TaTip", "T3L_TaTip", "T2-T3 Left (ipsilateral)"),
    ("T1R_TaTip", "T2R_TaTip", "T1-T2 Right (ipsilateral)"),
    ("T2R_TaTip", "T3R_TaTip", "T2-T3 Right (ipsilateral)"),
    # Diagonal pairs
    ("T1L_TaTip", "T2R_TaTip", "T1L-T2R (diagonal)"),
    ("T1R_TaTip", "T2L_TaTip", "T1R-T2L (diagonal)"),
    ("T2L_TaTip", "T3R_TaTip", "T2L-T3R (diagonal)"),
    ("T2R_TaTip", "T3L_TaTip", "T2R-T3L (diagonal)"),
    # Tripod groups
    ("T1L_TaTip", "T3L_TaTip", "T1L-T3L (same tripod)"),
    ("T1R_TaTip", "T3R_TaTip", "T1R-T3R (same tripod)"),
]

print("=" * 60)
print("INTER-LEG PHASE RELATIONSHIPS")
print("=" * 60)
print(f"{'Leg Pair':<30} {'Correlation':>12} {'Same Phase %':>14}")
print("-" * 60)

results = []
for leg1, leg2, name in leg_comparisons:
    if leg1 in leg_phases and leg2 in leg_phases:
        corr = compute_phase_correlation(leg_phases[leg1], leg_phases[leg2])
        overlap = compute_overlap_fraction(leg_phases[leg1], leg_phases[leg2])
        results.append((name, corr, overlap))
        
        # Interpret
        if corr > 0.5:
            interp = "IN PHASE"
        elif corr < -0.5:
            interp = "ANTI-PHASE"
        else:
            interp = "mixed"
        
        print(f"{name:<30} {corr:>+10.3f}   {overlap*100:>10.1f}%  ({interp})")

print("-" * 60)
print("\nInterpretation:")
print("  Correlation > +0.5: legs swing TOGETHER (in phase)")
print("  Correlation < -0.5: legs ALTERNATE (anti-phase)")
print("  Typical tripod gait: contralateral legs are anti-phase,")
print("                       diagonal pairs (T1L-T2R, T2L-T3R) are in phase")

## TRIPOD ANALYSIS: Scutellum peaks at mid-swing, troughs at transitions?


In [ ]:
# ============================================================
# TRIPOD ANALYSIS: Scutellum peaks at mid-swing, troughs at transitions?
# ============================================================
# Hypothesis: Body height peaks during mid-swing of each tripod,
#             and dips during tripod transitions

# Define tripods
LEFT_TRIPOD = ["T1L_TaTip", "T2R_TaTip", "T3L_TaTip"]   # L1, R2, L3
RIGHT_TRIPOD = ["T1R_TaTip", "T2L_TaTip", "T3R_TaTip"]  # R1, L2, R3

def compute_tripod_swing_signal(leg_phases, tripod_legs):
    """
    Compute how many legs in the tripod are in swing (0-3).
    """
    signals = [leg_phases[leg] for leg in tripod_legs if leg in leg_phases]
    return np.sum(signals, axis=0)

# Compute tripod swing counts
left_tripod_swing = compute_tripod_swing_signal(leg_phases, LEFT_TRIPOD)
right_tripod_swing = compute_tripod_swing_signal(leg_phases, RIGHT_TRIPOD)

# Identify tripod state at each frame
# "left" = left tripod in swing (>= 2 legs), "right" = right tripod in swing, "transition" = unclear
def get_tripod_state(left_count, right_count):
    """Determine which tripod is in swing or if transitioning."""
    states = []
    for l, r in zip(left_count, right_count):
        if l >= 2 and r <= 1:
            states.append('left_swing')
        elif r >= 2 and l <= 1:
            states.append('right_swing')
        else:
            states.append('transition')
    return np.array(states)

tripod_state = get_tripod_state(left_tripod_swing, right_tripod_swing)

# Find transitions (state changes)
state_changes = np.where(tripod_state[:-1] != tripod_state[1:])[0] + 1

# Find mid-points of each swing bout
def find_swing_midpoints(tripod_state, swing_label):
    """Find the center frame of each continuous swing bout."""
    in_swing = (tripod_state == swing_label)
    midpoints = []
    
    # Find contiguous swing regions
    diff = np.diff(in_swing.astype(int))
    starts = np.where(diff == 1)[0] + 1
    ends = np.where(diff == -1)[0] + 1
    
    # Handle edge cases
    if in_swing[0]:
        starts = np.concatenate([[0], starts])
    if in_swing[-1]:
        ends = np.concatenate([ends, [len(in_swing)]])
    
    for s, e in zip(starts, ends):
        midpoints.append((s + e) // 2)
    
    return np.array(midpoints)

left_swing_midpoints = find_swing_midpoints(tripod_state, 'left_swing')
right_swing_midpoints = find_swing_midpoints(tripod_state, 'right_swing')
all_swing_midpoints = np.sort(np.concatenate([left_swing_midpoints, right_swing_midpoints]))

# Find transition midpoints (center of transition periods)
transition_midpoints = find_swing_midpoints(tripod_state, 'transition')

print(f"Found {len(left_swing_midpoints)} left tripod swings, {len(right_swing_midpoints)} right tripod swings")
print(f"Found {len(transition_midpoints)} transition periods")

# ============================================================
# ANALYSIS: Compare scutellum Z at swing midpoints vs transitions
# ============================================================

scutellum_at_swing_mid = scutellum_smooth[all_swing_midpoints] if len(all_swing_midpoints) > 0 else []
scutellum_at_transitions = scutellum_smooth[transition_midpoints] if len(transition_midpoints) > 0 else []

print("\n" + "=" * 60)
print("SCUTELLUM HEIGHT AT TRIPOD EVENTS")
print("=" * 60)
if len(scutellum_at_swing_mid) > 0:
    print(f"At SWING MIDPOINTS:  mean = {np.mean(scutellum_at_swing_mid):.4f} mm")
if len(scutellum_at_transitions) > 0:
    print(f"At TRANSITIONS:      mean = {np.mean(scutellum_at_transitions):.4f} mm")
if len(scutellum_at_swing_mid) > 0 and len(scutellum_at_transitions) > 0:
    diff = np.mean(scutellum_at_swing_mid) - np.mean(scutellum_at_transitions)
    print(f"DIFFERENCE:          {diff:+.4f} mm")
    print(f"\n{'HYPOTHESIS SUPPORTED' if diff > 0.005 else 'HYPOTHESIS NOT CLEARLY SUPPORTED'}: ", end="")
    print("Body is HIGHER at swing midpoints" if diff > 0.005 else "No clear difference")

In [ ]:
# ============================================================
# VISUALIZATION: Gait Phase Diagram + Scutellum with Tripod Events
# ============================================================

fig, axes = plt.subplots(3, 1, figsize=(16, 10), sharex=True,
                         gridspec_kw={'height_ratios': [2, 1.5, 1.5]})

# --- TOP: Gait Phase Diagram (like the image) ---
ax_gait = axes[0]

leg_order = ["T1L_TaTip", "T2L_TaTip", "T3L_TaTip", "T1R_TaTip", "T2R_TaTip", "T3R_TaTip"]
leg_labels = ["L1", "L2", "L3", "R1", "R2", "R3"]

for i, (leg, label) in enumerate(zip(leg_order, leg_labels)):
    if leg not in leg_phases:
        continue
    phase = leg_phases[leg]
    y_pos = len(leg_order) - i - 1  # Reverse order so L1 is at top
    
    # Draw stance periods as black bars
    stance = (phase == 0)
    diff = np.diff(stance.astype(int))
    starts = np.where(diff == 1)[0] + 1
    ends = np.where(diff == -1)[0] + 1
    
    if stance[0]:
        starts = np.concatenate([[0], starts])
    if stance[-1]:
        ends = np.concatenate([ends, [len(stance)]])
    
    for s, e in zip(starts, ends):
        ax_gait.barh(y_pos, frames[e-1] - frames[s], left=frames[s], height=0.7, 
                    color='black', alpha=0.9)

ax_gait.set_yticks(range(len(leg_labels)))
ax_gait.set_yticklabels(leg_labels[::-1], fontsize=11)
ax_gait.set_ylabel('Leg', fontsize=11)
ax_gait.set_title('Gait Phase Diagram (black = STANCE, white = SWING)', fontsize=12, fontweight='bold')
ax_gait.set_ylim(-0.5, len(leg_labels) - 0.5)

# Add vertical lines at swing midpoints and transitions
for mid in all_swing_midpoints:
    ax_gait.axvline(frames[mid], color='green', alpha=0.5, lw=1, ls='-')
for trans in transition_midpoints:
    ax_gait.axvline(frames[trans], color='orange', alpha=0.5, lw=1, ls='--')

# --- MIDDLE: Tripod state ---
ax_tripod = axes[1]

# Color-code tripod state
tripod_colors = {'left_swing': 'coral', 'right_swing': 'steelblue', 'transition': 'gray'}
for state_label, color in tripod_colors.items():
    mask = tripod_state == state_label
    ax_tripod.fill_between(frames, 0, 1, where=mask, color=color, alpha=0.5, 
                           transform=ax_tripod.get_xaxis_transform())

ax_tripod.plot(frames, left_tripod_swing / 3, 'coral', lw=1.5, label='Left tripod (L1,R2,L3)')
ax_tripod.plot(frames, right_tripod_swing / 3, 'steelblue', lw=1.5, label='Right tripod (R1,L2,R3)')
ax_tripod.axhline(0.67, color='gray', ls=':', lw=1, alpha=0.5)  # 2/3 threshold
ax_tripod.set_ylabel('Tripod\nSwing Fraction', fontsize=10)
ax_tripod.set_ylim(0, 1)
ax_tripod.legend(loc='upper right', fontsize=9)
ax_tripod.set_title('Tripod Coordination (shaded = dominant tripod)', fontsize=11)

# --- BOTTOM: Scutellum height with markers ---
ax_scut = axes[2]

ax_scut.fill_between(frames, scutellum_smooth.min(), scutellum_smooth, color='gray', alpha=0.3)
ax_scut.plot(frames, scutellum_smooth, 'k-', lw=1.5, label='Scutellum Z')
ax_scut.axhline(np.mean(scutellum_smooth), color='red', ls='--', lw=1, 
                label=f'Mean: {np.mean(scutellum_smooth):.2f} mm')

# Mark swing midpoints (should be at peaks)
ax_scut.scatter(frames[all_swing_midpoints], scutellum_smooth[all_swing_midpoints], 
               c='green', s=80, zorder=5, marker='^', label='Swing midpoint')

# Mark transitions (should be at troughs)
if len(transition_midpoints) > 0:
    ax_scut.scatter(frames[transition_midpoints], scutellum_smooth[transition_midpoints], 
                   c='orange', s=80, zorder=5, marker='v', label='Tripod transition')

ax_scut.set_ylabel('Scutellum Z (mm)', fontsize=10)
ax_scut.set_xlabel('Frame', fontsize=11)
ax_scut.legend(loc='upper right', fontsize=9)
ax_scut.set_title('Scutellum Height - compare with tripod events above', fontsize=11)
ax_scut.grid(True, alpha=0.3)

plt.tight_layout()

# Save
output_path = OUTPUT_DIR / "tripod_scutellum_analysis.pdf"
fig.savefig(output_path, format='pdf', dpi=300, bbox_inches='tight')
print(f"Saved to: {output_path}")

plt.show()

print("\nLegend:")
print("  Green ^ markers = tripod swing midpoints (expect body HIGH)")
print("  Orange v markers = tripod transitions (expect body LOW)")
print("  If hypothesis is correct, green markers should be near scutellum peaks")

In [ ]:
# ============================================================
# PHASE-AVERAGED ANALYSIS: Scutellum ΔZ across tripod cycle
# ============================================================
# Phase in radians: 0 = swing start, π = mid-swing, 2π = next swing start
# ΔZ = deviation from cycle minimum (removes speed-dependent baseline)

def extract_tripod_cycles_full(tripod_state, scutellum, swing_label):
    """
    Extract scutellum values for full tripod cycles (swing start to swing start).
    Returns phase in radians (0 to 2π) and delta Z (deviation from cycle min).
    """
    in_swing = (tripod_state == swing_label)
    
    # Find swing bout starts
    diff = np.diff(in_swing.astype(int))
    swing_starts = np.where(diff == 1)[0] + 1
    
    if in_swing[0]:
        swing_starts = np.concatenate([[0], swing_starts])
    
    cycles = []
    for i in range(len(swing_starts) - 1):
        start = swing_starts[i]
        end = swing_starts[i + 1]
        
        if end - start < 10:  # Skip very short cycles
            continue
        
        # Extract scutellum for this cycle
        cycle_scutellum = scutellum[start:end]
        
        # Compute delta Z (deviation from cycle minimum)
        cycle_min = np.min(cycle_scutellum)
        delta_z = cycle_scutellum - cycle_min
        
        # Phase in radians: 0 to 2π
        cycle_phase = np.linspace(0, 2 * np.pi, len(delta_z))
        
        cycles.append({
            'phase': cycle_phase,
            'delta_z': delta_z,
            'raw_z': cycle_scutellum,
            'amplitude': np.max(delta_z),  # Peak-to-trough amplitude
            'duration_frames': end - start
        })
    
    return cycles

# Extract full cycles for both tripods
left_cycles = extract_tripod_cycles_full(tripod_state, scutellum_smooth, 'left_swing')
right_cycles = extract_tripod_cycles_full(tripod_state, scutellum_smooth, 'right_swing')

print(f"Found {len(left_cycles)} left tripod cycles, {len(right_cycles)} right tripod cycles")

# Interpolate all cycles to common phase bins (in radians)
n_bins = 50
phase_bins = np.linspace(0, 2 * np.pi, n_bins)

def interpolate_cycles_radians(cycles, phase_bins):
    """Interpolate all cycles to common phase bins."""
    interpolated = []
    amplitudes = []
    for c in cycles:
        interp_delta_z = np.interp(phase_bins, c['phase'], c['delta_z'])
        interpolated.append(interp_delta_z)
        amplitudes.append(c['amplitude'])
    return np.array(interpolated), np.array(amplitudes)

left_interp, left_amps = interpolate_cycles_radians(left_cycles, phase_bins) if left_cycles else (np.array([]), np.array([]))
right_interp, right_amps = interpolate_cycles_radians(right_cycles, phase_bins) if right_cycles else (np.array([]), np.array([]))

# Combine both tripods
if len(left_interp) > 0 and len(right_interp) > 0:
    all_interp = np.vstack([left_interp, right_interp])
    all_amps = np.concatenate([left_amps, right_amps])
elif len(left_interp) > 0:
    all_interp, all_amps = left_interp, left_amps
else:
    all_interp, all_amps = right_interp, right_amps

# Compute mean and std of delta Z
mean_delta_z = np.mean(all_interp, axis=0)
std_delta_z = np.std(all_interp, axis=0)

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Phase-averaged delta Z
ax1 = axes[0]
ax1.fill_between(phase_bins, mean_delta_z - std_delta_z, mean_delta_z + std_delta_z,
                 alpha=0.3, color='purple')
ax1.plot(phase_bins, mean_delta_z, 'purple', lw=2, label=f'Mean ΔZ (n={len(all_interp)} cycles)')

# Mark key phases
ax1.axvline(0, color='orange', ls='--', lw=2, alpha=0.7, label='Swing start (0)')
ax1.axvline(np.pi, color='green', ls='--', lw=2, alpha=0.7, label='Mid-swing (π)')
ax1.axvline(2*np.pi, color='orange', ls='--', lw=2, alpha=0.7)

ax1.set_xlabel('Tripod Step Phase (radians)', fontsize=11)
ax1.set_ylabel('ΔZ from cycle minimum (mm)', fontsize=11)
ax1.set_title('Phase-Averaged Scutellum Oscillation\n(ΔZ removes speed-dependent baseline)', 
              fontsize=12, fontweight='bold')
ax1.legend(fontsize=10, loc='upper right')
ax1.grid(True, alpha=0.3)
ax1.set_xlim(0, 2*np.pi)
ax1.set_xticks([0, np.pi/2, np.pi, 3*np.pi/2, 2*np.pi])
ax1.set_xticklabels(['0', 'π/2', 'π', '3π/2', '2π'])

# Find where peak occurs
peak_phase = phase_bins[np.argmax(mean_delta_z)]
peak_val = np.max(mean_delta_z)
ax1.annotate(f'Peak at {peak_phase:.2f} rad\n({np.degrees(peak_phase):.0f}°)', 
             xy=(peak_phase, peak_val),
             xytext=(peak_phase + 0.5, peak_val * 0.8),
             fontsize=10, arrowprops=dict(arrowstyle='->', color='black'))

# Right: Individual cycles overlaid
ax2 = axes[1]
for c in left_cycles[:15]:
    ax2.plot(c['phase'], c['delta_z'], 'coral', alpha=0.3, lw=0.8)
for c in right_cycles[:15]:
    ax2.plot(c['phase'], c['delta_z'], 'steelblue', alpha=0.3, lw=0.8)

ax2.plot(phase_bins, mean_delta_z, 'k-', lw=2.5, label='Mean')
ax2.axvline(np.pi, color='green', ls='--', lw=2, alpha=0.7)
ax2.set_xlabel('Tripod Step Phase (radians)', fontsize=11)
ax2.set_ylabel('ΔZ from cycle minimum (mm)', fontsize=11)
ax2.set_title('Individual Tripod Cycles\n(coral=left tripod, blue=right tripod)', fontsize=12)
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)
ax2.set_xlim(0, 2*np.pi)
ax2.set_xticks([0, np.pi/2, np.pi, 3*np.pi/2, 2*np.pi])
ax2.set_xticklabels(['0', 'π/2', 'π', '3π/2', '2π'])

plt.tight_layout()

# Save
output_path = OUTPUT_DIR / "phase_averaged_scutellum.pdf"
fig.savefig(output_path, format='pdf', dpi=300, bbox_inches='tight')
print(f"\nSaved to: {output_path}")

plt.show()

# Statistical summary
print("\n" + "=" * 60)
print("PHASE-AVERAGED ANALYSIS SUMMARY")
print("=" * 60)
print(f"Mean ΔZ amplitude (peak - trough): {np.mean(all_amps):.4f} ± {np.std(all_amps):.4f} mm")
print(f"Peak ΔZ occurs at phase: {peak_phase:.2f} rad ({np.degrees(peak_phase):.0f}°)")
print(f"  (0 = swing START, π = MID-swing, 2π = next swing start)")

if np.pi * 0.3 < peak_phase < np.pi * 1.3:
    print("\n✓ HYPOTHESIS SUPPORTED: Body height peaks near MID-SWING (π)")
else:
    print(f"\n? Peak is at {np.degrees(peak_phase):.0f}° - check alignment with mid-swing")

In [ ]:
## Some amputee paths 
# JARVIS_PREDICT_DIR = Path("/home/user/src/JARVIS-HybridNet/projects/fly44_l_V4/predictions/predictions3D/Predictions_3D_20260210-063709/")  #WT amputee Session7/2025_10_13_15_02_19/ 0dPA
# JARVIS_PREDICT_DIR = Path("/home/user/src/JARVIS-HybridNet/projects/fly50_V5_male/predictions/predictions3D/Predictions_3D_20260123-093310/")  #WT /courtship_videos/2025_10_20_13_20_04/videos/
#JARVIS_PREDICT_DIR = Path("/home/user/src/JARVIS-HybridNet/projects/fly44_l_V4/predictions/predictions3D/Predictions_3D_20260211-064849/") #Session2/2025_10_08_14_22_43/ 7dPA
# JARVIS_PREDICT_DIR = Path("/home/user/src/JARVIS-HybridNet/projects/fly44_l_V4/predictions/predictions3D/Predictions_3D_20260211-085656/") #Session2/2025_10_08_14_33_23/ 7dPA
# JARVIS_PREDICT_DIR = Path("/home/user/src/JARVIS-HybridNet/projects/fly44_l_V4/predictions/predictions3D/Predictions_3D_20260211-130159/")  #Session2/2025_10_08_14_55_58/ 7dPA
# JARVIS_PREDICT_DIR = Path("/home/user/src/JARVIS-HybridNet/projects/fly44_l_V4/predictions/predictions3D/Predictions_3D_20260211-164547/") #Session2/2025_10_08_15_07_37/ 7dPA
# JARVIS_PREDICT_DIR = Path("/home/user/src/JARVIS-HybridNet/projects/fly44_l_V4/predictions/predictions3D/Predictions_3D_20260211-195348/") #Session2/2025_10_08_15_32_54/ 7dPA
# JARVIS_PREDICT_DIR = Path("/home/user/src/JARVIS-HybridNet/projects/fly44_l_V4/predictions/predictions3D/Predictions_3D_20260211-233843/") #Session2/2025_10_08_15_45_31 7dPA"/home/user/src/JARVIS-HybridNet/projects/fly44_l_V4/predictions/predictions3D/Predictions_3D_20260210-063709/"  #Session7/2025_10_13_15_02_19/      0dPA
# "/home/user/src/JARVIS-HybridNet/projects/fly44_l_V4/predictions/predictions3D/Predictions_3D_20260212-032455/"  #Session2/2025_10_08_16_35_26/ 0dPA
# "/home/user/src/JARVIS-HybridNet/projects/fly44_l_V4/predictions/predictions3D/Predictions_3D_20260213-081351/"  #Session2/2025_10_08_17_21_07/ 0dPA
# "/home/user/src/JARVIS-HybridNet/projects/fly44_l_V4/predictions/predictions3D/Predictions_3D_20260212-061802/" # Session2/2025_10_08_17_55_41/ 0dPA
# "/home/user/src/JARVIS-HybridNet/projects/fly44_l_V4/predictions/predictions3D/Predictions_3D_20260212-090838/" #Session2/2025_10_08_18_06_20/ 0dPA
# "/home/user/src/JARVIS-HybridNet/projects/fly44_l_V4/predictions/predictions3D/Predictions_3D_20260212-122657/"  # Session2/2025_10_08_18_33_14/ 0dPA
# "/home/user/src/JARVIS-HybridNet/projects/fly44_l_V4/predictions/predictions3D/Predictions_3D_20260212-141448/"  #Session2/2025_10_08_18_44_28/ 0dPA
# "/home/user/src/JARVIS-HybridNet/projects/fly44_l_V4/predictions/predictions3D/Predictions_3D_20260212-165505/"  #Session2/2025_10_08_18_54_48/ 0dPA
# "/home/user/src/JARVIS-HybridNet/projects/fly44_l_V4/predictions/predictions3D/Predictions_3D_20260212-203919/"  #Session2/2025_10_08_19_06_09/ 0dPA

# Paths & loading

In [ ]:
from pathlib import Path
import csv
import json
import yaml
from typing import List, Dict, Tuple
from matplotlib.collections import LineCollection
import matplotlib as mpl
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import scipy
from scipy.signal import hilbert, butter, filtfilt, find_peaks, welch
from scipy.ndimage import uniform_filter1d
from scipy.stats import pearsonr, spearmanr, linregress, sem, circmean, circstd
from datetime import datetime



In [ ]:
# === SET YOUR DATA PATH HERE ===
# JARVIS_PREDICT_DIR = Path("/home/user/src/JARVIS-HybridNet/projects/fly50_V5/predictions/predictions3D/Predictions_3D_20260202-171900/") #WT male S6
#JARVIS_PREDICT_DIR = Path("/home/user/src/JARVIS-HybridNet/projects/fly50_V5/predictions/predictions3D/Predictions_3D_20260203-103416/") #WT male S1
#JARVIS_PREDICT_DIR = Path("/home/user/src/JARVIS-HybridNet/projects/fly50_V5/predictions/predictions3D/Predictions_3D_20260203-164328/") #WT male S5
#JARVIS_PREDICT_DIR = Path("/home/user/src/JARVIS-HybridNet/projects/fly50_V5/predictions/predictions3D/Predictions_3D_20260204-062628/") #WT female S7 
#JARVIS_PREDICT_DIR = Path("/home/user/src/JARVIS-HybridNet/projects/fly50_V5/predictions/predictions3D/Predictions_3D_20260205-111851/") #WT male S7
# JARVIS_PREDICT_DIR = Path("/home/user/src/JARVIS-HybridNet/projects/fly50_V5/predictions/predictions3D/Predictions_3D_20260209-094844/")  #WT female S5   
#Session10 
# JARVIS_PREDICT_DIR = Path("/home/user/src/JARVIS-HybridNet/projects/fly50_V5/predictions/predictions3D/Predictions_3D_20260331-145742")  #WT male Session10/2026_03_02_12_54_26   
# JARVIS_PREDICT_DIR = Path("/home/user/src/JARVIS-HybridNet/projects/fly50_V5/predictions/predictions3D/Predictions_3D_20260331-200523")  #WT male Session10/2026_03_02_13_11_00   
# JARVIS_PREDICT_DIR = Path("/home/user/src/JARVIS-HybridNet/projects/fly50_V5/predictions/predictions3D/Predictions_3D_20260401-012417")  #WT male Session10/2026_03_02_13_27_03
JARVIS_PREDICT_DIR = Path("/home/user/src/JARVIS-HybridNet/projects/fly50_V5/predictions/predictions3D/Predictions_3D_20260401-062914")  #WT male Session10/2026_03_02_13_39_54    
# JARVIS_PREDICT_DIR = Path("/home/user/src/JARVIS-HybridNet/projects/fly50_V5/predictions/predictions3D/Predictions_3D_20260401-115616")  #WT male Session10/2026_03_02_14_47_43    

#COURTSHIP VIDEOS

# JARVIS_PREDICT_DIR = Path("/home/user/src/JARVIS-HybridNet/projects/merge_courtship_V3/predictions/predictions3D/Predictions_3D_20260319-113115/")  #courtship_videos/2025_10_20_13_20_04   
# JARVIS_PREDICT_DIR = Path("/home/user/src/JARVIS-HybridNet/projects/merge_courtship_V3/predictions/predictions3D/Predictions_3D_20260320-181220/") #courtship_videos/2025_10_20_13_57_59/    
# JARVIS_PREDICT_DIR = Path("/home/user/src/JARVIS-HybridNet/projects/merge_courtship_V3/predictions/predictions3D/Predictions_3D_20260321-003902/") #2025_10_20_13_47_39    
# JARVIS_PREDICT_DIR = Path("/home/user/src/JARVIS-HybridNet/projects/merge_courtship_V3/predictions/predictions3D/Predictions_3D_20260321-053924/") #courtship_videos/2025_10_20_13_33_37/   

DATA_3D_PATH = Path(JARVIS_PREDICT_DIR / "data3D.csv")

# Load data
print(f"Loading data from: {DATA_3D_PATH}")
df_full = pd.read_csv(DATA_3D_PATH, skiprows=[1], low_memory=False)
df_full = df_full.iloc[:-1].reset_index(drop=True)  # Drop incomplete last row
print(f"Loaded {len(df_full)} rows")
# Output directory
OUTPUT_DIR = Path(JARVIS_PREDICT_DIR / "standing_analysis")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Load fly_id from info.yaml
info_yaml_path = JARVIS_PREDICT_DIR / "info.yaml"
with open(info_yaml_path, 'r') as f:
    info = yaml.safe_load(f)
recording_path = Path(info['recording_path'].rstrip('/'))
fly_id = f"{recording_path.parent.name}/{recording_path.name}"
print(f"Fly ID: {fly_id}")

## Overall testing of trajectories

In [ ]:
# # --- USER CONFIGURATION ---
# # Select which points to plot (use any keypoint name from the list above)
# CONFIDENCE_THRESHOLD = 0.85
# points_to_plot = [
#     "T1L_TaTip",  # Front left tarsus
#     "T2R_TaTip",
#     # "WingR_V12",
#     # "WingR_V13",
#     # "WingL_V12",
#     # "WingL_V13",
#     "T3R_TaTip",
#     "T1R_TaTip",  # Front right tarsus
#     "T2L_TaTip",  # Middle left tarsus
#     "T3L_TaTip", # Middle right tarsus
#     "Abd_tip"
#      ]
# FRAME_START=0
# FRAME_END= 3035
# # JARVIS scale factor (JARVIS outputs are 10x actual mm)
# JARVIS_SCALE = 10.0

# # Optional: custom colors (if not specified, colors are auto-generated)
# # Format: {"keypoint_name": "#hexcolor"}
# CUSTOM_COLORS = {
#     # Tarsus tips - left=warm, right=cool
#     # "T1L_TaTip": "#E63946",  "T1R_TaTip": "#457B9D",
#     "T2L_TaTip": "#F4A261",  "T2R_TaTip": "#2A9D8F", 
#     "T3L_TaTip": "#9B2226",  "T3R_TaTip": "#1D3557",
# }
# # --------------------------

# # Generate colors for any keypoints not in CUSTOM_COLORS
# def get_color(kp_name, idx, total):
#     if kp_name in CUSTOM_COLORS:
#         return CUSTOM_COLORS[kp_name]
#     # Auto-generate using a colormap
#     cmap = plt.cm.get_cmap('tab20')
#     return cmap(idx / max(total - 1, 1))

# def load_3d_data(csv_path: Path, frame_start: int, frame_end: int):
#     """
#     Load 3D keypoint data from JARVIS CSV format.
#     """
#     print(f"Loading 3D data from {csv_path}...")
#     print(f"  Frame range: {frame_start} to {frame_end}")
    
#     # Read header to get keypoint names
#     header_df = pd.read_csv(csv_path, nrows=0)
#     all_cols = header_df.columns.tolist()
    
#     # Get unique keypoint names (each appears 4 times: x, y, z, confidence)
#     keypoint_names = []
#     for i in range(0, len(all_cols), 4):
#         keypoint_names.append(all_cols[i])
    
#     print(f"  Found {len(keypoint_names)} keypoints")
    
#     # Load only the rows we need (skip header rows, then select frame range)
#     df1 = df_full.iloc[frame_start:frame_end + 1].reset_index(drop=True)
    
#     print(f"  Loaded {len(df1)} frames")
#     return df1, keypoint_names

# # Load 3D data
# df1, keypoint_names = load_3d_data(DATA_3D_PATH, FRAME_START, FRAME_END)
# # Extract data from keypoints 
# def extract_keypoint_xyz(df1, keypoint_name):
#     """Extract x, y, z coordinates for a specific keypoint."""
#     cols = df1.columns.tolist()
    
#     try:
#         start_idx = cols.index(keypoint_name)
#     except ValueError:
#         raise ValueError(f"Keypoint '{keypoint_name}' not found in data")
    
#     x = df1.iloc[:, start_idx].values.astype(float)/JARVIS_SCALE
#     y = df1.iloc[:, start_idx + 1].values.astype(float)/JARVIS_SCALE
#     z = df1.iloc[:, start_idx + 2].values.astype(float)/JARVIS_SCALE

#     return np.column_stack([x, y, z])

# def extract_keypoint_confidence(df1, keypoint_name):
#     """Extract confidence values for a specific keypoint."""
#     cols = df1.columns.tolist()
#     start_idx = cols.index(keypoint_name)
#     return df1.iloc[:, start_idx + 3].values.astype(float)

# # Extract world coordinates for selected points 
# selected_trajectories = []
# selected_names = []
# selected_colors = []
# selected_confidences = []

# for idx, kp_name in enumerate(points_to_plot):
#     if kp_name not in keypoint_names:
#         print(f"Warning: '{kp_name}' not found in keypoints, skipping.")
#         continue
#     world_pos = extract_keypoint_xyz(df1, kp_name)
#     conf = extract_keypoint_confidence(df1, kp_name)
#     selected_trajectories.append(world_pos)
#     selected_names.append(kp_name)
#     selected_colors.append(get_color(kp_name, idx, len(points_to_plot)))
#     selected_confidences.append(conf)

# # Create time axis (frame numbers)
# n_frames = selected_trajectories[0].shape[0]
# frames = np.arange(FRAME_START, FRAME_START + n_frames)

# # Plot X, Y, Z + Confidence
# fig, axes = plt.subplots(4, 1, figsize=(12, 11), sharex=True,
#                          gridspec_kw={'height_ratios': [1, 1, 1, 1]})

# coord_labels = ['X (mm)', 'Y (mm)', 'Z (mm)']

# for coord_idx, (ax, coord_label) in enumerate(zip(axes[:3], coord_labels)):
#     for traj, name, color in zip(selected_trajectories, selected_names, selected_colors):
#         ax.plot(frames, traj[:, coord_idx], label=name, color=color, lw=1.2, alpha=0.8)
    
#     ax.set_ylabel(coord_label, fontsize=12)
#     ax.grid(True, alpha=0.3)
#     ax.set_xlim(frames[0], frames[-1])

# axes[0].set_title(f"Keypoint Trajectories - Actual mm (Frames {FRAME_START}-{FRAME_END})", 
#                   fontsize=13, fontweight='bold')

# # Confidence subplot
# ax_conf = axes[3]
# for conf, name, color in zip(selected_confidences, selected_names, selected_colors):
#     ax_conf.plot(frames, conf, label=name, color=color, lw=1.0, alpha=0.8)

# ax_conf.axhline(y=CONFIDENCE_THRESHOLD, color='red', linestyle='--', lw=1.5, 
#                 alpha=0.7, label=f'Threshold ({CONFIDENCE_THRESHOLD})')
# ax_conf.set_ylabel('Confidence', fontsize=12)
# ax_conf.set_xlabel('Frame', fontsize=12)
# ax_conf.set_ylim(-0.05, 1.05)
# ax_conf.grid(True, alpha=0.3)
# ax_conf.set_xlim(frames[0], frames[-1])

# # Legend outside the plot (to the right)
# axes[0].legend(loc='upper left', bbox_to_anchor=(1.02, 1), fontsize=9, frameon=True)
# ax_conf.legend(loc='upper left', bbox_to_anchor=(1.02, 1), fontsize=9, frameon=True)

# plt.tight_layout()
# fig.subplots_adjust(right=0.82)

# plt.show()


## Comprehensive Walking Bout Detection


In [ ]:
# ============================================================================
# CONFIGURATION - Adjust these thresholds as needed
# ============================================================================

# PDF configuration for editable output
mpl.rcParams.update({
    'pdf.fonttype': 42,
    'pdf.use14corefonts': True,
    'font.family': 'sans-serif',
    'axes.spines.right': False,
    'axes.spines.top': False,
})

# Data scaling
JARVIS_SCALE = 10.0          # Divide raw values by this to get mm

# Frame rate
FPS = 800                    # Video frame rate

# ============================================================================
# FILTER BEHAVIOR MODES
# ============================================================================
# Filter modes for "soft" filters:
#   "split_bout"      - break bout at violations (original behavior)
#   "exclude_frames"  - mark frames invalid but keep bout together  
#   "ignore"          - ignore the violation entirely

# --- HARD FILTERS (can be enabled/disabled) ---
# Confidence filter
CONFIDENCE_FILTER_ENABLED = True
CONFIDENCE_THRESHOLD = 0.80 # Minimum confidence (0-1) for ALL keypoints
CONFIDENCE_GAP_BRIDGE = 15   # Bridge confidence gaps shorter than this many frames (0 = no bridging)

# Upright posture filter (Scutellum Z > all leg tips)
UPRIGHT_FILTER_ENABLED = True

# --- SOFT FILTERS (enable/disable + mode) ---
# Floor Z filter
FLOOR_Z_FILTER_ENABLED = True
FLOOR_Z_FILTER_MODE = "split_bout"  # "split_bout", "exclude_frames", "ignore"
FLOOR_Z_THRESHOLD = 0.40                 # mm - minimum Z for leg tips

# Y-wall upper boundary filter (Y >= Y_WALL_MAX - margin)
Y_WALL_UPPER_ENABLED = True
Y_WALL_UPPER_MODE = "split_bout"        # "split_bout", "exclude_frames", "ignore"
Y_WALL_MAX = 5.4                      # mm - upper wall boundary
Y_WALL_UPPER_MARGIN = 0.0              # mm

# Y-wall lower boundary filter (Y <= Y_WALL_MIN + margin)
Y_WALL_LOWER_ENABLED = True
Y_WALL_LOWER_MODE = "split_bout"    # "split_bout", "exclude_frames", "ignore"
Y_WALL_MIN = 0                        # mm - lower wall boundary
Y_WALL_LOWER_MARGIN = 0.0               # mm (exclude if Y <= 0.1)

# X-wall boundary filter (front/back legs touching X limits of the chamber)
# Checks T1L, T1R, T3L, T3R only
X_WALL_ENABLED = True
X_WALL_MODE = "split_bout"             # "split_bout", "exclude_frames", "ignore"
X_WALL_MARGIN = 0.0                    # mm - margin from arena X edges
X_WALL_TIPS = ["T1L_TaTip", "T1R_TaTip", "T3L_TaTip", "T3R_TaTip"]  # Front + back legs only
# X_WALL_TIPS = ["T1R_TaTip", "T3L_TaTip", "T3R_TaTip"]  # Front + back legs only

# Prolonged immobility filter
IMMOBILITY_FILTER_ENABLED = True        
IMMOBILITY_FILTER_MODE = "split_bout"   # "split_bout", "exclude_frames", "ignore"
MAX_STATIONARY_FRAMES = 25         # Break if stationary for > this many frames
STATIONARY_SPEED_THRESHOLD = 5       # mm/s - speeds below this = stationary

# --- WALKING VALIDATION ---
MIN_WALKING_CYCLES = 2       # Minimum swing phases per leg
MIN_DISTANCE_MM = 5        # Minimum total path distance in mm
MAX_IMMOBILITY_FRAMES = 100  # Maximum gap to bridge (immobility tolerance)
MIN_BOUT_FRAMES = 100        # Minimum bout duration in frames

# Swing detection parameters
MAX_SWING_DURATION = 35      # Maximum frames for a swing phase
SWING_PROMINENCE = 0.05      # Peak prominence threshold (mm)

# Straightness filter
ENFORCE_STRAIGHTNESS = False  # Toggle: if True, only keep straight bouts
STRAIGHTNESS_THRESHOLD = 0.2  # net_displacement/total_distance ratio

# --- GAIT PHASE DIAGRAM ---
STANCE_Z_THRESHOLD = 0.55    # mm - leg is in stance when Z <= this
SHOW_GAIT_DIAGRAM = False    # Add gait phase diagram to bout figures

# --- DIAGNOSTICS ---
SHOW_BOUNDARY_DIAGNOSTICS = True   # Print which filters fail at bout boundaries
BOUNDARY_DIAGNOSTIC_FRAMES = 50    # How many frames before/after bout to analyze

# Arena dimensions (rectangular chamber)
ARENA_X_MM = 23.5            # Arena length in X (mm)
ARENA_Y_MM = 5.5             # Arena width in Y (mm)

# Visualization
PDF_DPI = 300                # Output resolution

# Keypoint names
SCUTELLUM = "Scutellum"
LEG_TIPS = ["T1L_TaTip", "T1R_TaTip", "T2L_TaTip", "T2R_TaTip", "T3L_TaTip", "T3R_TaTip"] # All leg tips
# LEG_TIPS = ["T1R_TaTip", "T2L_TaTip", "T2R_TaTip", "T3L_TaTip", "T3R_TaTip"] # Amputee L
#LEG_TIPS = ["T1L_TaTip", "T1R_TaTip", "T2L_TaTip", "T2R_TaTip", "T3L_TaTip", "T3R_TaTip"] # Amputee R

def save_run_config(output_dir, data_3d_path, diagnostics=None, frame_start=None, frame_end=None):
    """
    Save all run configuration parameters to a YAML file in the output directory.
    """
    config = {
        'run_timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        'data_3d_path': str(data_3d_path),
        'frame_range': {
            'frame_start': frame_start,
            'frame_end': frame_end,
        },
        'data_scaling': {
            'jarvis_scale': JARVIS_SCALE,
        },
        'frame_rate_fps': FPS,
        'arena': {
            'x_mm': ARENA_X_MM,
            'y_mm': ARENA_Y_MM,
        },
        'filters': {
            'confidence': {
                'enabled': CONFIDENCE_FILTER_ENABLED,
                'threshold': CONFIDENCE_THRESHOLD,
                'gap_bridge_frames': CONFIDENCE_GAP_BRIDGE,
            },
            'upright': {
                'enabled': UPRIGHT_FILTER_ENABLED,
            },
            'floor_z': {
                'enabled': FLOOR_Z_FILTER_ENABLED,
                'mode': FLOOR_Z_FILTER_MODE,
                'threshold_mm': FLOOR_Z_THRESHOLD,
            },
            'y_wall_upper': {
                'enabled': Y_WALL_UPPER_ENABLED,
                'mode': Y_WALL_UPPER_MODE,
                'y_max_mm': Y_WALL_MAX,
                'margin_mm': Y_WALL_UPPER_MARGIN,
            },
            'y_wall_lower': {
                'enabled': Y_WALL_LOWER_ENABLED,
                'mode': Y_WALL_LOWER_MODE,
                'y_min_mm': Y_WALL_MIN,
                'margin_mm': Y_WALL_LOWER_MARGIN,
            },
            'x_wall': {
                'enabled': X_WALL_ENABLED,
                'mode': X_WALL_MODE,
                'margin_mm': X_WALL_MARGIN,
                'leg_tips_checked': X_WALL_TIPS,
            },
            'immobility': {
                'enabled': IMMOBILITY_FILTER_ENABLED,
                'mode': IMMOBILITY_FILTER_MODE,
                'max_stationary_frames': MAX_STATIONARY_FRAMES,
                'speed_threshold_mm_s': STATIONARY_SPEED_THRESHOLD,
            },
        },
        'walking_validation': {
            'min_walking_cycles': MIN_WALKING_CYCLES,
            'min_distance_mm': MIN_DISTANCE_MM,
            'max_immobility_gap_frames': MAX_IMMOBILITY_FRAMES,
            'min_bout_frames': MIN_BOUT_FRAMES,
            'max_swing_duration': MAX_SWING_DURATION,
            'swing_prominence': SWING_PROMINENCE,
            'enforce_straightness': ENFORCE_STRAIGHTNESS,
            'straightness_threshold': STRAIGHTNESS_THRESHOLD,
        },
        'gait_phase': {
            'stance_z_threshold_mm': STANCE_Z_THRESHOLD,
        },
        'keypoints': {
            'leg_tips': LEG_TIPS,
            'scutellum': SCUTELLUM,
        },
    }

    if diagnostics is not None:
        config['results_summary'] = {
            'total_frames': diagnostics.get('total_frames'),
            'n_valid_bouts': diagnostics.get('n_valid_bouts'),
            'n_rejected_bouts': diagnostics.get('n_rejected_bouts'),
            'confidence_pass_pct': round(diagnostics.get('confidence_pass_pct', 0), 2),
            'upright_pass_pct': round(diagnostics.get('upright_pass_pct', 0), 2),
            'combined_pass_pct': round(diagnostics.get('combined_pass_pct', 0), 2),
        }

    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    config_path = output_dir / 'run_config.yml'
    with open(config_path, 'w') as f:
        yaml.dump(config, f, default_flow_style=False, sort_keys=False)
    print(f"Run config saved: {config_path}")


print("Configuration loaded.")
print(f"Arena: {ARENA_X_MM} x {ARENA_Y_MM} mm (rectangular)")
print(f"Output directory: {OUTPUT_DIR.absolute()}")
print(f"Data 3D directory: {DATA_3D_PATH.absolute()}")
print(f"\nFilter settings:")
print(f"  Confidence filter: {'ENABLED' if CONFIDENCE_FILTER_ENABLED else 'DISABLED'} (threshold: {CONFIDENCE_THRESHOLD}, gap bridge: {CONFIDENCE_GAP_BRIDGE} frames)")
print(f"  Upright filter: {'ENABLED' if UPRIGHT_FILTER_ENABLED else 'DISABLED'}")
print(f"  Floor Z filter: {'ENABLED' if FLOOR_Z_FILTER_ENABLED else 'DISABLED'} (mode: {FLOOR_Z_FILTER_MODE}, threshold: {FLOOR_Z_THRESHOLD}mm)")
print(f"  Y-wall upper filter: {'ENABLED' if Y_WALL_UPPER_ENABLED else 'DISABLED'} (mode: {Y_WALL_UPPER_MODE})")
print(f"  Y-wall lower filter: {'ENABLED' if Y_WALL_LOWER_ENABLED else 'DISABLED'} (mode: {Y_WALL_LOWER_MODE})")
print(f"  X-wall filter: {'ENABLED' if X_WALL_ENABLED else 'DISABLED'} (mode: {X_WALL_MODE}, legs: {X_WALL_TIPS})")
print(f"  Immobility filter: {'ENABLED' if IMMOBILITY_FILTER_ENABLED else 'DISABLED'} (mode: {IMMOBILITY_FILTER_MODE})")
print(f"  Gait diagram: {'ENABLED' if SHOW_GAIT_DIAGRAM else 'DISABLED'}")
print(f"  Boundary diagnostics: {'ENABLED' if SHOW_BOUNDARY_DIAGNOSTICS else 'DISABLED'}")

In [ ]:
# ============================================================================
# DATA EXTRACTION FUNCTIONS
# ============================================================================

def get_all_keypoint_names(df):
    """Get all unique keypoint names from dataframe columns."""
    cols = df.columns.tolist()
    # Each keypoint has 4 columns: name, name.1, name.2, name.3 (x, y, z, conf)
    kp_names = []
    seen = set()
    for col in cols:
        base_name = col.split('.')[0] if '.' in col else col
        if base_name not in seen:
            seen.add(base_name)
            kp_names.append(base_name)
    return kp_names


def extract_xyzc(df, kp_name, scale=JARVIS_SCALE):
    """
    Extract x, y, z coordinates and confidence for a keypoint.
    
    Args:
        df: DataFrame with tracking data
        kp_name: Keypoint name (e.g., "T1L_TaTip")
        scale: JARVIS_SCALE factor (default 10.0)
    
    Returns:
        x, y, z: Arrays of coordinates in mm
        conf: Array of confidence values (0-1)
    """
    cols = df.columns.tolist()
    start_idx = cols.index(kp_name)
    
    x = df.iloc[:, start_idx].values.astype(float) / scale
    y = df.iloc[:, start_idx + 1].values.astype(float) / scale
    z = df.iloc[:, start_idx + 2].values.astype(float) / scale
    conf = df.iloc[:, start_idx + 3].values.astype(float)  # Confidence NOT scaled
    
    return x, y, z, conf


def extract_all_data(df, scale=JARVIS_SCALE):
    """
    Extract xyz and confidence for all leg tips and scutellum.
    
    Returns:
        leg_tip_data: Dict mapping leg name -> {'x', 'y', 'z', 'conf'}
        scutellum_data: Dict with {'x', 'y', 'z', 'conf'}
        all_keypoint_conf: Dict mapping all kp names -> conf array
    """
    leg_tip_data = {}
    for tip in LEG_TIPS:
        x, y, z, conf = extract_xyzc(df, tip, scale)
        leg_tip_data[tip] = {'x': x, 'y': y, 'z': z, 'conf': conf}
    
    scut_x, scut_y, scut_z, scut_conf = extract_xyzc(df, SCUTELLUM, scale)
    scutellum_data = {'x': scut_x, 'y': scut_y, 'z': scut_z, 'conf': scut_conf}
    
    # Extract confidence for ALL keypoints
    all_kp_names = get_all_keypoint_names(df)
    all_keypoint_conf = {}
    for kp in all_kp_names:
        try:
            _, _, _, conf = extract_xyzc(df, kp, scale)
            all_keypoint_conf[kp] = conf
        except (ValueError, IndexError):
            pass  # Skip if keypoint not found
    
    return leg_tip_data, scutellum_data, all_keypoint_conf


print("Data extraction functions defined.")

In [ ]:
# ============================================================================
# FRAME-LEVEL FILTER FUNCTIONS
# ============================================================================

def bridge_short_gaps(mask, max_gap):
    """
    Bridge short gaps (runs of False) in a boolean mask.
    
    If a run of False values is <= max_gap frames and is bounded by True
    on both sides, set those frames to True.
    
    Args:
        mask: Boolean array (True = passes filter)
        max_gap: Maximum gap length to bridge (0 = no bridging)
    
    Returns:
        Bridged boolean array (copy)
    """
    if max_gap <= 0:
        return mask
    
    bridged = mask.copy()
    in_gap = False
    gap_start = 0
    
    for i in range(len(bridged)):
        if not bridged[i]:
            if not in_gap:
                gap_start = i
                in_gap = True
        else:
            if in_gap:
                gap_len = i - gap_start
                if gap_len <= max_gap and gap_start > 0:
                    bridged[gap_start:i] = True
                in_gap = False
    
    return bridged


def apply_comprehensive_walking_filter(
    df,
    scale=JARVIS_SCALE,
    # Confidence filter
    confidence_filter_enabled=CONFIDENCE_FILTER_ENABLED,
    confidence_threshold=CONFIDENCE_THRESHOLD,
    confidence_gap_bridge=CONFIDENCE_GAP_BRIDGE,
    # Upright filter
    upright_filter_enabled=UPRIGHT_FILTER_ENABLED,
    # Floor Z filter
    floor_z_filter_enabled=FLOOR_Z_FILTER_ENABLED,
    floor_z_filter_mode=FLOOR_Z_FILTER_MODE,
    floor_z_threshold=FLOOR_Z_THRESHOLD,
    # Y-wall upper filter
    y_wall_upper_enabled=Y_WALL_UPPER_ENABLED,
    y_wall_upper_mode=Y_WALL_UPPER_MODE,
    y_wall_max=Y_WALL_MAX,
    y_wall_upper_margin=Y_WALL_UPPER_MARGIN,
    # Y-wall lower filter
    y_wall_lower_enabled=Y_WALL_LOWER_ENABLED,
    y_wall_lower_mode=Y_WALL_LOWER_MODE,
    y_wall_min=Y_WALL_MIN,
    y_wall_lower_margin=Y_WALL_LOWER_MARGIN,
    # X-wall filter
    x_wall_enabled=X_WALL_ENABLED,
    x_wall_mode=X_WALL_MODE,
    x_wall_margin=X_WALL_MARGIN,
    x_wall_tips=X_WALL_TIPS,
    arena_x_mm=ARENA_X_MM,
):
    """
    Apply comprehensive walking filter with all frame-level criteria.
    
    All filters can be enabled/disabled independently.
    "Soft" filters (floor, y-wall, x-wall) have modes: "split_bout", "exclude_frames", "ignore"
    
    Returns:
        valid_mask: Boolean array (True = valid walking frame for bout detection)
        leg_tip_data: Dict of leg data
        scutellum_data: Dict of scutellum data  
        filter_masks: Dict of individual filter masks for downstream use
        diagnostics: Dict with per-criterion pass rates
    """
    n_frames = len(df)
    
    # Extract all data
    leg_tip_data, scutellum_data, all_keypoint_conf = extract_all_data(df, scale)
    
    # =========================================================================
    # COMPUTE INDIVIDUAL FILTER MASKS
    # =========================================================================
    
    # Criterion 1: Confidence filter (ALL keypoints >= threshold)
    confidence_mask = np.ones(n_frames, dtype=bool)
    if confidence_filter_enabled:
        for kp_name, conf_arr in all_keypoint_conf.items():
            confidence_mask &= (conf_arr >= confidence_threshold)
        
        # Bridge short confidence gaps to prevent brief dips from splitting bouts
        if confidence_gap_bridge > 0:
            n_bridged = int((~confidence_mask).sum())
            confidence_mask = bridge_short_gaps(confidence_mask, confidence_gap_bridge)
            n_bridged_after = int((~confidence_mask).sum())
            n_recovered = n_bridged - n_bridged_after
    
    # Criterion 2: Upright posture (scutellum Z > all leg tips)
    upright_mask = np.ones(n_frames, dtype=bool)
    if upright_filter_enabled:
        for tip in LEG_TIPS:
            upright_mask &= (scutellum_data['z'] > leg_tip_data[tip]['z'])
    
    # Criterion 3: Floor Z violations (any leg Z < threshold)
    floor_violation_mask = np.zeros(n_frames, dtype=bool)
    if floor_z_filter_enabled:
        for tip in LEG_TIPS:
            floor_violation_mask |= (leg_tip_data[tip]['z'] < floor_z_threshold)
    
    # Criterion 4: Y-wall UPPER violations (any leg Y >= max - margin)
    y_wall_upper_violation_mask = np.zeros(n_frames, dtype=bool)
    if y_wall_upper_enabled:
        for tip in LEG_TIPS:
            y = leg_tip_data[tip]['y']
            y_wall_upper_violation_mask |= (y >= y_wall_max - y_wall_upper_margin)
    
    # Criterion 5: Y-wall LOWER violations (any leg Y <= min + margin)
    y_wall_lower_violation_mask = np.zeros(n_frames, dtype=bool)
    if y_wall_lower_enabled:
        for tip in LEG_TIPS:
            y = leg_tip_data[tip]['y']
            y_wall_lower_violation_mask |= (y <= y_wall_min + y_wall_lower_margin)
    
    # Criterion 6: X-wall violations (front/back leg X at arena edges)
    x_wall_violation_mask = np.zeros(n_frames, dtype=bool)
    if x_wall_enabled:
        for tip in x_wall_tips:
            x = leg_tip_data[tip]['x']
            x_wall_violation_mask |= (x >= arena_x_mm - x_wall_margin)  # upper X edge
            x_wall_violation_mask |= (x <= x_wall_margin)               # lower X edge (0)
    
    # =========================================================================
    # BUILD VALID MASK FOR BOUT DETECTION
    # =========================================================================
    # Start with hard filters (confidence, upright)
    valid_mask = confidence_mask & upright_mask
    
    # Apply soft filters based on mode
    if floor_z_filter_enabled and floor_z_filter_mode != "ignore":
        valid_mask &= ~floor_violation_mask
    
    if y_wall_upper_enabled and y_wall_upper_mode != "ignore":
        valid_mask &= ~y_wall_upper_violation_mask
    
    if y_wall_lower_enabled and y_wall_lower_mode != "ignore":
        valid_mask &= ~y_wall_lower_violation_mask
    
    if x_wall_enabled and x_wall_mode != "ignore":
        valid_mask &= ~x_wall_violation_mask
    
    # =========================================================================
    # STORE FILTER MASKS FOR DOWNSTREAM USE
    # =========================================================================
    filter_masks = {
        'confidence': confidence_mask,              # True = passes confidence
        'upright': upright_mask,                    # True = passes upright
        'floor_violation': floor_violation_mask,    # True = VIOLATION (Z < threshold)
        'y_wall_upper_violation': y_wall_upper_violation_mask,  # True = VIOLATION
        'y_wall_lower_violation': y_wall_lower_violation_mask,  # True = VIOLATION
        'x_wall_violation': x_wall_violation_mask,              # True = VIOLATION
    }
    
    # =========================================================================
    # DIAGNOSTICS
    # =========================================================================
    diagnostics = {
        'total_frames': n_frames,
        'confidence_pass': int(confidence_mask.sum()),
        'upright_pass': int(upright_mask.sum()),
        'floor_violations': int(floor_violation_mask.sum()),
        'y_wall_upper_violations': int(y_wall_upper_violation_mask.sum()),
        'y_wall_lower_violations': int(y_wall_lower_violation_mask.sum()),
        'x_wall_violations': int(x_wall_violation_mask.sum()),
        'combined_pass': int(valid_mask.sum()),
        'confidence_pass_pct': 100 * confidence_mask.sum() / n_frames,
        'upright_pass_pct': 100 * upright_mask.sum() / n_frames,
        'floor_violation_pct': 100 * floor_violation_mask.sum() / n_frames,
        'y_wall_upper_violation_pct': 100 * y_wall_upper_violation_mask.sum() / n_frames,
        'y_wall_lower_violation_pct': 100 * y_wall_lower_violation_mask.sum() / n_frames,
        'x_wall_violation_pct': 100 * x_wall_violation_mask.sum() / n_frames,
        'combined_pass_pct': 100 * valid_mask.sum() / n_frames,
        'n_keypoints_checked': len(all_keypoint_conf),
        # Filter settings
        'confidence_filter_enabled': confidence_filter_enabled,
        'confidence_gap_bridge': confidence_gap_bridge,
        'upright_filter_enabled': upright_filter_enabled,
        'floor_z_filter_enabled': floor_z_filter_enabled,
        'floor_z_filter_mode': floor_z_filter_mode,
        'y_wall_upper_enabled': y_wall_upper_enabled,
        'y_wall_upper_mode': y_wall_upper_mode,
        'y_wall_lower_enabled': y_wall_lower_enabled,
        'y_wall_lower_mode': y_wall_lower_mode,
        'x_wall_enabled': x_wall_enabled,
        'x_wall_mode': x_wall_mode,
    }
    
    # Add bridging stats if applicable
    if confidence_filter_enabled and confidence_gap_bridge > 0:
        diagnostics['confidence_frames_bridged'] = n_recovered
    
    return valid_mask, leg_tip_data, scutellum_data, filter_masks, diagnostics


def diagnose_bout_boundaries(bout_start, bout_end, filter_masks, n_frames_context=BOUNDARY_DIAGNOSTIC_FRAMES):
    """
    Analyze frames around bout boundaries to show which filters fail.
    """
    total_frames = len(filter_masks['confidence'])
    
    pre_start = max(0, bout_start - n_frames_context)
    pre_range = slice(pre_start, bout_start)
    
    post_end = min(total_frames, bout_end + n_frames_context + 1)
    post_range = slice(bout_end + 1, post_end)
    
    pre_failures = {
        'confidence': int((~filter_masks['confidence'][pre_range]).sum()),
        'upright': int((~filter_masks['upright'][pre_range]).sum()),
        'floor': int(filter_masks['floor_violation'][pre_range].sum()),
        'y_wall_upper': int(filter_masks['y_wall_upper_violation'][pre_range].sum()),
        'y_wall_lower': int(filter_masks['y_wall_lower_violation'][pre_range].sum()),
        'x_wall': int(filter_masks['x_wall_violation'][pre_range].sum()),
        'n_frames_analyzed': bout_start - pre_start,
    }
    
    post_failures = {
        'confidence': int((~filter_masks['confidence'][post_range]).sum()),
        'upright': int((~filter_masks['upright'][post_range]).sum()),
        'floor': int(filter_masks['floor_violation'][post_range].sum()),
        'y_wall_upper': int(filter_masks['y_wall_upper_violation'][post_range].sum()),
        'y_wall_lower': int(filter_masks['y_wall_lower_violation'][post_range].sum()),
        'x_wall': int(filter_masks['x_wall_violation'][post_range].sum()),
        'n_frames_analyzed': post_end - bout_end - 1,
    }
    
    pre_max = max(pre_failures.items(), key=lambda x: x[1] if x[0] != 'n_frames_analyzed' else 0)
    post_max = max(post_failures.items(), key=lambda x: x[1] if x[0] != 'n_frames_analyzed' else 0)
    
    return {
        'pre_bout_failures': pre_failures,
        'post_bout_failures': post_failures,
        'primary_cause_before': pre_max[0] if pre_max[1] > 0 else None,
        'primary_cause_after': post_max[0] if post_max[1] > 0 else None,
    }


def print_boundary_diagnostics(bout_idx, diagnostics):
    """Print formatted boundary diagnostics for a bout."""
    pre = diagnostics['pre_bout_failures']
    post = diagnostics['post_bout_failures']
    
    filter_names = ['confidence', 'upright', 'floor', 'y_wall_upper', 'y_wall_lower', 'x_wall']
    
    print(f"\n  Bout {bout_idx} boundary analysis:")
    print(f"    Before bout ({pre['n_frames_analyzed']} frames):")
    for name in filter_names:
        count = pre[name]
        if count > 0:
            marker = " <-- PRIMARY" if diagnostics['primary_cause_before'] == name else ""
            print(f"      {name}: {count} failures{marker}")
    
    if post['n_frames_analyzed'] > 0:
        print(f"    After bout ({post['n_frames_analyzed']} frames):")
        for name in filter_names:
            count = post[name]
            if count > 0:
                marker = " <-- PRIMARY" if diagnostics['primary_cause_after'] == name else ""
                print(f"      {name}: {count} failures{marker}")


print("Frame filter functions defined (with confidence gap bridging, X/Y-wall filters, and boundary diagnostics).")

In [ ]:
# ============================================================================
# BOUT DETECTION WITH GAP BRIDGING
# ============================================================================

def find_contiguous_bouts_with_gap_bridging(
    valid_mask,
    min_frames=MIN_BOUT_FRAMES,
    max_gap=MAX_IMMOBILITY_FRAMES
):
    """
    Find contiguous walking bouts, bridging small gaps (immobility tolerance).
    
    Args:
        valid_mask: Boolean array of valid frames
        min_frames: Minimum bout length
        max_gap: Maximum gap to bridge (immobility tolerance)
    
    Returns:
        bouts: List of (start_idx, end_idx) tuples
    """
    # First, bridge gaps <= max_gap
    bridged = valid_mask.copy()
    
    # Find gaps and bridge them if they're bounded by valid frames
    in_gap = False
    gap_start = 0
    
    for i in range(len(bridged)):
        if not bridged[i]:
            if not in_gap:
                gap_start = i
                in_gap = True
        else:
            if in_gap:
                gap_len = i - gap_start
                # Bridge if gap is small and bounded by valid frames
                if gap_len <= max_gap and gap_start > 0 and bridged[gap_start - 1]:
                    bridged[gap_start:i] = True
                in_gap = False
    
    # Now find contiguous segments
    bouts = []
    in_bout = False
    start = 0
    
    for i, valid in enumerate(bridged):
        if valid and not in_bout:
            start = i
            in_bout = True
        elif not valid and in_bout:
            if i - start >= min_frames:
                bouts.append((start, i - 1))
            in_bout = False
    
    # Handle bout extending to end
    if in_bout and len(bridged) - start >= min_frames:
        bouts.append((start, len(bridged) - 1))
    
    return bouts


def split_bouts_at_violations(
    bouts, 
    leg_tip_data, 
    scutellum_data, 
    filter_masks,
    fps=FPS,
    # Floor filter config
    floor_z_filter_enabled=FLOOR_Z_FILTER_ENABLED,
    floor_z_filter_mode=FLOOR_Z_FILTER_MODE,
    # Y-wall upper config
    y_wall_upper_enabled=Y_WALL_UPPER_ENABLED,
    y_wall_upper_mode=Y_WALL_UPPER_MODE,
    # Y-wall lower config
    y_wall_lower_enabled=Y_WALL_LOWER_ENABLED,
    y_wall_lower_mode=Y_WALL_LOWER_MODE,
    # X-wall config
    x_wall_enabled=X_WALL_ENABLED,
    x_wall_mode=X_WALL_MODE,
    # Immobility config
    immobility_filter_enabled=IMMOBILITY_FILTER_ENABLED,
    immobility_filter_mode=IMMOBILITY_FILTER_MODE,
    max_stationary=MAX_STATIONARY_FRAMES, 
    stationary_thresh=STATIONARY_SPEED_THRESHOLD,
    min_frames=MIN_BOUT_FRAMES
):
    """
    Split bouts at violation points based on filter modes.
    
    Only splits on violations where mode == "split_bout".
    Modes "exclude_frames" and "ignore" do NOT cause splits.
    
    Returns:
        split_bouts: List of (start, end) tuples for valid sub-bouts
        bout_excluded_masks: List of boolean arrays marking excluded frames per bout
    """
    split_bouts_list = []
    bout_excluded_masks = []
    
    for start, end in bouts:
        n = end - start + 1
        
        # Track violations that SPLIT bouts (only mode="split_bout")
        split_violation = np.zeros(n, dtype=bool)
        
        # Track frames to EXCLUDE from analysis (mode="exclude_frames")
        excluded_mask = np.zeros(n, dtype=bool)
        
        # =====================================================================
        # FLOOR VIOLATIONS
        # =====================================================================
        if floor_z_filter_enabled:
            floor_viol = filter_masks['floor_violation'][start:end+1]
            if floor_z_filter_mode == "split_bout":
                split_violation |= floor_viol
            elif floor_z_filter_mode == "exclude_frames":
                excluded_mask |= floor_viol
            # mode="ignore" - do nothing
        
        # =====================================================================
        # Y-WALL UPPER VIOLATIONS
        # =====================================================================
        if y_wall_upper_enabled:
            y_upper_viol = filter_masks['y_wall_upper_violation'][start:end+1]
            if y_wall_upper_mode == "split_bout":
                split_violation |= y_upper_viol
            elif y_wall_upper_mode == "exclude_frames":
                excluded_mask |= y_upper_viol
        
        # =====================================================================
        # Y-WALL LOWER VIOLATIONS
        # =====================================================================
        if y_wall_lower_enabled:
            y_lower_viol = filter_masks['y_wall_lower_violation'][start:end+1]
            if y_wall_lower_mode == "split_bout":
                split_violation |= y_lower_viol
            elif y_wall_lower_mode == "exclude_frames":
                excluded_mask |= y_lower_viol
        
        # =====================================================================
        # X-WALL VIOLATIONS
        # =====================================================================
        if x_wall_enabled:
            x_wall_viol = filter_masks['x_wall_violation'][start:end+1]
            if x_wall_mode == "split_bout":
                split_violation |= x_wall_viol
            elif x_wall_mode == "exclude_frames":
                excluded_mask |= x_wall_viol
        
        # =====================================================================
        # PROLONGED IMMOBILITY
        # =====================================================================
        if immobility_filter_enabled:
            speed, _, _ = compute_instant_speed(scutellum_data, start, end, fps)
            is_stationary = speed < stationary_thresh
            
            # Find runs of stationary frames > threshold
            immobility_viol = np.zeros(n, dtype=bool)
            run_len = 0
            run_start = 0
            for i, stat in enumerate(is_stationary):
                if stat:
                    if run_len == 0:
                        run_start = i
                    run_len += 1
                else:
                    if run_len > max_stationary:
                        immobility_viol[run_start:run_start + run_len] = True
                    run_len = 0
            # Handle run extending to end
            if run_len > max_stationary:
                immobility_viol[run_start:run_start + run_len] = True
            
            if immobility_filter_mode == "split_bout":
                split_violation |= immobility_viol
            elif immobility_filter_mode == "exclude_frames":
                excluded_mask |= immobility_viol
        
        # =====================================================================
        # SPLIT INTO SUB-BOUTS (only at split_violation points)
        # =====================================================================
        valid = ~split_violation
        in_valid = False
        sub_start = 0
        
        for i, v in enumerate(valid):
            if v and not in_valid:
                sub_start = i
                in_valid = True
            elif not v and in_valid:
                if i - sub_start >= min_frames:
                    split_bouts_list.append((start + sub_start, start + i - 1))
                    # Store excluded mask for this sub-bout
                    bout_excluded_masks.append(excluded_mask[sub_start:i].copy())
                in_valid = False
        
        if in_valid and (n - sub_start) >= min_frames:
            split_bouts_list.append((start + sub_start, end))
            bout_excluded_masks.append(excluded_mask[sub_start:].copy())
    
    return split_bouts_list, bout_excluded_masks


def get_excluded_frames_for_bout(start, end, filter_masks,
                                  floor_z_filter_mode=FLOOR_Z_FILTER_MODE,
                                  y_wall_upper_mode=Y_WALL_UPPER_MODE,
                                  y_wall_lower_mode=Y_WALL_LOWER_MODE,
                                  x_wall_mode=X_WALL_MODE,
                                  immobility_filter_mode=IMMOBILITY_FILTER_MODE):
    """
    Compute which frames within a bout should be excluded from analysis.
    
    Frames are excluded if they violate a filter with mode="exclude_frames".
    This is used for computing statistics on "clean" frames only.
    """
    n = end - start + 1
    excluded = np.zeros(n, dtype=bool)
    
    if floor_z_filter_mode == "exclude_frames":
        excluded |= filter_masks['floor_violation'][start:end+1]
    
    if y_wall_upper_mode == "exclude_frames":
        excluded |= filter_masks['y_wall_upper_violation'][start:end+1]
    
    if y_wall_lower_mode == "exclude_frames":
        excluded |= filter_masks['y_wall_lower_violation'][start:end+1]
    
    if x_wall_mode == "exclude_frames":
        excluded |= filter_masks['x_wall_violation'][start:end+1]
    
    return excluded


print("Bout detection functions defined (with mode-aware violation splitting).")

In [ ]:
# ============================================================================
# SWING PHASE DETECTION
# ============================================================================

def detect_swing_phases(
    z_signal,
    max_swing_duration=MAX_SWING_DURATION,
    min_prominence=SWING_PROMINENCE
):
    """
    Detect swing phases as sharp Z peaks.
    
    A swing phase is:
    - A peak in the Z signal (leg lifted off ground)
    - Duration under max_swing_duration frames
    
    Args:
        z_signal: Z coordinate array for one leg tip
        max_swing_duration: Maximum frames for a valid swing
        min_prominence: Minimum peak prominence (mm)
    
    Returns:
        swing_peaks: Array of peak indices
        swing_count: Number of valid swing phases
    """
    # Handle NaN values
    z_clean = z_signal.copy()
    nan_mask = np.isnan(z_clean)
    
    if nan_mask.all():
        return np.array([]), 0
    
    if nan_mask.any():
        valid_idx = np.where(~nan_mask)[0]
        z_clean[nan_mask] = np.interp(
            np.where(nan_mask)[0],
            valid_idx,
            z_clean[valid_idx]
        )
    
    # Light smoothing
    z_smooth = uniform_filter1d(z_clean, size=5)
    
    # Find peaks with constraints
    peaks, properties = find_peaks(
        z_smooth,
        prominence=min_prominence,
        distance=8,  # Minimum distance between peaks
        width=(1, max_swing_duration)  # Width constraint for "sharp" peaks
    )
    
    return peaks, len(peaks)


def count_swing_phases_all_legs(
    leg_tip_data,
    start,
    end,
    max_swing_duration=MAX_SWING_DURATION,
    min_prominence=SWING_PROMINENCE
):
    """
    Count swing phases for all 6 legs.
    
    Args:
        leg_tip_data: Dict of leg data
        start, end: Bout boundaries
        max_swing_duration: Max frames for swing detection
        min_prominence: Peak prominence threshold
    
    Returns:
        per_leg_cycles: Dict of swing counts per leg
        min_cycles: Minimum across all legs
    """
    per_leg_cycles = {}
    
    for tip in LEG_TIPS:
        z_bout = leg_tip_data[tip]['z'][start:end+1]
        _, swing_count = detect_swing_phases(
            z_bout, max_swing_duration, min_prominence
        )
        per_leg_cycles[tip] = swing_count
    
    min_cycles = min(per_leg_cycles.values()) if per_leg_cycles else 0
    
    return per_leg_cycles, min_cycles


print("Swing detection functions defined.")

In [ ]:
# ============================================================================
# DISTANCE AND SPEED COMPUTATION
# ============================================================================

def compute_total_distance(scutellum_data, start, end):
    """
    Compute total path distance traveled using scutellum position.
    
    Uses cumulative displacement (not just start-to-end distance).
    
    Returns:
        total_distance: Total path length in mm
        net_displacement: Straight-line start-to-end distance in mm
    """
    x = scutellum_data['x'][start:end+1]
    y = scutellum_data['y'][start:end+1]
    
    # Handle NaN
    valid = ~np.isnan(x) & ~np.isnan(y)
    if valid.sum() < 2:
        return 0.0, 0.0
    
    x_valid = x[valid]
    y_valid = y[valid]
    
    # Total path distance (cumulative)
    dx = np.diff(x_valid)
    dy = np.diff(y_valid)
    step_distances = np.sqrt(dx**2 + dy**2)
    total_distance = np.sum(step_distances)
    
    # Net displacement (straight line)
    net_displacement = np.sqrt(
        (x_valid[-1] - x_valid[0])**2 + 
        (y_valid[-1] - y_valid[0])**2
    )
    
    return total_distance, net_displacement


def compute_instant_speed(scutellum_data, start, end, fps=FPS):
    """
    Compute instantaneous speed at each frame.
    
    Returns:
        speed: Array of speeds in mm/s (same length as bout)
        mean_speed: Mean speed over bout
        max_speed: Maximum speed
    """
    x = scutellum_data['x'][start:end+1]
    y = scutellum_data['y'][start:end+1]
    
    # Compute displacements
    dx = np.diff(x)
    dy = np.diff(y)
    displacement = np.sqrt(dx**2 + dy**2)
    
    # Speed = displacement * fps
    speed = displacement * fps  # mm/s
    
    # Pad to match original length
    speed = np.concatenate([[speed[0] if len(speed) > 0 else 0], speed])
    
    # Handle NaN
    mean_speed = np.nanmean(speed)
    max_speed = np.nanmax(speed)
    
    return speed, mean_speed, max_speed


print("Distance and speed functions defined.")

In [ ]:
# ============================================================================
# BOUT VALIDATION
# ============================================================================

def validate_walking_bouts(
    bouts,
    leg_tip_data,
    scutellum_data,
    min_cycles=MIN_WALKING_CYCLES,
    min_distance_mm=MIN_DISTANCE_MM,
    max_swing_duration=MAX_SWING_DURATION,
    min_prominence=SWING_PROMINENCE,
    enforce_straightness=ENFORCE_STRAIGHTNESS,
    straightness_threshold=STRAIGHTNESS_THRESHOLD
):
    """
    Validate candidate bouts against walking cycle, distance, and straightness criteria.
    
    Criteria:
    - Each leg must have >= min_cycles swing phases
    - Total path distance >= min_distance_mm
    - Straightness >= threshold (if enforce_straightness is True)
    
    Returns:
        valid_bouts: List of validated bout dicts with metadata
        rejected_bouts: List of rejected bout dicts with rejection reason
    """
    valid_bouts = []
    rejected_bouts = []
    
    for start, end in bouts:
        # Count swing phases for each leg
        per_leg_cycles, min_leg_cycles = count_swing_phases_all_legs(
            leg_tip_data, start, end, max_swing_duration, min_prominence
        )
        
        # Compute distance
        total_distance, net_displacement = compute_total_distance(
            scutellum_data, start, end
        )
        
        # Compute straightness (net_displacement / total_distance)
        straightness = net_displacement / total_distance if total_distance > 0 else 0
        
        bout_info = {
            'start': start,
            'end': end,
            'n_frames': end - start + 1,
            'min_cycles': min_leg_cycles,
            'per_leg_cycles': per_leg_cycles,
            'total_distance_mm': total_distance,
            'net_displacement_mm': net_displacement,
            'straightness': straightness
        }
        
        # Check if ALL legs have enough cycles
        all_legs_pass = all(
            cycles >= min_cycles for cycles in per_leg_cycles.values()
        )
        passes_distance = total_distance >= min_distance_mm
        
        # Check straightness (only if enforced)
        passes_straightness = (not enforce_straightness) or (straightness >= straightness_threshold)
        
        if all_legs_pass and passes_distance and passes_straightness:
            bout_info['valid'] = True
            valid_bouts.append(bout_info)
        else:
            bout_info['valid'] = False
            bout_info['rejection_reason'] = []
            if not all_legs_pass:
                failing_legs = [
                    f"{leg}:{c}" for leg, c in per_leg_cycles.items() 
                    if c < min_cycles
                ]
                bout_info['rejection_reason'].append(
                    f"legs with <{min_cycles} cycles: {failing_legs}"
                )
            if not passes_distance:
                bout_info['rejection_reason'].append(
                    f"distance: {total_distance:.2f}mm < {min_distance_mm}mm"
                )
            if not passes_straightness:
                bout_info['rejection_reason'].append(
                    f"straightness: {straightness:.2f} < {straightness_threshold}"
                )
            rejected_bouts.append(bout_info)
    
    return valid_bouts, rejected_bouts


print("Bout validation functions defined (including straightness filter).")

In [ ]:
# ============================================================================
# MAIN PIPELINE FUNCTION
# ============================================================================

def detect_walking_bouts(
    df,
    scale=JARVIS_SCALE,
    # Hard filter parameters
    confidence_filter_enabled=CONFIDENCE_FILTER_ENABLED,
    confidence_threshold=CONFIDENCE_THRESHOLD,
    confidence_gap_bridge=CONFIDENCE_GAP_BRIDGE,
    upright_filter_enabled=UPRIGHT_FILTER_ENABLED,
    # Floor filter parameters
    floor_z_filter_enabled=FLOOR_Z_FILTER_ENABLED,
    floor_z_filter_mode=FLOOR_Z_FILTER_MODE,
    floor_z_threshold=FLOOR_Z_THRESHOLD,
    # Y-wall upper parameters
    y_wall_upper_enabled=Y_WALL_UPPER_ENABLED,
    y_wall_upper_mode=Y_WALL_UPPER_MODE,
    y_wall_max=Y_WALL_MAX,
    y_wall_upper_margin=Y_WALL_UPPER_MARGIN,
    # Y-wall lower parameters
    y_wall_lower_enabled=Y_WALL_LOWER_ENABLED,
    y_wall_lower_mode=Y_WALL_LOWER_MODE,
    y_wall_min=Y_WALL_MIN,
    y_wall_lower_margin=Y_WALL_LOWER_MARGIN,
    # X-wall parameters
    x_wall_enabled=X_WALL_ENABLED,
    x_wall_mode=X_WALL_MODE,
    x_wall_margin=X_WALL_MARGIN,
    x_wall_tips=X_WALL_TIPS,
    arena_x_mm=ARENA_X_MM,
    # Immobility parameters
    immobility_filter_enabled=IMMOBILITY_FILTER_ENABLED,
    immobility_filter_mode=IMMOBILITY_FILTER_MODE,
    max_stationary_frames=MAX_STATIONARY_FRAMES,
    stationary_speed_threshold=STATIONARY_SPEED_THRESHOLD,
    # Bout detection parameters
    min_bout_frames=MIN_BOUT_FRAMES,
    max_immobility_gap=MAX_IMMOBILITY_FRAMES,
    # Validation parameters
    min_walking_cycles=MIN_WALKING_CYCLES,
    min_distance_mm=MIN_DISTANCE_MM,
    max_swing_duration=MAX_SWING_DURATION,
    swing_prominence=SWING_PROMINENCE,
    enforce_straightness=ENFORCE_STRAIGHTNESS,
    straightness_threshold=STRAIGHTNESS_THRESHOLD,
    # Diagnostics
    show_boundary_diagnostics=SHOW_BOUNDARY_DIAGNOSTICS,
    boundary_diagnostic_frames=BOUNDARY_DIAGNOSTIC_FRAMES,
    verbose=True
):
    """
    Complete walking bout detection pipeline with configurable filters.
    
    All filters can be enabled/disabled independently.
    Soft filters (floor, y-wall, x-wall, immobility) have modes:
      - "split_bout": Break bout at violations
      - "exclude_frames": Mark frames as excluded but keep bout together
      - "ignore": Ignore violations entirely
    
    Returns:
        valid_bouts: List of validated bout dicts (with excluded_frame_mask)
        rejected_bouts: List of rejected bout dicts
        leg_tip_data: Extracted leg data
        scutellum_data: Extracted scutellum data
        filter_masks: Individual filter masks for analysis
        diagnostics: Filter statistics
    """
    if verbose:
        print("=" * 70)
        print("WALKING BOUT DETECTION PIPELINE")
        print("=" * 70)
        print(f"\nFilter settings:")
        print(f"  Confidence: {'ENABLED' if confidence_filter_enabled else 'DISABLED'} (threshold: {confidence_threshold}, gap bridge: {confidence_gap_bridge} frames)")
        print(f"  Upright: {'ENABLED' if upright_filter_enabled else 'DISABLED'}")
        print(f"  Floor Z: {'ENABLED' if floor_z_filter_enabled else 'DISABLED'} (mode: {floor_z_filter_mode}, threshold: {floor_z_threshold}mm)")
        print(f"  Y-wall upper: {'ENABLED' if y_wall_upper_enabled else 'DISABLED'} (mode: {y_wall_upper_mode}, Y >= {y_wall_max - y_wall_upper_margin}mm)")
        print(f"  Y-wall lower: {'ENABLED' if y_wall_lower_enabled else 'DISABLED'} (mode: {y_wall_lower_mode}, Y <= {y_wall_min + y_wall_lower_margin}mm)")
        print(f"  X-wall: {'ENABLED' if x_wall_enabled else 'DISABLED'} (mode: {x_wall_mode}, X <= {x_wall_margin} or X >= {arena_x_mm - x_wall_margin}mm)")
        print(f"  Immobility: {'ENABLED' if immobility_filter_enabled else 'DISABLED'} (mode: {immobility_filter_mode})")
        print(f"\nValidation:")
        print(f"  Min walking cycles: {min_walking_cycles}")
        print(f"  Min distance: {min_distance_mm} mm")
        print(f"  Min bout duration: {min_bout_frames} frames")
    
    # =========================================================================
    # STEP 1: Apply frame-level filters
    # =========================================================================
    valid_mask, leg_tip_data, scutellum_data, filter_masks, diagnostics = (
        apply_comprehensive_walking_filter(
            df, scale,
            confidence_filter_enabled, confidence_threshold, confidence_gap_bridge,
            upright_filter_enabled,
            floor_z_filter_enabled, floor_z_filter_mode, floor_z_threshold,
            y_wall_upper_enabled, y_wall_upper_mode, y_wall_max, y_wall_upper_margin,
            y_wall_lower_enabled, y_wall_lower_mode, y_wall_min, y_wall_lower_margin,
            x_wall_enabled, x_wall_mode, x_wall_margin, x_wall_tips, arena_x_mm,
        )
    )
    
    if verbose:
        print(f"\n--- Frame-level filtering ---")
        print(f"  Total frames: {diagnostics['total_frames']}")
        print(f"  Confidence pass: {diagnostics['confidence_pass_pct']:.1f}%")
        if 'confidence_frames_bridged' in diagnostics:
            print(f"  Confidence frames bridged: {diagnostics['confidence_frames_bridged']} (gap <= {confidence_gap_bridge} frames)")
        print(f"  Upright pass: {diagnostics['upright_pass_pct']:.1f}%")
        print(f"  Floor violations: {diagnostics['floor_violation_pct']:.1f}%")
        print(f"  Y-wall upper violations: {diagnostics['y_wall_upper_violation_pct']:.1f}%")
        print(f"  Y-wall lower violations: {diagnostics['y_wall_lower_violation_pct']:.1f}%")
        print(f"  X-wall violations: {diagnostics['x_wall_violation_pct']:.1f}%")
        print(f"  Combined valid: {diagnostics['combined_pass_pct']:.1f}%")
    
    # =========================================================================
    # STEP 2: Find contiguous bouts with gap bridging
    # =========================================================================
    candidate_bouts = find_contiguous_bouts_with_gap_bridging(
        valid_mask, min_bout_frames, max_immobility_gap
    )
    
    if verbose:
        print(f"\n--- Candidate bouts (after gap bridging) ---")
        print(f"  Found: {len(candidate_bouts)} candidate bouts")
    
    # =========================================================================
    # STEP 3: Split bouts at violations (mode="split_bout" only)
    # =========================================================================
    split_candidate_bouts, bout_excluded_masks = split_bouts_at_violations(
        candidate_bouts, leg_tip_data, scutellum_data, filter_masks, FPS,
        floor_z_filter_enabled, floor_z_filter_mode,
        y_wall_upper_enabled, y_wall_upper_mode,
        y_wall_lower_enabled, y_wall_lower_mode,
        x_wall_enabled, x_wall_mode,
        immobility_filter_enabled, immobility_filter_mode,
        max_stationary_frames, stationary_speed_threshold,
        min_bout_frames
    )
    
    if verbose:
        print(f"\n--- After violation splitting ---")
        print(f"  Bouts after splitting: {len(split_candidate_bouts)}")
        if len(split_candidate_bouts) != len(candidate_bouts):
            print(f"  (Split {len(candidate_bouts)} → {len(split_candidate_bouts)} due to split_bout violations)")
    
    # =========================================================================
    # STEP 4: Validate bouts (cycles + distance + straightness)
    # =========================================================================
    valid_bouts, rejected_bouts = validate_walking_bouts(
        split_candidate_bouts,
        leg_tip_data,
        scutellum_data,
        min_walking_cycles,
        min_distance_mm,
        max_swing_duration,
        swing_prominence,
        enforce_straightness,
        straightness_threshold
    )
    
    if verbose:
        print(f"\n--- Bout validation ---")
        print(f"  Valid bouts: {len(valid_bouts)}")
        print(f"  Rejected bouts: {len(rejected_bouts)}")
    
    # =========================================================================
    # STEP 5: Add excluded frame masks and boundary diagnostics
    # =========================================================================
    for i, bout in enumerate(valid_bouts):
        # Add excluded frame mask
        excluded_mask = get_excluded_frames_for_bout(
            bout['start'], bout['end'], filter_masks,
            floor_z_filter_mode, y_wall_upper_mode, y_wall_lower_mode,
            x_wall_mode, immobility_filter_mode
        )
        bout['excluded_frame_mask'] = excluded_mask
        bout['n_excluded_frames'] = int(excluded_mask.sum())
        bout['n_valid_frames'] = bout['n_frames'] - bout['n_excluded_frames']
        bout['bout_idx'] = i + 1
        
        # Add boundary diagnostics
        if show_boundary_diagnostics:
            bout['boundary_diagnostics'] = diagnose_bout_boundaries(
                bout['start'], bout['end'], filter_masks, boundary_diagnostic_frames
            )
    
    # Print boundary diagnostics summary
    if verbose and show_boundary_diagnostics and valid_bouts:
        print(f"\n--- Boundary diagnostics (first 5 bouts) ---")
        for bout in valid_bouts[:5]:
            print_boundary_diagnostics(bout['bout_idx'], bout['boundary_diagnostics'])
    
    # Print valid bout summary
    if verbose and valid_bouts:
        print(f"\n--- Valid bout summary ---")
        for bout in valid_bouts[:10]:
            excl_info = f", {bout['n_excluded_frames']} excluded" if bout['n_excluded_frames'] > 0 else ""
            print(f"  Bout {bout['bout_idx']}: frames {bout['start']}-{bout['end']} "
                  f"({bout['n_frames']} frames, {bout['n_frames']/FPS:.2f}s, "
                  f"dist={bout['total_distance_mm']:.1f}mm{excl_info})")
        if len(valid_bouts) > 10:
            print(f"  ... and {len(valid_bouts) - 10} more bouts")
    
    return valid_bouts, rejected_bouts, leg_tip_data, scutellum_data, filter_masks, diagnostics


print("Main pipeline function defined (with configurable filters and diagnostics).")

In [ ]:
# ============================================================================
# VISUALIZATION FUNCTION
# ============================================================================

def _add_bout_boundaries(ax, actual_start, actual_end, add_legend=False):
    """Add bout boundary shading and vertical lines to an axis."""
    ax.axvspan(actual_start, actual_end, color='#2ca02c', alpha=0.08, zorder=0,
               label='Detected bout' if add_legend else None)
    ax.axvline(x=actual_start, color='#d62728', ls='--', lw=1.5, alpha=0.8, zorder=3,
               label='Bout start/end' if add_legend else None)
    ax.axvline(x=actual_end, color='#d62728', ls='--', lw=1.5, alpha=0.8, zorder=3)


def _format_boundary_cause(cause, failures_dict, check_start, check_end, leg_tip_data):
    """
    Format a boundary cause string with actual data values.
    
    Args:
        cause: Raw cause string from boundary_diagnostics (e.g. 'confidence', 'y_wall_upper')
        failures_dict: Pre/post failure counts dict from boundary_diagnostics
        check_start, check_end: Frame indices just outside the bout boundary to inspect
        leg_tip_data: Leg tip data dict for computing actual values
    """
    if cause is None:
        return None
    
    n_analyzed = failures_dict.get('n_frames_analyzed', 0)
    
    if cause == 'confidence':
        n_fail = failures_dict.get('confidence', 0)
        return f'Low confidence ({n_fail}/{n_analyzed}f)'
    
    elif cause == 'y_wall_upper' and leg_tip_data is not None:
        max_y = -np.inf
        worst_leg = ''
        for tip in LEG_TIPS:
            y_vals = leg_tip_data[tip]['y'][check_start:check_end]
            if len(y_vals) > 0:
                tip_max = np.nanmax(y_vals)
                if tip_max > max_y:
                    max_y = tip_max
                    worst_leg = tip.replace('_TaTip', '')
        limit = Y_WALL_MAX - Y_WALL_UPPER_MARGIN
        if np.isfinite(max_y):
            return f'Near upper wall ({worst_leg} Y={max_y:.2f}, lim={limit:.2f}mm)'
        return f'Near upper wall ({failures_dict.get("y_wall_upper", 0)}/{n_analyzed}f)'
    
    elif cause == 'y_wall_lower' and leg_tip_data is not None:
        min_y = np.inf
        worst_leg = ''
        for tip in LEG_TIPS:
            y_vals = leg_tip_data[tip]['y'][check_start:check_end]
            if len(y_vals) > 0:
                tip_min = np.nanmin(y_vals)
                if tip_min < min_y:
                    min_y = tip_min
                    worst_leg = tip.replace('_TaTip', '')
        limit = Y_WALL_MIN + Y_WALL_LOWER_MARGIN
        if np.isfinite(min_y):
            return f'Near lower wall ({worst_leg} Y={min_y:.2f}, lim={limit:.2f}mm)'
        return f'Near lower wall ({failures_dict.get("y_wall_lower", 0)}/{n_analyzed}f)'
    
    elif cause == 'floor' and leg_tip_data is not None:
        min_z = np.inf
        worst_leg = ''
        for tip in LEG_TIPS:
            z_vals = leg_tip_data[tip]['z'][check_start:check_end]
            if len(z_vals) > 0:
                tip_min = np.nanmin(z_vals)
                if tip_min < min_z:
                    min_z = tip_min
                    worst_leg = tip.replace('_TaTip', '')
        if np.isfinite(min_z):
            return f'Floor Z ({worst_leg} Z={min_z:.2f}, lim={FLOOR_Z_THRESHOLD:.2f}mm)'
        return f'Floor Z violation ({failures_dict.get("floor", 0)}/{n_analyzed}f)'
    
    elif cause == 'upright':
        return f'Not upright ({failures_dict.get("upright", 0)}/{n_analyzed}f)'
    
    return cause


def _get_boundary_reasons(bout_info, speed, ctx_start, start, end, ctx_end, fps, leg_tip_data=None):
    """
    Determine why the bout started/stopped where it did, with detailed values.
    
    Uses boundary_diagnostics (filter-based) and also checks for immobility
    near the boundaries (which isn't covered by filter_masks).
    """
    bd = bout_info.get('boundary_diagnostics', None)
    
    start_reason = None
    end_reason = None
    n_check = 10  # frames to inspect just outside boundary
    
    if bd:
        pre = bd.get('pre_bout_failures', {})
        post = bd.get('post_bout_failures', {})
        
        # Frames just before bout start
        before_start = max(0, start - n_check)
        start_reason = _format_boundary_cause(
            bd.get('primary_cause_before'), pre,
            before_start, start, leg_tip_data
        )
        
        # Frames just after bout end
        data_len = len(leg_tip_data[LEG_TIPS[0]]['y']) if leg_tip_data else end + n_check + 1
        after_end = min(data_len, end + 1 + n_check)
        end_reason = _format_boundary_cause(
            bd.get('primary_cause_after'), post,
            end + 1, after_end, leg_tip_data
        )
    
    # Also check for immobility near boundaries using speed data
    if speed is not None and len(speed) > 0:
        pre_idx = start - ctx_start
        post_idx = end - ctx_start
        
        check_n = min(n_check, pre_idx)
        if check_n > 0 and start_reason is None:
            pre_speed = np.nanmean(speed[pre_idx - check_n:pre_idx])
            if pre_speed < STATIONARY_SPEED_THRESHOLD:
                start_reason = f'Prolonged immobility (speed={pre_speed:.1f}mm/s)'
        
        check_n = min(n_check, len(speed) - post_idx - 1)
        if check_n > 0 and end_reason is None:
            post_speed = np.nanmean(speed[post_idx + 1:post_idx + 1 + check_n])
            if post_speed < STATIONARY_SPEED_THRESHOLD:
                end_reason = f'Prolonged immobility (speed={post_speed:.1f}mm/s)'
    
    return start_reason, end_reason


def plot_walking_bout_figure(
    bout_info,
    leg_tip_data,
    scutellum_data,
    frame_offset=0,
    fps=FPS,
    arena_x_mm=ARENA_X_MM,
    arena_y_mm=ARENA_Y_MM,
    save_path=None,
    dpi=PDF_DPI,
    show_gait_diagram=SHOW_GAIT_DIAGRAM,
    stance_threshold=STANCE_Z_THRESHOLD,
    context_frames=200
):
    """
    Create 5-panel walking bout analysis figure with context frames.
    
    Panels:
    1. Leg tip Z trajectories (6 legs)
    2. Gait phase diagram (stance/swing for each leg)
    3. Scutellum Z (body height)
    4. Instantaneous speed
    5. XY trajectory in rectangular arena
    
    Args:
        context_frames: Number of frames to show before and after the detected bout
                       (default 200). The detected bout region is highlighted with
                       green shading and red dashed boundary lines. Boundary reasons
                       with actual values are annotated at the edges.
    
    Returns:
        fig: matplotlib Figure object
    """
    start = bout_info['start']
    end = bout_info['end']
    bout_idx = bout_info.get('bout_idx', '?')
    
    # Compute extended range with context
    data_len = len(scutellum_data['x'])
    ctx_start = max(0, start - context_frames)
    ctx_end = min(data_len - 1, end + context_frames)
    
    # Create frame arrays (actual frame numbers)
    actual_start = start + frame_offset
    actual_end = end + frame_offset
    actual_ctx_start = ctx_start + frame_offset
    actual_ctx_end = ctx_end + frame_offset
    frames = np.arange(actual_ctx_start, actual_ctx_end + 1)
    n_frames_bout = end - start + 1
    
    # Get excluded frame mask if available (pad for context)
    excluded_mask = bout_info.get('excluded_frame_mask', None)
    if excluded_mask is not None and context_frames > 0:
        pad_before = start - ctx_start
        pad_after = ctx_end - end
        excluded_mask = np.concatenate([
            np.zeros(pad_before, dtype=bool),
            excluded_mask,
            np.zeros(pad_after, dtype=bool)
        ])
    
    # Extract data for extended range
    scut_x = scutellum_data['x'][ctx_start:ctx_end+1]
    scut_y = scutellum_data['y'][ctx_start:ctx_end+1]
    scut_z = scutellum_data['z'][ctx_start:ctx_end+1]
    
    # Compute speed for extended range (for plotting)
    speed, _, _ = compute_instant_speed(scutellum_data, ctx_start, ctx_end, fps)
    # Compute bout-only metrics for labels/titles
    _, mean_speed, max_speed = compute_instant_speed(scutellum_data, start, end, fps)
    
    # Determine boundary reasons with detailed values
    start_reason, end_reason = _get_boundary_reasons(
        bout_info, speed, ctx_start, start, end, ctx_end, fps,
        leg_tip_data=leg_tip_data
    )
    
    # Create figure with 5 panels (or 4 if gait diagram disabled)
    if show_gait_diagram:
        fig = plt.figure(figsize=(14, 17))
        gs = fig.add_gridspec(5, 1, height_ratios=[1, 0.5, 0.7, 0.7, 1.2], hspace=0.35)
        ax1 = fig.add_subplot(gs[0])
        ax_gait = fig.add_subplot(gs[1], sharex=ax1)
        ax2 = fig.add_subplot(gs[2], sharex=ax1)
        ax3 = fig.add_subplot(gs[3], sharex=ax1)
        ax4 = fig.add_subplot(gs[4])
    else:
        fig = plt.figure(figsize=(14, 14))
        gs = fig.add_gridspec(4, 1, height_ratios=[1, 0.7, 0.7, 1.2], hspace=0.35)
        ax1 = fig.add_subplot(gs[0])
        ax_gait = None
        ax2 = fig.add_subplot(gs[1], sharex=ax1)
        ax3 = fig.add_subplot(gs[2], sharex=ax1)
        ax4 = fig.add_subplot(gs[3])
    
    # Color scheme for legs
    leg_colors = {
        # 'T1L_TaTip': '#E63946',  # Front left - red
        'T1R_TaTip': '#457B9D',  # Front right - blue
        'T2L_TaTip': '#F4A261',  # Mid left - orange
        'T2R_TaTip': '#2A9D8F',  # Mid right - teal
        'T3L_TaTip': '#9B2226',  # Hind left - dark red
        'T3R_TaTip': '#1D3557',  # Hind right - navy
    }
    
    # ===== PANEL 1: Leg Tip Z Trajectories =====
    _add_bout_boundaries(ax1, actual_start, actual_end, add_legend=True)
    for tip, color in leg_colors.items():
        z_leg = leg_tip_data[tip]['z'][ctx_start:ctx_end+1]
        label = tip.replace('_TaTip', '')
        ax1.plot(frames, z_leg, label=label, color=color, lw=1.2, alpha=0.8)
    
    ax1.axhline(y=FLOOR_Z_THRESHOLD, color='gray', ls=':', lw=1.5, 
                alpha=0.7, label=f'Floor ({FLOOR_Z_THRESHOLD}mm)')
    ax1.set_ylabel('Leg Tip Z (mm)', fontsize=12)
    ax1.set_title(f'Leg Tip Z Trajectories | Bout {bout_idx}', fontsize=13)
    ax1.legend(loc='upper left', bbox_to_anchor=(1.02, 1), fontsize=9, ncol=1)
    ax1.grid(True, alpha=0.3)
    plt.setp(ax1.get_xticklabels(), visible=False)
    
    # Annotate boundary reasons on the leg Z panel
    if start_reason:
        ax1.annotate(
            start_reason,
            xy=(actual_start, 1), xycoords=('data', 'axes fraction'),
            xytext=(6, -8), textcoords='offset points',
            fontsize=8, color='#d62728', fontweight='bold',
            ha='left', va='top',
            bbox=dict(boxstyle='round,pad=0.3', fc='white', ec='#d62728', alpha=0.85)
        )
    if end_reason:
        ax1.annotate(
            end_reason,
            xy=(actual_end, 1), xycoords=('data', 'axes fraction'),
            xytext=(-6, -8), textcoords='offset points',
            fontsize=8, color='#d62728', fontweight='bold',
            ha='right', va='top',
            bbox=dict(boxstyle='round,pad=0.3', fc='white', ec='#d62728', alpha=0.85)
        )
    
    # ===== PANEL 2: Gait Phase Diagram =====
    if show_gait_diagram and ax_gait is not None:
        _add_bout_boundaries(ax_gait, actual_start, actual_end)
        plot_gait_phase_diagram(
            leg_tip_data, ctx_start, ctx_end, frames,
            stance_threshold=stance_threshold,
            ax=ax_gait,
            excluded_mask=excluded_mask
        )
        plt.setp(ax_gait.get_xticklabels(), visible=False)
    
    # ===== PANEL 3: Scutellum Z (Body Height) =====
    _add_bout_boundaries(ax2, actual_start, actual_end)
    ax2.plot(frames, scut_z, color='#2D3142', lw=1.8, label='Scutellum Z')
    ax2.fill_between(frames, np.nanmin(scut_z), scut_z, alpha=0.3, color='#2D3142')
    
    scut_z_bout = scutellum_data['z'][start:end+1]
    scut_z_mean = np.nanmean(scut_z_bout)
    ax2.axhline(y=scut_z_mean, color='#E76F51', ls='--', lw=1.5, 
                label=f'Bout mean: {scut_z_mean:.2f} mm')
    ax2.set_ylabel('Body Height Z (mm)', fontsize=12)
    ax2.set_title('Scutellum Z (Body Height) - compare with speed below', fontsize=13)
    ax2.legend(loc='upper right', fontsize=10)
    ax2.grid(True, alpha=0.3)
    plt.setp(ax2.get_xticklabels(), visible=False)
    
    # ===== PANEL 4: Instantaneous Speed =====
    _add_bout_boundaries(ax3, actual_start, actual_end)
    ax3.plot(frames, speed, color='#264653', lw=1.2)
    ax3.axhline(y=mean_speed, color='#E76F51', ls='--', lw=1.5, 
                label=f'Bout mean: {mean_speed:.1f} mm/s')
    ax3.fill_between(frames, 0, speed, alpha=0.3, color='#264653')
    ax3.set_ylabel('Speed (mm/s)', fontsize=12)
    ax3.set_xlabel('Frame', fontsize=11)
    ax3.set_title(f'Instantaneous Speed | Bout max: {max_speed:.1f} mm/s', fontsize=13)
    ax3.legend(loc='upper right', fontsize=10)
    ax3.grid(True, alpha=0.3)
    ax3.set_ylim(bottom=0)
    
    # ===== PANEL 5: XY Trajectory in Rectangular Arena =====
    rect_x = [0, arena_x_mm, arena_x_mm, 0, 0]
    rect_y = [0, 0, arena_y_mm, arena_y_mm, 0]
    ax4.plot(rect_x, rect_y, 'k-', lw=2, label=f'Arena ({arena_x_mm}x{arena_y_mm}mm)')
    
    # Context trajectory (before bout) in gray
    pre_len = start - ctx_start
    if pre_len > 1:
        pre_x = scutellum_data['x'][ctx_start:start+1]
        pre_y = scutellum_data['y'][ctx_start:start+1]
        valid_pre = ~np.isnan(pre_x) & ~np.isnan(pre_y)
        if np.sum(valid_pre) > 1:
            ax4.plot(pre_x[valid_pre], pre_y[valid_pre], color='gray', ls='--',
                     lw=1.2, alpha=0.5, label='Context (before)')
    
    # Context trajectory (after bout) in gray
    post_len = ctx_end - end
    if post_len > 1:
        post_x = scutellum_data['x'][end:ctx_end+1]
        post_y = scutellum_data['y'][end:ctx_end+1]
        valid_post = ~np.isnan(post_x) & ~np.isnan(post_y)
        if np.sum(valid_post) > 1:
            ax4.plot(post_x[valid_post], post_y[valid_post], color='gray', ls=':',
                     lw=1.2, alpha=0.5, label='Context (after)')
    
    # Bout trajectory in color
    bout_x = scutellum_data['x'][start:end+1]
    bout_y = scutellum_data['y'][start:end+1]
    valid = ~np.isnan(bout_x) & ~np.isnan(bout_y)
    x_valid = bout_x[valid]
    y_valid = bout_y[valid]
    
    if len(x_valid) > 1:
        points = np.array([x_valid, y_valid]).T.reshape(-1, 1, 2)
        segments = np.concatenate([points[:-1], points[1:]], axis=1)
        t = np.linspace(0, 1, len(segments))
        lc = LineCollection(segments, cmap='viridis', norm=plt.Normalize(0, 1))
        lc.set_array(t)
        lc.set_linewidth(2.5)
        ax4.add_collection(lc)
        ax4.plot(x_valid[0], y_valid[0], 'go', ms=12, label='Bout start', zorder=5)
        ax4.plot(x_valid[-1], y_valid[-1], 'r^', ms=12, label='Bout end', zorder=5)
    
    padding_x = arena_x_mm * 0.05
    padding_y = arena_y_mm * 0.15
    ax4.set_xlim(-padding_x, arena_x_mm + padding_x)
    ax4.set_ylim(-padding_y, arena_y_mm + padding_y)
    ax4.set_aspect('equal')
    ax4.set_xlabel('X (mm)', fontsize=12)
    ax4.set_ylabel('Y (mm)', fontsize=12)
    ax4.set_title(f'XY Trajectory | Distance: {bout_info["total_distance_mm"]:.2f} mm', fontsize=13)
    ax4.legend(loc='upper right', fontsize=10)
    ax4.grid(True, alpha=0.3)
    
    sm = plt.cm.ScalarMappable(cmap='viridis', norm=plt.Normalize(0, 1))
    sm.set_array([])
    cbar = plt.colorbar(sm, ax=ax4, shrink=0.6, pad=0.02, orientation='vertical')
    cbar.set_label('Time (normalized)', fontsize=10)
    
    # Overall title with boundary reasons
    duration_s = n_frames_bout / fps
    excl_info = ""
    if excluded_mask is not None:
        n_excl = bout_info.get('n_excluded_frames', 0)
        if n_excl > 0:
            excl_info = f" | {n_excl} frames excluded"
    
    ctx_info = f" | showing {context_frames}f context" if context_frames > 0 else ""
    
    boundary_str = ""
    if start_reason or end_reason:
        parts = []
        if start_reason:
            parts.append(f"start: {start_reason}")
        if end_reason:
            parts.append(f"end: {end_reason}")
        boundary_str = f"\nBoundary causes: {' | '.join(parts)}"
    
    fig.suptitle(
        f'Walking Bout Analysis | Bout frames {actual_start}-{actual_end} | '
        f'{duration_s:.2f}s | {bout_info["min_cycles"]} min cycles{excl_info}{ctx_info}'
        f'{boundary_str}',
        fontsize=13, fontweight='bold', y=0.998
    )
    
    plt.tight_layout()
    
    if save_path:
        fig.savefig(save_path, format='pdf', dpi=dpi, bbox_inches='tight')
        print(f"  Saved: {save_path}")
    
    return fig


print("Visualization function defined (5 panels with context frames and detailed boundary annotations).")

In [ ]:
# ============================================================================
# GAIT PHASE DIAGRAM FUNCTION
# ============================================================================

def compute_stance_swing_phases(leg_tip_data, start, end, stance_threshold=STANCE_Z_THRESHOLD):
    """
    Compute stance (Z <= threshold) and swing (Z > threshold) phases for each leg.
    
    Args:
        leg_tip_data: Dict with leg tip z data
        start, end: Bout frame indices
        stance_threshold: Z threshold for stance detection (mm)
    
    Returns:
        phases: Dict mapping leg name -> boolean array (True = stance, False = swing)
    """
    phases = {}
    for tip in LEG_TIPS:
        z = leg_tip_data[tip]['z'][start:end+1]
        # Stance = leg is on ground (Z <= threshold)
        phases[tip] = z <= stance_threshold
    return phases


def plot_gait_phase_diagram(
    leg_tip_data, 
    start, 
    end, 
    frames,
    stance_threshold=STANCE_Z_THRESHOLD,
    ax=None,
    excluded_mask=None
):
    """
    Plot gait phase diagram showing stance/swing for each leg.
    
    Layout:
    - 6 rows (one per leg): R3, R2, R1, L3, L2, L1 (top to bottom)
    - Filled/black horizontal bars = STANCE (Z <= threshold, leg on ground)
    - Empty/white = SWING (Z > threshold, leg lifted)
    - Gray shading = excluded frames (optional)
    
    Args:
        leg_tip_data: Dict with leg tip data
        start, end: Bout frame indices
        frames: Frame number array for x-axis
        stance_threshold: Z threshold for stance (default: STANCE_Z_THRESHOLD)
        ax: Matplotlib axis (creates new if None)
        excluded_mask: Boolean array of frames to shade as excluded
    
    Returns:
        ax: Matplotlib axis
    """
    if ax is None:
        fig, ax = plt.subplots(figsize=(14, 3))
    
    # Compute phases
    phases = compute_stance_swing_phases(leg_tip_data, start, end, stance_threshold)
    
    # Leg order (top to bottom): R3, R2, R1, L3, L2, L1
    leg_order = [
        'T3R_TaTip',  # R3 (top)
        'T2R_TaTip',  # R2
        'T1R_TaTip',  # R1
        'T3L_TaTip',  # L3
        'T2L_TaTip',  # L2
        # 'T1L_TaTip',  # L1 (bottom)
    ]
    leg_labels = ['R3', 'R2', 'R1', 'L3', 'L2', 'L1']
    
    n_frames = len(frames)
    bar_height = 0.6
    
    # Optional: shade excluded frames
    if excluded_mask is not None and excluded_mask.any():
        # Find contiguous excluded regions
        in_excluded = False
        excl_start = 0
        for i in range(n_frames):
            if excluded_mask[i] and not in_excluded:
                excl_start = i
                in_excluded = True
            elif not excluded_mask[i] and in_excluded:
                ax.axvspan(frames[excl_start], frames[i-1], 
                          color='lightgray', alpha=0.5, zorder=0)
                in_excluded = False
        # Handle excluded extending to end
        if in_excluded:
            ax.axvspan(frames[excl_start], frames[-1], 
                      color='lightgray', alpha=0.5, zorder=0)
    
    # Plot each leg
    for y_pos, (leg, label) in enumerate(zip(leg_order, leg_labels)):
        stance_mask = phases[leg]
        
        # Find contiguous stance periods
        in_stance = False
        stance_start_idx = 0
        
        for i, is_stance in enumerate(stance_mask):
            if is_stance and not in_stance:
                stance_start_idx = i
                in_stance = True
            elif not is_stance and in_stance:
                # Draw stance bar
                ax.barh(
                    y_pos, 
                    frames[i-1] - frames[stance_start_idx] + 1,
                    left=frames[stance_start_idx],
                    height=bar_height,
                    color='black',
                    edgecolor='black',
                    linewidth=0.5
                )
                in_stance = False
        
        # Handle stance extending to end
        if in_stance:
            ax.barh(
                y_pos,
                frames[-1] - frames[stance_start_idx] + 1,
                left=frames[stance_start_idx],
                height=bar_height,
                color='black',
                edgecolor='black',
                linewidth=0.5
            )
    
    # Formatting
    ax.set_yticks(range(len(leg_order)))
    ax.set_yticklabels(leg_labels, fontsize=10)
    ax.set_ylabel('Leg', fontsize=11)
    ax.set_title(f'Gait Phase Diagram (black = stance, Z ≤ {stance_threshold}mm)', fontsize=12)
    ax.set_xlim(frames[0] - 5, frames[-1] + 5)
    ax.set_ylim(-0.5, len(leg_order) - 0.5)
    ax.grid(True, axis='x', alpha=0.3, linestyle=':')
    
    # Add legend
    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor='black', edgecolor='black', label='Stance (on ground)'),
        Patch(facecolor='white', edgecolor='black', label='Swing (in air)'),
    ]
    if excluded_mask is not None and excluded_mask.any():
        legend_elements.append(
            Patch(facecolor='lightgray', alpha=0.5, edgecolor='gray', label='Excluded frames')
        )
    ax.legend(handles=legend_elements, loc='upper right', fontsize=9, ncol=len(legend_elements))
    
    return ax


print("Gait phase diagram functions defined.")

In [ ]:
# ============================================================================
# BATCH ANALYSIS FUNCTION
# ============================================================================

def analyze_and_plot_all_bouts(
    valid_bouts,
    leg_tip_data,
    scutellum_data,
    frame_offset=0,
    output_dir=OUTPUT_DIR,
    fps=FPS,
    arena_x_mm=ARENA_X_MM,
    arena_y_mm=ARENA_Y_MM,
    dpi=PDF_DPI,
    show_plots=True,
    max_bouts=None,  # None = all bouts, or integer for N longest bouts
    sort_by_distance=True,  # If max_bouts is set, sort by distance (longest first)
    context_frames=200,  # Number of frames before/after bout to show in plots
    fly_id=None  # Fly identifier from info.yaml
):
    """
    Analyze and create figures for walking bouts.
    
    Parameters:
        max_bouts: None to plot all bouts, or integer (e.g., 5) to plot only N longest bouts
        sort_by_distance: If True and max_bouts is set, select the N longest bouts by distance
        context_frames: Number of frames to show before/after each bout (default 200).
                       Set to 0 to only show the bout itself.
    
    Returns:
        figures: List of figure objects
        summary: Summary DataFrame
    """
    if output_dir:
        output_dir = Path(output_dir)
        output_dir.mkdir(parents=True, exist_ok=True)
    
    # Select bouts to analyze
    bouts_to_analyze = valid_bouts.copy()
    
    if max_bouts is not None and max_bouts < len(valid_bouts):
        if sort_by_distance:
            # Sort by total_distance_mm descending and take top N
            bouts_to_analyze = sorted(valid_bouts, key=lambda b: b['total_distance_mm'], reverse=True)[:max_bouts]
            print(f"\nSelected {max_bouts} longest bouts (by distance) out of {len(valid_bouts)} total")
        else:
            bouts_to_analyze = valid_bouts[:max_bouts]
            print(f"\nSelected first {max_bouts} bouts out of {len(valid_bouts)} total")
    
    figures = []
    summary_data = []
    
    print(f"\n" + "=" * 70)
    print(f"GENERATING FIGURES FOR {len(bouts_to_analyze)} WALKING BOUTS")
    if context_frames > 0:
        print(f"  Showing {context_frames} context frames before/after each bout")
    print("=" * 70)
    
    for i, bout in enumerate(bouts_to_analyze):
        bout['bout_idx'] = i + 1
        
        # Compute additional metrics
        speed, mean_speed, max_speed = compute_instant_speed(
            scutellum_data, bout['start'], bout['end'], fps
        )
        bout['mean_speed_mm_s'] = mean_speed
        bout['max_speed_mm_s'] = max_speed
        bout['duration_s'] = bout['n_frames'] / fps
        
        # Compute scutellum Z stats for correlation analysis
        scut_z_bout = scutellum_data['z'][bout['start']:bout['end']+1]
        bout['scut_z_mean'] = np.nanmean(scut_z_bout)
        bout['scut_z_std'] = np.nanstd(scut_z_bout)
        bout['scut_z_min'] = np.nanmin(scut_z_bout)
        bout['scut_z_max'] = np.nanmax(scut_z_bout)
        
        # Compute frame-by-frame height-speed correlation within this bout
        valid_mask = ~np.isnan(scut_z_bout) & ~np.isnan(speed)
        if np.sum(valid_mask) > 10:
            r_pearson, p_pearson = pearsonr(scut_z_bout[valid_mask], speed[valid_mask])
            r_spearman, p_spearman = spearmanr(scut_z_bout[valid_mask], speed[valid_mask])
        else:
            r_pearson, p_pearson = np.nan, np.nan
            r_spearman, p_spearman = np.nan, np.nan
        bout['height_speed_pearson_r'] = r_pearson
        bout['height_speed_pearson_p'] = p_pearson
        bout['height_speed_spearman_r'] = r_spearman
        bout['height_speed_spearman_p'] = p_spearman
        
        # Generate save path
        save_path = None
        if output_dir:
            actual_start = bout['start'] + frame_offset
            actual_end = bout['end'] + frame_offset
            save_path = output_dir / f"walking_bout_{i+1:03d}_frames_{actual_start}-{actual_end}.pdf"
        
        print(f"\nBout {i+1}/{len(bouts_to_analyze)}: frames {bout['start']+frame_offset}-{bout['end']+frame_offset}")
        print(f"    Distance: {bout['total_distance_mm']:.2f} mm")
        print(f"    Mean speed: {mean_speed:.1f} mm/s, Mean height: {bout['scut_z_mean']:.2f} mm")
        print(f"    Height-Speed correlation (Pearson r): {r_pearson:.3f} (p={p_pearson:.4f})")
        
        # Create figure
        fig = plot_walking_bout_figure(
            bout,
            leg_tip_data,
            scutellum_data,
            frame_offset=frame_offset,
            fps=fps,
            arena_x_mm=arena_x_mm,
            arena_y_mm=arena_y_mm,
            save_path=save_path,
            dpi=dpi,
            context_frames=context_frames
        )
        
        figures.append(fig)
        
        # Collect summary data (including Z correlation metrics)
        summary_data.append({
            'fly_id': fly_id,
            'bout_idx': i + 1,
            'start_frame': bout['start'] + frame_offset,
            'end_frame': bout['end'] + frame_offset,
            'n_frames': bout['n_frames'],
            'duration_s': bout['duration_s'],
            'min_cycles': bout['min_cycles'],
            'total_distance_mm': bout['total_distance_mm'],
            'net_displacement_mm': bout['net_displacement_mm'],
            'mean_speed_mm_s': mean_speed,
            'max_speed_mm_s': max_speed,
            'scut_z_mean': bout['scut_z_mean'],
            'scut_z_std': bout['scut_z_std'],
            'scut_z_min': bout['scut_z_min'],
            'scut_z_max': bout['scut_z_max'],
            'height_speed_pearson_r': r_pearson,
            'height_speed_pearson_p': p_pearson,
            'height_speed_spearman_r': r_spearman,
            'height_speed_spearman_p': p_spearman
        })
        
        if show_plots:
            plt.show()
        else:
            plt.close(fig)
    
    # Create summary DataFrame
    summary = pd.DataFrame(summary_data)
    
    if output_dir and len(summary_data) > 0:
        summary_path = output_dir.parent / "walking_bouts_summary.csv"
        summary.to_csv(summary_path, index=False)
        print(f"\nSummary saved: {summary_path}")
    
    return figures, summary


print("Batch analysis function defined (with bout selection and height-speed correlation).")

## Run Walking Bout Detection

Execute the cell below to run the walking bout detection on your data.
Make sure to set the correct `DATA_3D_PATH` before running.

In [ ]:
# ============================================================================
# RUN WALKING BOUT DETECTION
# ============================================================================

# Optional: specify frame range (set to None for full video)
FRAME_START = None  # e.g., 0
FRAME_END = None    # e.g., 100000


# Apply frame range if specified
frame_offset = 0
if FRAME_START is not None or FRAME_END is not None:
    start = FRAME_START if FRAME_START is not None else 0
    end = FRAME_END if FRAME_END is not None else len(df_full)
    df = df_full.iloc[start:end].reset_index(drop=True)
    frame_offset = start
    print(f"Using frame range: {start} to {end} ({len(df)} frames)")
else:
    df = df_full
    print(f"Using full data: {len(df)} frames")

# Run walking bout detection (uses configuration constants from Cell 23)
valid_bouts, rejected_bouts, leg_tip_data, scutellum_data, filter_masks, diagnostics = detect_walking_bouts(
    df,
    verbose=True
)

# Save run configuration alongside results
diagnostics['n_valid_bouts'] = len(valid_bouts)
diagnostics['n_rejected_bouts'] = len(rejected_bouts)
save_run_config(OUTPUT_DIR, DATA_3D_PATH, diagnostics=diagnostics,
                frame_start=FRAME_START, frame_end=FRAME_END)

print(f"\n" + "=" * 70)
print(f"DETECTION COMPLETE: {len(valid_bouts)} valid walking bouts found")
print("=" * 70)

In [ ]:
# ============================================================================
# GENERATE AND SAVE FIGURES
# ============================================================================

# Configuration for bout selection
PLOT_ALL_BOUTS = True# Set to True to plot all bouts, False to plot only longest N
N_LONGEST_BOUTS = 10    # Number of longest bouts to plot (only used if PLOT_ALL_BOUTS=False)

if len(valid_bouts) > 0:
    figures, summary = analyze_and_plot_all_bouts(
        valid_bouts,
        leg_tip_data,
        scutellum_data,
        frame_offset=frame_offset,
        output_dir=OUTPUT_DIR,
        fps=FPS,
        arena_x_mm=ARENA_X_MM,
        arena_y_mm=ARENA_Y_MM,
        dpi=PDF_DPI,
        show_plots=True,  # Set to False to only save without displaying
        max_bouts=None if PLOT_ALL_BOUTS else N_LONGEST_BOUTS,
        sort_by_distance=True,
        context_frames=200,
        fly_id=fly_id
    )
    
    # Display summary
    print("\n" + "=" * 70)
    print("WALKING BOUTS SUMMARY")
    print("=" * 70)
    display(summary)
else:
    print("No valid walking bouts found. Try adjusting the filter thresholds.")

## HEIGHT-SPEED CORRELATION ANALYSIS (uses ALL valid bouts)

In [ ]:
# ============================================================================
# HEIGHT-SPEED CORRELATION ANALYSIS (uses ALL valid bouts)
# ============================================================================

def analyze_height_speed_correlation(scutellum_data, valid_bouts, fps=FPS, output_dir=OUTPUT_DIR):
    """
    Comprehensive analysis of body height vs speed correlation.
    Uses ALL valid bouts for robust statistical analysis.
    Creates visualizations and statistical summaries.
    """
    
    # First, compute metrics for ALL valid bouts
    all_bout_metrics = []
    all_heights_frames = []
    all_speeds_frames = []
    
    print(f"Computing height-speed metrics for ALL {len(valid_bouts)} valid bouts...")
    
    for bout in valid_bouts:
        start, end = bout['start'], bout['end']
        scut_z_bout = scutellum_data['z'][start:end+1]
        speed, mean_speed, max_speed = compute_instant_speed(scutellum_data, start, end, fps)
        
        # Bout-level metrics
        scut_z_mean = np.nanmean(scut_z_bout)
        scut_z_std = np.nanstd(scut_z_bout)
        
        # Within-bout correlation
        valid_mask = ~np.isnan(scut_z_bout) & ~np.isnan(speed)
        if np.sum(valid_mask) > 10:
            r_pearson, p_pearson = pearsonr(scut_z_bout[valid_mask], speed[valid_mask])
        else:
            r_pearson, p_pearson = np.nan, np.nan
        
        all_bout_metrics.append({
            'scut_z_mean': scut_z_mean,
            'scut_z_std': scut_z_std,
            'mean_speed_mm_s': mean_speed,
            'max_speed_mm_s': max_speed,
            'height_speed_pearson_r': r_pearson,
            'height_speed_pearson_p': p_pearson,
            'total_distance_mm': bout['total_distance_mm']
        })
        
        # Collect frame-level data
        all_heights_frames.extend(scut_z_bout[valid_mask])
        all_speeds_frames.extend(speed[valid_mask])
    
    # Convert to DataFrame for easy analysis
    all_bouts_df = pd.DataFrame(all_bout_metrics)
    all_heights = np.array(all_heights_frames)
    all_speeds = np.array(all_speeds_frames)
    
    # Create figure
    fig, axes = plt.subplots(2, 2, figsize=(14, 12))
    
    # ===== Panel 1: Bout-level correlation (mean height vs mean speed) - ALL BOUTS =====
    ax1 = axes[0, 0]
    mean_heights = all_bouts_df['scut_z_mean'].values
    mean_speeds = all_bouts_df['mean_speed_mm_s'].values
    
    valid_mask = ~np.isnan(mean_heights) & ~np.isnan(mean_speeds)
    if np.sum(valid_mask) > 2:
        h_valid = mean_heights[valid_mask]
        s_valid = mean_speeds[valid_mask]
        
        ax1.scatter(h_valid, s_valid, s=80, c='#264653', alpha=0.7, edgecolor='white', lw=1.5)
        
        # Regression line
        slope, intercept, r_val, p_val, std_err = linregress(h_valid, s_valid)
        x_line = np.linspace(h_valid.min(), h_valid.max(), 100)
        ax1.plot(x_line, slope * x_line + intercept, 'r--', lw=2, 
                 label=f'r={r_val:.3f}, p={p_val:.4f}')
        
        ax1.set_xlabel('Mean Body Height (mm)', fontsize=12)
        ax1.set_ylabel('Mean Speed (mm/s)', fontsize=12)
        ax1.set_title(f'Bout-Level: Mean Height vs Mean Speed (n={len(h_valid)} bouts)', fontsize=13)
        ax1.legend(loc='best', fontsize=10)
        ax1.grid(True, alpha=0.3)
    
    # ===== Panel 2: Distribution of within-bout correlations =====
    ax2 = axes[0, 1]
    correlations = all_bouts_df['height_speed_pearson_r'].dropna().values
    
    if len(correlations) > 0:
        ax2.hist(correlations, bins=min(20, len(correlations)), color='#457B9D', edgecolor='white', alpha=0.8)
        ax2.axvline(x=0, color='gray', ls='--', lw=1.5, alpha=0.7)
        ax2.axvline(x=np.mean(correlations), color='#E63946', ls='-', lw=2, 
                    label=f'Mean r = {np.mean(correlations):.3f}')
        ax2.set_xlabel('Pearson r (Height vs Speed)', fontsize=12)
        ax2.set_ylabel('Number of Bouts', fontsize=12)
        ax2.set_title(f'Distribution of Within-Bout Correlations (n={len(correlations)})', fontsize=13)
        ax2.legend(loc='best', fontsize=10)
        ax2.grid(True, alpha=0.3)
    
    # ===== Panel 3: Pooled frame-by-frame correlation (all bouts combined) =====
    ax3 = axes[1, 0]
    
    if len(all_heights) > 100:
        # Subsample for visualization if too many points
        if len(all_heights) > 5000:
            idx = np.random.choice(len(all_heights), 5000, replace=False)
            h_plot, s_plot = all_heights[idx], all_speeds[idx]
        else:
            h_plot, s_plot = all_heights, all_speeds
        
        ax3.scatter(h_plot, s_plot, s=5, alpha=0.3, c='#264653')
        
        # Compute correlation on all data
        r_pooled, p_pooled = pearsonr(all_heights, all_speeds)
        slope, intercept, _, _, _ = linregress(all_heights, all_speeds)
        x_line = np.linspace(all_heights.min(), all_heights.max(), 100)
        ax3.plot(x_line, slope * x_line + intercept, 'r-', lw=2, 
                 label=f'r={r_pooled:.3f}, p={p_pooled:.2e}\nn={len(all_heights):,} frames')
        
        ax3.set_xlabel('Body Height (mm)', fontsize=12)
        ax3.set_ylabel('Instantaneous Speed (mm/s)', fontsize=12)
        ax3.set_title('Frame-Level: All Bouts Pooled', fontsize=13)
        ax3.legend(loc='best', fontsize=10)
        ax3.grid(True, alpha=0.3)
    else:
        r_pooled, p_pooled = np.nan, np.nan
    
    # ===== Panel 4: Summary statistics table =====
    ax4 = axes[1, 1]
    ax4.axis('off')
    
    # Calculate summary statistics
    n_positive = (all_bouts_df['height_speed_pearson_r'] > 0).sum()
    n_negative = (all_bouts_df['height_speed_pearson_r'] < 0).sum()
    n_significant = (all_bouts_df['height_speed_pearson_p'] < 0.05).sum()
    
    stats_text = f"""
HEIGHT-SPEED CORRELATION SUMMARY
{'='*45}

BOUT-LEVEL ANALYSIS (n={len(all_bouts_df)} bouts):
  Mean body height:     {all_bouts_df['scut_z_mean'].mean():.2f} ± {all_bouts_df['scut_z_mean'].std():.2f} mm
  Mean speed:           {all_bouts_df['mean_speed_mm_s'].mean():.1f} ± {all_bouts_df['mean_speed_mm_s'].std():.1f} mm/s
  Bout-level r:         {r_val:.3f} (p={p_val:.4f})
  
WITHIN-BOUT CORRELATIONS:
  Mean Pearson r:       {all_bouts_df['height_speed_pearson_r'].mean():.3f}
  Median Pearson r:     {all_bouts_df['height_speed_pearson_r'].median():.3f}
  Range:                [{all_bouts_df['height_speed_pearson_r'].min():.3f}, {all_bouts_df['height_speed_pearson_r'].max():.3f}]
  
  Positive correlations: {n_positive} bouts ({100*n_positive/len(all_bouts_df):.1f}%)
  Negative correlations: {n_negative} bouts ({100*n_negative/len(all_bouts_df):.1f}%)
  Significant (p<0.05):  {n_significant} bouts ({100*n_significant/len(all_bouts_df):.1f}%)

POOLED FRAME-LEVEL (n={len(all_heights):,} frames):
  Pearson r:            {r_pooled:.3f} (p={p_pooled:.2e})
  
INTERPRETATION:
  {'Higher body = faster walking' if r_pooled > 0.1 else 'Lower body = faster walking' if r_pooled < -0.1 else 'No strong height-speed relationship'}
"""
    
    ax4.text(0.05, 0.95, stats_text, transform=ax4.transAxes, fontsize=11,
             verticalalignment='top', fontfamily='monospace',
             bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    
    plt.suptitle('Body Height vs Walking Speed Correlation Analysis (ALL BOUTS)', 
                 fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    
    # Save figure
    if output_dir:
        save_path = Path(output_dir) / "height_speed_correlation_analysis.pdf"
        fig.savefig(save_path, format='pdf', dpi=PDF_DPI, bbox_inches='tight')
        print(f"\nCorrelation analysis saved: {save_path}")
    
    plt.show()
    
    return fig, all_bouts_df

# Run the correlation analysis on ALL valid bouts
if 'valid_bouts' in dir() and len(valid_bouts) > 0:
    print("\n" + "=" * 70)
    print("HEIGHT-SPEED CORRELATION ANALYSIS (ALL VALID BOUTS)")
    print("=" * 70)
    correlation_fig, all_bouts_correlation_df = analyze_height_speed_correlation(
        scutellum_data, valid_bouts, fps=FPS, output_dir=OUTPUT_DIR
    )
else:
    print("No valid bouts available. Run the bout detection first.")

In [ ]:
# ============================================================================
# VIEW REJECTED BOUTS (OPTIONAL)
# ============================================================================

if len(rejected_bouts) > 0:
    print(f"\nRejected bouts: {len(rejected_bouts)}")
    print("-" * 70)
    for i, bout in enumerate(rejected_bouts):  # Show first 10
        print(f"\nBout {i+1}: frames {bout['start'] + frame_offset}-{bout['end'] + frame_offset}")
        print(f"  Duration: {bout['n_frames']} frames")
        print(f"  Min cycles: {bout['min_cycles']}")
        print(f"  Distance: {bout['total_distance_mm']:.2f} mm")
        print(f"  Rejection: {bout['rejection_reason']}")
    
    if len(rejected_bouts) > 10:
        print(f"\n... and {len(rejected_bouts) - 10} more rejected bouts")
else:
    print("No rejected bouts.")

## Confidence overview


In [ ]:
# ============================================================================
# CONFIDENCE OVERVIEW: Mean confidence per keypoint
# ============================================================================
# Quick overview of prediction quality across all 50 keypoints

def plot_confidence_overview(df, output_dir=None):
    """
    Plot mean confidence value for each keypoint across all frames.
    """
    # Get all keypoint names
    all_kp_names = get_all_keypoint_names(df)
    print(f"Found {len(all_kp_names)} keypoints")
    
    # Extract confidence for each keypoint
    kp_conf_stats = []
    all_conf_arrays = []  # Store all confidence arrays for time series
    
    for kp in all_kp_names:
        try:
            _, _, _, conf = extract_xyzc(df, kp, scale=1.0)  # scale doesn't affect conf
            all_conf_arrays.append(conf)
            kp_conf_stats.append({
                'keypoint': kp,
                'mean_conf': np.nanmean(conf),
                'std_conf': np.nanstd(conf),
                'min_conf': np.nanmin(conf),
                'median_conf': np.nanmedian(conf),
                'pct_above_85': 100 * np.nanmean(conf >= 0.85),
                'pct_above_90': 100 * np.nanmean(conf >= 0.90),
            })
        except Exception as e:
            print(f"  Warning: Could not extract {kp}: {e}")
    
    conf_df = pd.DataFrame(kp_conf_stats)
    conf_df = conf_df.sort_values('mean_conf', ascending=True).reset_index(drop=True)
    
    # Compute mean confidence across all keypoints per frame
    all_conf_matrix = np.array(all_conf_arrays)  # shape: (n_keypoints, n_frames)
    mean_conf_per_frame = np.nanmean(all_conf_matrix, axis=0)  # shape: (n_frames,)
    frame_numbers = np.arange(len(mean_conf_per_frame))
    
    # === VISUALIZATION ===
    fig = plt.figure(figsize=(16, 14))
    
    # Create grid: top row for time series (spans full width), bottom 2x2 for other plots
    gs = fig.add_gridspec(3, 2, height_ratios=[1, 1, 1], hspace=0.3, wspace=0.3)
    
    # Panel 0 (top, full width): Time series of mean confidence
    ax0 = fig.add_subplot(gs[0, :])
    # Only plot where we have valid values
    valid_mask = ~np.isnan(mean_conf_per_frame)
    ax0.plot(frame_numbers[valid_mask], mean_conf_per_frame[valid_mask], 
             color='#264653', linewidth=0.5, alpha=0.8)
    ax0.axhline(y=0.85, color='red', ls='--', lw=2, alpha=0.7, label='Threshold (0.85)')
    ax0.axhline(y=np.nanmean(mean_conf_per_frame), color='blue', ls='-', lw=1.5, alpha=0.7,
                label=f'Overall mean: {np.nanmean(mean_conf_per_frame):.3f}')
    ax0.set_xlabel('Frame Number', fontsize=11)
    ax0.set_ylabel('Mean Confidence\n(across all keypoints)', fontsize=11)
    ax0.set_title('Mean Confidence Over Time', fontsize=12)
    ax0.set_ylim(0, 1.05)
    ax0.legend(loc='lower right', fontsize=9)
    ax0.grid(True, alpha=0.3)
    
    # Panel 1: Bar chart of mean confidence per keypoint
    ax1 = fig.add_subplot(gs[1, 0])
    colors = plt.cm.RdYlGn(conf_df['mean_conf'])  # Red=low, Green=high
    bars = ax1.barh(range(len(conf_df)), conf_df['mean_conf'], color=colors, edgecolor='white', linewidth=0.5)
    ax1.set_yticks(range(len(conf_df)))
    ax1.set_yticklabels(conf_df['keypoint'], fontsize=7)
    ax1.set_xlabel('Mean Confidence', fontsize=11)
    ax1.set_title('Mean Confidence per Keypoint (sorted)', fontsize=12)
    ax1.axvline(x=0.85, color='red', ls='--', lw=2, alpha=0.7, label='Threshold (0.85)')
    ax1.axvline(x=conf_df['mean_conf'].mean(), color='blue', ls='-', lw=2, alpha=0.7, 
                label=f'Overall mean: {conf_df["mean_conf"].mean():.3f}')
    ax1.set_xlim(0, 1)
    ax1.legend(loc='lower right', fontsize=9)
    ax1.grid(True, alpha=0.3, axis='x')
    
    # Panel 2: Distribution of mean confidences
    ax2 = fig.add_subplot(gs[1, 1])
    ax2.hist(conf_df['mean_conf'], bins=20, color='#2A9D8F', edgecolor='white', alpha=0.8)
    ax2.axvline(x=0.85, color='red', ls='--', lw=2, alpha=0.7, label='Threshold (0.85)')
    ax2.axvline(x=conf_df['mean_conf'].mean(), color='blue', ls='-', lw=2, 
                label=f'Mean: {conf_df["mean_conf"].mean():.3f}')
    ax2.set_xlabel('Mean Confidence', fontsize=11)
    ax2.set_ylabel('Number of Keypoints', fontsize=11)
    ax2.set_title('Distribution of Mean Confidences', fontsize=12)
    ax2.legend(fontsize=10)
    ax2.grid(True, alpha=0.3)
    
    # Panel 3: % frames above threshold per keypoint
    ax3 = fig.add_subplot(gs[2, 0])
    colors_pct = plt.cm.RdYlGn(conf_df['pct_above_85'] / 100)
    ax3.barh(range(len(conf_df)), conf_df['pct_above_85'], color=colors_pct, edgecolor='white', linewidth=0.5)
    ax3.set_yticks(range(len(conf_df)))
    ax3.set_yticklabels(conf_df['keypoint'], fontsize=7)
    ax3.set_xlabel('% Frames with Confidence ≥ 0.85', fontsize=11)
    ax3.set_title('Percentage of High-Confidence Frames per Keypoint', fontsize=12)
    ax3.axvline(x=conf_df['pct_above_85'].mean(), color='blue', ls='-', lw=2, alpha=0.7,
                label=f'Mean: {conf_df["pct_above_85"].mean():.1f}%')
    ax3.set_xlim(0, 100)
    ax3.legend(loc='lower right', fontsize=9)
    ax3.grid(True, alpha=0.3, axis='x')
    
    # Panel 4: Summary statistics
    ax4 = fig.add_subplot(gs[2, 1])
    ax4.axis('off')
    
    # Identify worst keypoints
    worst_5 = conf_df.head(5)
    best_5 = conf_df.tail(5).iloc[::-1]
    
    summary_text = f"""
PREDICTION CONFIDENCE OVERVIEW
{'='*50}

Total keypoints analyzed: {len(conf_df)}
Total frames: {len(df):,}

OVERALL STATISTICS:
  Mean confidence:     {conf_df['mean_conf'].mean():.3f}
  Median confidence:   {conf_df['mean_conf'].median():.3f}
  Min mean conf:       {conf_df['mean_conf'].min():.3f}
  Max mean conf:       {conf_df['mean_conf'].max():.3f}

THRESHOLD ANALYSIS (0.85):
  Keypoints with mean ≥ 0.85:  {(conf_df['mean_conf'] >= 0.85).sum()}/{len(conf_df)}
  Mean % frames above 0.85:    {conf_df['pct_above_85'].mean():.1f}%

5 LOWEST CONFIDENCE KEYPOINTS:
{chr(10).join([f"  {row['keypoint']:20s}  {row['mean_conf']:.3f}" for _, row in worst_5.iterrows()])}

5 HIGHEST CONFIDENCE KEYPOINTS:
{chr(10).join([f"  {row['keypoint']:20s}  {row['mean_conf']:.3f}" for _, row in best_5.iterrows()])}
"""
    ax4.text(0.02, 0.98, summary_text, transform=ax4.transAxes, fontsize=10,
             verticalalignment='top', fontfamily='monospace',
             bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))
    
    plt.suptitle('SLEAP 3D Prediction Confidence Analysis', fontsize=14, fontweight='bold')
    
    # Save
    if output_dir:
        save_path = Path(output_dir) / 'confidence_overview.pdf'
        fig.savefig(save_path, format='pdf', dpi=PDF_DPI, bbox_inches='tight')
        print(f"\nSaved: {save_path}")
    
    plt.show()
    
    return conf_df, mean_conf_per_frame


# Run the confidence overview
if 'df' in dir():
    print("\n" + "=" * 70)
    print("CONFIDENCE OVERVIEW")
    print("=" * 70)
    conf_overview_df, mean_conf_timeseries = plot_confidence_overview(df, output_dir=OUTPUT_DIR)
    
    # Save the data
    conf_csv = OUTPUT_DIR / 'confidence_per_keypoint.csv'
    conf_overview_df.to_csv(conf_csv, index=False)
    print(f"Confidence data saved: {conf_csv}")
else:
    print("DataFrame 'df' not found. Run data loading cell first.")

## WALKING BOUTS SUMMARY STATISTICS


In [ ]:
# ============================================================================
# WALKING BOUTS SUMMARY STATISTICS
# ============================================================================
# Comprehensive overview of all detected walking bouts

def plot_bout_statistics(summary_csv_path, scutellum_data=None, valid_bouts=None, fps=FPS, output_dir=None):
    """
    Generate comprehensive statistics and visualizations for walking bouts.
    """
    # Load summary data
    bout_df = pd.read_csv(summary_csv_path)
    n_bouts = len(bout_df)
    
    print(f"Loaded {n_bouts} walking bouts from summary")
    print("=" * 70)
    
    # Calculate additional metrics
    bout_df['mid_frame'] = (bout_df['start_frame'] + bout_df['end_frame']) / 2
    bout_df['time_in_video_s'] = bout_df['mid_frame'] / fps
    bout_df['time_in_video_min'] = bout_df['time_in_video_s'] / 60
    bout_df['straightness'] = bout_df['net_displacement_mm'] / bout_df['total_distance_mm']
    
    # Get X position for left/right analysis (from scutellum data if available)
    if scutellum_data is not None and valid_bouts is not None:
        bout_x_positions = []
        for bout in valid_bouts:
            start, end = bout['start'], bout['end']
            mean_x = np.nanmean(scutellum_data['x'][start:end])
            bout_x_positions.append(mean_x)
        bout_df['mean_x_mm'] = bout_x_positions
        
        # Determine chamber center (assume center is at median X)
        chamber_center_x = np.nanmedian(bout_df['mean_x_mm'])
        bout_df['side'] = np.where(bout_df['mean_x_mm'] < chamber_center_x, 'Left', 'Right')
    
    # === CREATE FIGURE ===
    fig = plt.figure(figsize=(18, 16))
    gs = fig.add_gridspec(4, 3, hspace=0.35, wspace=0.3)
    
    # --- Row 1 ---
    
    # Panel 1: Temporal distribution of bouts
    ax1 = fig.add_subplot(gs[0, 0])
    ax1.hist(bout_df['time_in_video_min'], bins=30, color='#264653', edgecolor='white', alpha=0.8)
    ax1.set_xlabel('Time in Video (minutes)', fontsize=10)
    ax1.set_ylabel('Number of Bouts', fontsize=10)
    ax1.set_title('Temporal Distribution of Bouts', fontsize=11)
    ax1.grid(True, alpha=0.3)
    
    # Panel 2: Bout duration distribution
    ax2 = fig.add_subplot(gs[0, 1])
    ax2.hist(bout_df['duration_s'], bins=30, color='#2A9D8F', edgecolor='white', alpha=0.8)
    ax2.axvline(bout_df['duration_s'].mean(), color='red', ls='--', lw=2, 
                label=f'Mean: {bout_df["duration_s"].mean():.2f}s')
    ax2.axvline(bout_df['duration_s'].median(), color='orange', ls='--', lw=2,
                label=f'Median: {bout_df["duration_s"].median():.2f}s')
    ax2.set_xlabel('Duration (seconds)', fontsize=10)
    ax2.set_ylabel('Number of Bouts', fontsize=10)
    ax2.set_title('Bout Duration Distribution', fontsize=11)
    ax2.legend(fontsize=9)
    ax2.grid(True, alpha=0.3)
    
    # Panel 3: Mean speed distribution
    ax3 = fig.add_subplot(gs[0, 2])
    ax3.hist(bout_df['mean_speed_mm_s'], bins=30, color='#E9C46A', edgecolor='white', alpha=0.8)
    ax3.axvline(bout_df['mean_speed_mm_s'].mean(), color='red', ls='--', lw=2,
                label=f'Mean: {bout_df["mean_speed_mm_s"].mean():.1f} mm/s')
    ax3.set_xlabel('Mean Speed (mm/s)', fontsize=10)
    ax3.set_ylabel('Number of Bouts', fontsize=10)
    ax3.set_title('Mean Speed Distribution', fontsize=11)
    ax3.legend(fontsize=9)
    ax3.grid(True, alpha=0.3)
    
    # --- Row 2 ---
    
    # Panel 4: Max speed distribution
    ax4 = fig.add_subplot(gs[1, 0])
    ax4.hist(bout_df['max_speed_mm_s'], bins=30, color='#F4A261', edgecolor='white', alpha=0.8)
    ax4.axvline(bout_df['max_speed_mm_s'].mean(), color='red', ls='--', lw=2,
                label=f'Mean: {bout_df["max_speed_mm_s"].mean():.1f} mm/s')
    ax4.set_xlabel('Max Speed (mm/s)', fontsize=10)
    ax4.set_ylabel('Number of Bouts', fontsize=10)
    ax4.set_title('Max Speed Distribution', fontsize=11)
    ax4.legend(fontsize=9)
    ax4.grid(True, alpha=0.3)
    
    # Panel 5: Total distance distribution
    ax5 = fig.add_subplot(gs[1, 1])
    ax5.hist(bout_df['total_distance_mm'], bins=30, color='#E76F51', edgecolor='white', alpha=0.8)
    ax5.axvline(bout_df['total_distance_mm'].mean(), color='red', ls='--', lw=2,
                label=f'Mean: {bout_df["total_distance_mm"].mean():.1f} mm')
    ax5.set_xlabel('Total Distance (mm)', fontsize=10)
    ax5.set_ylabel('Number of Bouts', fontsize=10)
    ax5.set_title('Total Distance Distribution', fontsize=11)
    ax5.legend(fontsize=9)
    ax5.grid(True, alpha=0.3)
    
    # Panel 6: Net displacement distribution
    ax6 = fig.add_subplot(gs[1, 2])
    ax6.hist(bout_df['net_displacement_mm'], bins=30, color='#457B9D', edgecolor='white', alpha=0.8)
    ax6.axvline(bout_df['net_displacement_mm'].mean(), color='red', ls='--', lw=2,
                label=f'Mean: {bout_df["net_displacement_mm"].mean():.1f} mm')
    ax6.set_xlabel('Net Displacement (mm)', fontsize=10)
    ax6.set_ylabel('Number of Bouts', fontsize=10)
    ax6.set_title('Net Displacement Distribution', fontsize=11)
    ax6.legend(fontsize=9)
    ax6.grid(True, alpha=0.3)
    
    # --- Row 3 ---
    
    # Panel 7: Left/Right distribution (if X data available)
    ax7 = fig.add_subplot(gs[2, 0])
    if 'side' in bout_df.columns:
        side_counts = bout_df['side'].value_counts()
        colors = ['#1D3557', '#A8DADC']
        ax7.bar(side_counts.index, side_counts.values, color=colors, edgecolor='white')
        ax7.set_xlabel('Side of Chamber', fontsize=10)
        ax7.set_ylabel('Number of Bouts', fontsize=10)
        ax7.set_title(f'Left vs Right Distribution\n(center at X={chamber_center_x:.1f}mm)', fontsize=11)
        for i, (side, count) in enumerate(side_counts.items()):
            ax7.text(i, count + 1, f'{count}\n({100*count/n_bouts:.0f}%)', ha='center', fontsize=10)
        ax7.grid(True, alpha=0.3, axis='y')
    else:
        ax7.text(0.5, 0.5, 'X position data\nnot available', ha='center', va='center', 
                 fontsize=12, transform=ax7.transAxes)
        ax7.set_title('Left vs Right Distribution', fontsize=11)
    
    # Panel 8: Straightness index distribution
    ax8 = fig.add_subplot(gs[2, 1])
    ax8.hist(bout_df['straightness'], bins=30, color='#6A994E', edgecolor='white', alpha=0.8)
    ax8.axvline(bout_df['straightness'].mean(), color='red', ls='--', lw=2,
                label=f'Mean: {bout_df["straightness"].mean():.2f}')
    ax8.set_xlabel('Straightness Index (net/total)', fontsize=10)
    ax8.set_ylabel('Number of Bouts', fontsize=10)
    ax8.set_title('Path Straightness Distribution\n(1.0 = perfectly straight)', fontsize=11)
    ax8.legend(fontsize=9)
    ax8.grid(True, alpha=0.3)
    
    # Panel 9: Walking cycles distribution
    ax9 = fig.add_subplot(gs[2, 2])
    ax9.hist(bout_df['min_cycles'], bins=range(int(bout_df['min_cycles'].min()), 
             int(bout_df['min_cycles'].max()) + 2), color='#BC6C25', edgecolor='white', alpha=0.8)
    ax9.axvline(bout_df['min_cycles'].mean(), color='red', ls='--', lw=2,
                label=f'Mean: {bout_df["min_cycles"].mean():.1f}')
    ax9.set_xlabel('Minimum Walking Cycles', fontsize=10)
    ax9.set_ylabel('Number of Bouts', fontsize=10)
    ax9.set_title('Walking Cycles per Bout', fontsize=11)
    ax9.legend(fontsize=9)
    ax9.grid(True, alpha=0.3)
    
    # --- Row 4 ---
    
    # Panel 10: Speed vs Duration scatter
    ax10 = fig.add_subplot(gs[3, 0])
    scatter = ax10.scatter(bout_df['duration_s'], bout_df['mean_speed_mm_s'], 
                           c=bout_df['time_in_video_min'], cmap='viridis', 
                           s=50, alpha=0.7, edgecolor='white')
    ax10.set_xlabel('Duration (s)', fontsize=10)
    ax10.set_ylabel('Mean Speed (mm/s)', fontsize=10)
    ax10.set_title('Speed vs Duration\n(color = time in video)', fontsize=11)
    plt.colorbar(scatter, ax=ax10, label='Time (min)')
    ax10.grid(True, alpha=0.3)
    
    # Panel 11: Body height distribution
    ax11 = fig.add_subplot(gs[3, 1])
    ax11.hist(bout_df['scut_z_mean'], bins=30, color='#9B2335', edgecolor='white', alpha=0.8)
    ax11.axvline(bout_df['scut_z_mean'].mean(), color='blue', ls='--', lw=2,
                 label=f'Mean: {bout_df["scut_z_mean"].mean():.3f} mm')
    ax11.set_xlabel('Mean Body Height (mm)', fontsize=10)
    ax11.set_ylabel('Number of Bouts', fontsize=10)
    ax11.set_title('Body Height During Bouts', fontsize=11)
    ax11.legend(fontsize=9)
    ax11.grid(True, alpha=0.3)
    
    # Panel 12: Summary statistics text
    ax12 = fig.add_subplot(gs[3, 2])
    ax12.axis('off')
    
    total_walking_time = bout_df['duration_s'].sum()
    total_distance = bout_df['total_distance_mm'].sum()
    video_duration = bout_df['end_frame'].max() / fps
    
    summary_text = f"""
WALKING BOUTS SUMMARY
{'='*45}

GENERAL:
  Total bouts detected:    {n_bouts}
  Video duration:          {video_duration/60:.1f} min
  Total walking time:      {total_walking_time:.1f} s ({100*total_walking_time/video_duration:.1f}%)
  Total distance walked:   {total_distance:.1f} mm ({total_distance/10:.1f} cm)

DURATION (seconds):
  Mean ± Std:   {bout_df['duration_s'].mean():.2f} ± {bout_df['duration_s'].std():.2f}
  Median:       {bout_df['duration_s'].median():.2f}
  Range:        {bout_df['duration_s'].min():.2f} - {bout_df['duration_s'].max():.2f}

SPEED (mm/s):
  Mean speed:   {bout_df['mean_speed_mm_s'].mean():.1f} ± {bout_df['mean_speed_mm_s'].std():.1f}
  Max speed:    {bout_df['max_speed_mm_s'].mean():.1f} ± {bout_df['max_speed_mm_s'].std():.1f}
  Peak max:     {bout_df['max_speed_mm_s'].max():.1f}

DISTANCE (mm):
  Per bout:     {bout_df['total_distance_mm'].mean():.1f} ± {bout_df['total_distance_mm'].std():.1f}
  Net displ.:   {bout_df['net_displacement_mm'].mean():.1f} ± {bout_df['net_displacement_mm'].std():.1f}
  Straightness: {bout_df['straightness'].mean():.2f} ± {bout_df['straightness'].std():.2f}

BODY HEIGHT (mm):
  Mean:         {bout_df['scut_z_mean'].mean():.3f} ± {bout_df['scut_z_mean'].std():.3f}
  Range:        {bout_df['scut_z_mean'].min():.3f} - {bout_df['scut_z_mean'].max():.3f}
"""
    ax12.text(0.02, 0.98, summary_text, transform=ax12.transAxes, fontsize=9,
              verticalalignment='top', fontfamily='monospace',
              bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.9))
    
    plt.suptitle(f'Walking Bouts Statistics Overview (n={n_bouts} bouts)', fontsize=14, fontweight='bold')
    
    # Save
    if output_dir:
        save_path = Path(output_dir) / 'bout_statistics_overview.pdf'
        fig.savefig(save_path, format='pdf', dpi=PDF_DPI, bbox_inches='tight')
        print(f"\nSaved: {save_path}")
    
    plt.show()
    
    # Print detailed statistics
    print("\n" + "=" * 70)
    print("DETAILED STATISTICS")
    print("=" * 70)
    
    print("\nDuration (seconds):")
    print(bout_df['duration_s'].describe())
    
    print("\nMean Speed (mm/s):")
    print(bout_df['mean_speed_mm_s'].describe())
    
    print("\nTotal Distance (mm):")
    print(bout_df['total_distance_mm'].describe())
    
    if 'side' in bout_df.columns:
        print("\n" + "=" * 70)
        print("LEFT vs RIGHT COMPARISON")
        print("=" * 70)
        for side in ['Left', 'Right']:
            side_data = bout_df[bout_df['side'] == side]
            print(f"\n{side} side ({len(side_data)} bouts):")
            print(f"  Mean duration:  {side_data['duration_s'].mean():.2f}s")
            print(f"  Mean speed:     {side_data['mean_speed_mm_s'].mean():.1f} mm/s")
            print(f"  Mean distance:  {side_data['total_distance_mm'].mean():.1f} mm")
    
    return bout_df


# Run the bout statistics analysis
summary_path = Path(OUTPUT_DIR.parent) / 'walking_bouts_summary.csv'

if summary_path.exists():
    print("\n" + "=" * 70)
    print("WALKING BOUTS STATISTICS OVERVIEW")
    print("=" * 70)
    
    # Pass scutellum_data and valid_bouts if available for X position analysis
    if 'scutellum_data' in dir() and 'valid_bouts' in dir():
        bout_stats_df = plot_bout_statistics(
            summary_path, 
            scutellum_data=scutellum_data, 
            valid_bouts=valid_bouts,
            fps=FPS,
            output_dir=OUTPUT_DIR
        )
    else:
        bout_stats_df = plot_bout_statistics(
            summary_path,
            fps=FPS,
            output_dir=OUTPUT_DIR
        )
else:
    print(f"Summary file not found: {summary_path}")
    print("Run bout detection and analysis first.")

In [ ]:
# ============================================================================
# GAIT PHASE EXTRACTION & SCUTELLUM Z OSCILLATION ANALYSIS
# ============================================================================
# Analyze the rhythm of vertical body oscillations (Scutellum Z) during walking
# and its relationship to leg step phases.
#
# Key questions:
# 1. Does the Scutellum show periodic vertical oscillations during walking?
# 2. How do these oscillations relate to leg step phases?
# 3. Does the oscillation amplitude/frequency correlate with walking speed?

def extract_gait_phase_from_leg(z_signal, fps=FPS, smooth_window=5):
    """
    Extract instantaneous phase from a leg's Z trajectory using Hilbert transform.
    
    The phase indicates where the leg is in its step cycle:
    - Phase ~0: Swing phase start (leg lifting off)
    - Phase ~π/2: Mid-swing (leg at peak height)
    - Phase ~π: Swing phase end / stance start (leg touching down)
    - Phase ~3π/2: Mid-stance (leg on ground)
    
    Returns:
        phase: Instantaneous phase (0 to 2π) for each frame
        amplitude: Instantaneous amplitude (oscillation strength)
        z_detrended: Detrended Z signal used for analysis
    """
    # Handle NaN values
    z_clean = z_signal.copy()
    nan_mask = np.isnan(z_clean)
    if nan_mask.all():
        return np.full_like(z_signal, np.nan), np.full_like(z_signal, np.nan), z_clean
    
    if nan_mask.any():
        valid_idx = np.where(~nan_mask)[0]
        z_clean[nan_mask] = np.interp(np.where(nan_mask)[0], valid_idx, z_clean[valid_idx])
    
    # Smooth to reduce noise
    z_smooth = uniform_filter1d(z_clean, size=smooth_window)
    
    # Detrend by subtracting low-pass filtered signal (removes drift)
    # Butterworth low-pass at ~1 Hz to capture baseline
    nyq = fps / 2
    low_cutoff = 1.0 / nyq  # 1 Hz normalized
    if low_cutoff < 1.0:  # Only apply if valid
        b, a = butter(2, low_cutoff, btype='low')
        baseline = filtfilt(b, a, z_smooth, padlen=min(len(z_smooth)-1, 100))
    else:
        baseline = np.mean(z_smooth) * np.ones_like(z_smooth)
    
    z_detrended = z_smooth - baseline
    
    # Apply Hilbert transform to get analytic signal
    analytic_signal = hilbert(z_detrended)
    
    # Extract instantaneous amplitude and phase
    amplitude = np.abs(analytic_signal)
    phase = np.angle(analytic_signal)  # Range: -π to π
    
    # Convert to 0 to 2π range
    phase = np.mod(phase, 2 * np.pi)
    
    return phase, amplitude, z_detrended


def compute_composite_gait_phase(leg_tip_data, start, end, fps=FPS):
    """
    Compute composite gait phase from all 6 legs.
    
    For tripod gait, legs alternate in two groups:
    - Group A (tripod 1): T1L, T2R, T3L
    - Group B (tripod 2): T1R, T2L, T3R
    
    Returns:
        tripod1_phase: Average phase of tripod 1 legs
        tripod2_phase: Average phase of tripod 2 legs
        all_phases: Dict of individual leg phases
        gait_cycle_frames: Estimated frames per gait cycle
    """
    tripod1_legs = ["T1L_TaTip", "T2R_TaTip", "T3L_TaTip"]
    tripod2_legs = ["T1R_TaTip", "T2L_TaTip", "T3R_TaTip"]
    
    all_phases = {}
    tripod1_phases = []
    tripod2_phases = []
    
    for tip in LEG_TIPS:
        z_bout = leg_tip_data[tip]['z'][start:end+1]
        phase, amplitude, _ = extract_gait_phase_from_leg(z_bout, fps)
        all_phases[tip] = {'phase': phase, 'amplitude': amplitude}
        
        if tip in tripod1_legs:
            tripod1_phases.append(phase)
        else:
            tripod2_phases.append(phase)
    
    # Average phases using circular mean
    def circular_mean(phases_list):
        sin_sum = np.mean([np.sin(p) for p in phases_list], axis=0)
        cos_sum = np.mean([np.cos(p) for p in phases_list], axis=0)
        return np.arctan2(sin_sum, cos_sum) % (2 * np.pi)
    
    tripod1_phase = circular_mean(tripod1_phases)
    tripod2_phase = circular_mean(tripod2_phases)
    
    # Estimate gait cycle duration from phase unwrapping
    phase_unwrapped = np.unwrap(tripod1_phase)
    total_cycles = (phase_unwrapped[-1] - phase_unwrapped[0]) / (2 * np.pi)
    n_frames = end - start + 1
    gait_cycle_frames = n_frames / max(total_cycles, 1) if total_cycles > 0 else np.nan
    
    return tripod1_phase, tripod2_phase, all_phases, gait_cycle_frames


def analyze_scutellum_z_periodicity(scutellum_data, start, end, fps=FPS):
    """
    Analyze periodicity in Scutellum Z signal using FFT.
    
    Returns:
        freqs: Frequency array (Hz)
        psd: Power spectral density
        dominant_freq: Dominant oscillation frequency (Hz)
        dominant_period_ms: Dominant period in milliseconds
        z_oscillation: Detrended Scutellum Z showing oscillations
    """
    scut_z = scutellum_data['z'][start:end+1].copy()
    
    # Handle NaN
    nan_mask = np.isnan(scut_z)
    if nan_mask.any():
        valid_idx = np.where(~nan_mask)[0]
        if len(valid_idx) < 10:
            return None, None, np.nan, np.nan, scut_z
        scut_z[nan_mask] = np.interp(np.where(nan_mask)[0], valid_idx, scut_z[valid_idx])
    
    # Detrend
    z_smooth = uniform_filter1d(scut_z, size=5)
    nyq = fps / 2
    low_cutoff = 1.0 / nyq
    if low_cutoff < 1.0 and len(z_smooth) > 20:
        b, a = butter(2, low_cutoff, btype='low')
        baseline = filtfilt(b, a, z_smooth, padlen=min(len(z_smooth)-1, 100))
    else:
        baseline = np.mean(z_smooth) * np.ones_like(z_smooth)
    
    z_oscillation = z_smooth - baseline
    
    # Compute power spectral density using Welch's method
    nperseg = min(len(z_oscillation) // 2, 256)
    if nperseg < 16:
        return None, None, np.nan, np.nan, z_oscillation
    
    freqs, psd = welch(z_oscillation, fs=fps, nperseg=nperseg)
    
    # Find dominant frequency (exclude DC component, focus on gait-relevant range 5-40 Hz)
    gait_mask = (freqs >= 5) & (freqs <= 40)
    if gait_mask.sum() > 0:
        gait_freqs = freqs[gait_mask]
        gait_psd = psd[gait_mask]
        dominant_idx = np.argmax(gait_psd)
        dominant_freq = gait_freqs[dominant_idx]
        dominant_period_ms = 1000 / dominant_freq if dominant_freq > 0 else np.nan
    else:
        dominant_freq = np.nan
        dominant_period_ms = np.nan
    
    return freqs, psd, dominant_freq, dominant_period_ms, z_oscillation


print("Gait phase and periodicity functions defined.")


In [ ]:
# ============================================================================
# VISUALIZE SCUTELLUM Z OSCILLATIONS ALIGNED WITH GAIT PHASE
# ============================================================================
# Analyze a single bout in detail to see the relationship between
# Scutellum Z and leg step phases.

def plot_gait_phase_analysis(bout_info, leg_tip_data, scutellum_data, 
                              frame_offset=0, fps=FPS, save_path=None):
    """
    Create detailed visualization of gait phase vs Scutellum Z oscillations.
    
    Panels:
    1. Leg Z trajectories with detected swing phases
    2. Gait phase (tripod groups)
    3. Scutellum Z oscillations (detrended) vs gait phase
    4. Power spectrum of Scutellum Z
    """
    start = bout_info['start']
    end = bout_info['end']
    bout_idx = bout_info.get('bout_idx', '?')
    
    actual_start = start + frame_offset
    actual_end = end + frame_offset
    frames = np.arange(actual_start, actual_end + 1)
    n_frames = len(frames)
    time_ms = np.arange(n_frames) / fps * 1000  # Time in ms
    
    # Extract gait phases
    tripod1_phase, tripod2_phase, all_phases, gait_cycle_frames = \
        compute_composite_gait_phase(leg_tip_data, start, end, fps)
    
    # Analyze Scutellum Z periodicity
    freqs, psd, dominant_freq, dominant_period_ms, z_oscillation = \
        analyze_scutellum_z_periodicity(scutellum_data, start, end, fps)
    
    # Get raw Scutellum Z for comparison
    scut_z_raw = scutellum_data['z'][start:end+1]
    
    # Compute speed
    speed, mean_speed, _ = compute_instant_speed(scutellum_data, start, end, fps)
    
    # Create figure
    fig = plt.figure(figsize=(16, 14))
    gs = fig.add_gridspec(5, 2, height_ratios=[1, 0.8, 1, 0.8, 1], width_ratios=[2, 1], hspace=0.35, wspace=0.3)
    
    # Color scheme
    leg_colors = {
        'T1L_TaTip': '#E63946', 'T1R_TaTip': '#457B9D',
        'T2L_TaTip': '#F4A261', 'T2R_TaTip': '#2A9D8F',
        'T3L_TaTip': '#9B2226', 'T3R_TaTip': '#1D3557',
    }
    tripod1_color = '#8B0000'  # Dark red
    tripod2_color = '#00008B'  # Dark blue
    
    # ===== PANEL 1: Leg Z trajectories =====
    ax1 = fig.add_subplot(gs[0, :])
    for tip, color in leg_colors.items():
        z = leg_tip_data[tip]['z'][start:end+1]
        label = tip.replace('_TaTip', '')
        linestyle = '-' if tip in ["T1L_TaTip", "T2R_TaTip", "T3L_TaTip"] else '--'
        ax1.plot(time_ms, z, label=label, color=color, lw=1.2, alpha=0.8, ls=linestyle)
    ax1.set_ylabel('Leg tip Z (mm)', fontsize=11)
    ax1.legend(loc='upper right', ncol=6, fontsize=8)
    ax1.set_title(f'Bout {bout_idx}: Leg Step Patterns (solid=Tripod1, dashed=Tripod2)', fontsize=12)
    ax1.grid(True, alpha=0.3)
    
    # ===== PANEL 2: Tripod gait phases =====
    ax2 = fig.add_subplot(gs[1, :], sharex=ax1)
    ax2.plot(time_ms, tripod1_phase, color=tripod1_color, lw=1.5, alpha=0.8, label='Tripod 1 (T1L,T2R,T3L)')
    ax2.plot(time_ms, tripod2_phase, color=tripod2_color, lw=1.5, alpha=0.8, label='Tripod 2 (T1R,T2L,T3R)')
    ax2.set_ylabel('Phase (rad)', fontsize=11)
    ax2.set_yticks([0, np.pi/2, np.pi, 3*np.pi/2, 2*np.pi])
    ax2.set_yticklabels(['0', 'π/2', 'π', '3π/2', '2π'])
    ax2.legend(loc='upper right', fontsize=9)
    ax2.set_title('Gait Phase (Hilbert transform)', fontsize=12)
    ax2.grid(True, alpha=0.3)
    
    # ===== PANEL 3: Scutellum Z (raw and oscillation) =====
    ax3 = fig.add_subplot(gs[2, :], sharex=ax1)
    ax3_twin = ax3.twinx()
    
    # Raw Scutellum Z
    ax3.plot(time_ms, scut_z_raw, color='#5C4D7D', lw=1.5, alpha=0.6, label='Scutellum Z (raw)')
    ax3.set_ylabel('Scutellum Z (mm)', fontsize=11, color='#5C4D7D')
    ax3.tick_params(axis='y', labelcolor='#5C4D7D')
    
    # Detrended oscillation (scaled for visibility)
    osc_scale = (np.nanmax(scut_z_raw) - np.nanmin(scut_z_raw)) / (2 * np.nanstd(z_oscillation) + 1e-6)
    z_osc_scaled = z_oscillation * min(osc_scale, 5) + np.nanmean(scut_z_raw)
    ax3_twin.plot(time_ms, z_oscillation * 1000, color='#2D8F5C', lw=1.2, alpha=0.8, label='Oscillation (μm)')
    ax3_twin.set_ylabel('Oscillation (μm)', fontsize=11, color='#2D8F5C')
    ax3_twin.tick_params(axis='y', labelcolor='#2D8F5C')
    ax3_twin.axhline(0, color='#2D8F5C', ls=':', alpha=0.5)
    
    ax3.set_title(f'Scutellum Z Vertical Motion (dominant freq: {dominant_freq:.1f} Hz, period: {dominant_period_ms:.1f} ms)', fontsize=12)
    ax3.grid(True, alpha=0.3)
    
    # ===== PANEL 4: Speed =====
    ax4 = fig.add_subplot(gs[3, :], sharex=ax1)
    ax4.plot(time_ms, speed, color='#2D3142', lw=1.2)
    ax4.axhline(mean_speed, color='red', ls='--', lw=1, alpha=0.7, label=f'Mean: {mean_speed:.1f} mm/s')
    ax4.set_ylabel('Speed (mm/s)', fontsize=11)
    ax4.set_xlabel('Time (ms)', fontsize=11)
    ax4.legend(loc='upper right', fontsize=9)
    ax4.set_title('Walking Speed', fontsize=12)
    ax4.grid(True, alpha=0.3)
    
    # ===== PANEL 5: Power Spectrum =====
    ax5 = fig.add_subplot(gs[4, 0])
    if freqs is not None and psd is not None:
        ax5.semilogy(freqs, psd, color='#5C4D7D', lw=1.5)
        if not np.isnan(dominant_freq):
            ax5.axvline(dominant_freq, color='red', ls='--', lw=1.5, alpha=0.7, 
                       label=f'Peak: {dominant_freq:.1f} Hz')
        ax5.set_xlabel('Frequency (Hz)', fontsize=11)
        ax5.set_ylabel('Power Spectral Density', fontsize=11)
        ax5.set_xlim(0, 50)
        ax5.legend(loc='upper right', fontsize=9)
        ax5.set_title('Scutellum Z Power Spectrum', fontsize=12)
        ax5.grid(True, alpha=0.3)
    
    # ===== PANEL 6: Phase-locked average =====
    ax6 = fig.add_subplot(gs[4, 1])
    
    # Bin oscillation by gait phase
    n_bins = 36  # 10-degree bins
    phase_bins = np.linspace(0, 2*np.pi, n_bins + 1)
    phase_centers = (phase_bins[:-1] + phase_bins[1:]) / 2
    
    # Use tripod 1 phase as reference
    osc_by_phase = []
    for i in range(n_bins):
        mask = (tripod1_phase >= phase_bins[i]) & (tripod1_phase < phase_bins[i+1])
        if mask.sum() > 0:
            osc_by_phase.append(np.nanmean(z_oscillation[mask]))
        else:
            osc_by_phase.append(np.nan)
    
    osc_by_phase = np.array(osc_by_phase) * 1000  # Convert to μm
    ax6.plot(np.degrees(phase_centers), osc_by_phase, 'o-', color='#2D8F5C', lw=2, markersize=4)
    ax6.axhline(0, color='gray', ls=':', alpha=0.5)
    ax6.set_xlabel('Gait Phase (degrees)', fontsize=11)
    ax6.set_ylabel('Mean Z oscillation (μm)', fontsize=11)
    ax6.set_xticks([0, 90, 180, 270, 360])
    ax6.set_title('Scutellum Z vs Gait Phase', fontsize=12)
    ax6.grid(True, alpha=0.3)
    
    # Overall title
    gait_freq = fps / gait_cycle_frames if gait_cycle_frames > 0 else np.nan
    fig.suptitle(f'Walking Bout {bout_idx}: Gait Phase Analysis\n'
                 f'Frames {actual_start}-{actual_end} | Duration: {n_frames/fps*1000:.0f} ms | '
                 f'Speed: {mean_speed:.1f} mm/s | Gait freq: {gait_freq:.1f} Hz',
                 fontsize=14, fontweight='bold', y=0.98)
    
    plt.tight_layout(rect=[0, 0, 1, 0.96])
    
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"Saved: {save_path}")
    
    plt.show()
    
    return {
        'dominant_freq_hz': dominant_freq,
        'dominant_period_ms': dominant_period_ms,
        'gait_freq_hz': gait_freq,
        'gait_cycle_ms': gait_cycle_frames / fps * 1000 if not np.isnan(gait_cycle_frames) else np.nan,
        'mean_speed_mm_s': mean_speed,
        'z_oscillation_std_um': np.nanstd(z_oscillation) * 1000
    }


print("Gait phase visualization function defined.")


In [ ]:
# ============================================================================
# RUN GAIT PHASE ANALYSIS ON WALKING BOUTS
# ============================================================================
# Analyze selected bouts and aggregate results across all bouts

# Analyze a few example bouts in detail
N_EXAMPLE_BOUTS = 15  # Number of bouts to visualize in detail

if len(valid_bouts) > 0:
    # Select bouts with good walking characteristics (long duration, high speed)
    sorted_bouts = sorted(valid_bouts, key=lambda b: b['total_distance_mm'], reverse=True)
    example_bouts = sorted_bouts[:N_EXAMPLE_BOUTS]
    
    print(f"\nAnalyzing {N_EXAMPLE_BOUTS} example bouts in detail...")
    print("=" * 70)
    
    for i, bout in enumerate(example_bouts):
        bout['bout_idx'] = i + 1
        print(f"\n--- Bout {i+1}: frames {bout['start'] + frame_offset}-{bout['end'] + frame_offset} ---")
        
        save_path = OUTPUT_DIR / f"gait_phase_bout_{i+1:03d}.pdf"
        metrics = plot_gait_phase_analysis(
            bout, leg_tip_data, scutellum_data,
            frame_offset=frame_offset, fps=FPS, save_path=save_path
        )
        
        print(f"  Dominant Z freq: {metrics['dominant_freq_hz']:.1f} Hz")
        print(f"  Gait freq: {metrics['gait_freq_hz']:.1f} Hz") 
        print(f"  Z oscillation amplitude: {metrics['z_oscillation_std_um']:.1f} μm (std)")
        print(f"  Mean speed: {metrics['mean_speed_mm_s']:.1f} mm/s")
else:
    print("No valid bouts to analyze. Run bout detection first.")


In [ ]:
# ============================================================================
# AGGREGATE ANALYSIS: Z OSCILLATIONS VS WALKING SPEED (ALL BOUTS)
# ============================================================================
# Analyze the relationship between Scutellum Z oscillation characteristics
# and walking speed across ALL valid bouts.

def analyze_all_bouts_oscillations(valid_bouts, leg_tip_data, scutellum_data, 
                                    fps=FPS, output_dir=OUTPUT_DIR):
    """
    Compute oscillation metrics for all bouts and analyze correlations with speed.
    """
    print(f"\nAnalyzing Z oscillations for ALL {len(valid_bouts)} valid bouts...")
    print("=" * 70)
    
    all_metrics = []
    
    for i, bout in enumerate(valid_bouts):
        start, end = bout['start'], bout['end']
        
        # Analyze Scutellum Z periodicity
        freqs, psd, dominant_freq, dominant_period, z_oscillation = \
            analyze_scutellum_z_periodicity(scutellum_data, start, end, fps)
        
        # Get gait cycle info
        _, _, _, gait_cycle_frames = compute_composite_gait_phase(leg_tip_data, start, end, fps)
        gait_freq = fps / gait_cycle_frames if gait_cycle_frames > 0 and not np.isnan(gait_cycle_frames) else np.nan
        
        # Compute speed
        _, mean_speed, max_speed = compute_instant_speed(scutellum_data, start, end, fps)
        
        # Compute oscillation metrics
        z_osc_std = np.nanstd(z_oscillation) * 1000  # μm
        z_osc_range = (np.nanmax(z_oscillation) - np.nanmin(z_oscillation)) * 1000  # μm
        
        all_metrics.append({
            'bout_idx': i + 1,
            'start_frame': start,
            'end_frame': end,
            'n_frames': end - start + 1,
            'duration_ms': (end - start + 1) / fps * 1000,
            'mean_speed_mm_s': mean_speed,
            'max_speed_mm_s': max_speed,
            'dominant_z_freq_hz': dominant_freq,
            'dominant_z_period_ms': dominant_period,
            'gait_freq_hz': gait_freq,
            'z_oscillation_std_um': z_osc_std,
            'z_oscillation_range_um': z_osc_range,
            'total_distance_mm': bout['total_distance_mm'],
        })
        
        if (i + 1) % 20 == 0:
            print(f"  Processed {i+1}/{len(valid_bouts)} bouts...")
    
    # Convert to DataFrame
    metrics_df = pd.DataFrame(all_metrics)
    
    # Remove invalid entries
    valid_mask = (~np.isnan(metrics_df['dominant_z_freq_hz'])) & \
                 (~np.isnan(metrics_df['gait_freq_hz'])) & \
                 (metrics_df['z_oscillation_std_um'] > 0)
    metrics_clean = metrics_df[valid_mask].copy()
    
    print(f"\nValid bouts for correlation analysis: {len(metrics_clean)}/{len(metrics_df)}")
    
    # ===== CREATE FIGURE =====
    fig, axes = plt.subplots(2, 3, figsize=(16, 10))
    
    # Panel 1: Speed vs Z oscillation amplitude
    ax = axes[0, 0]
    ax.scatter(metrics_clean['mean_speed_mm_s'], metrics_clean['z_oscillation_std_um'], 
               alpha=0.6, c='#5C4D7D', s=30)
    
    # Correlation
    r, p = pearsonr(metrics_clean['mean_speed_mm_s'], metrics_clean['z_oscillation_std_um'])
    ax.set_xlabel('Mean Walking Speed (mm/s)', fontsize=11)
    ax.set_ylabel('Z Oscillation Amplitude (μm, std)', fontsize=11)
    ax.set_title(f'Speed vs Z Oscillation\nPearson r = {r:.3f}, p = {p:.2e}', fontsize=12)
    ax.grid(True, alpha=0.3)
    
    # Panel 2: Speed vs dominant Z frequency
    ax = axes[0, 1]
    ax.scatter(metrics_clean['mean_speed_mm_s'], metrics_clean['dominant_z_freq_hz'], 
               alpha=0.6, c='#2D8F5C', s=30)
    r, p = pearsonr(metrics_clean['mean_speed_mm_s'], metrics_clean['dominant_z_freq_hz'])
    ax.set_xlabel('Mean Walking Speed (mm/s)', fontsize=11)
    ax.set_ylabel('Dominant Z Frequency (Hz)', fontsize=11)
    ax.set_title(f'Speed vs Z Oscillation Frequency\nPearson r = {r:.3f}, p = {p:.2e}', fontsize=12)
    ax.grid(True, alpha=0.3)
    
    # Panel 3: Gait freq vs Z oscillation freq
    ax = axes[0, 2]
    ax.scatter(metrics_clean['gait_freq_hz'], metrics_clean['dominant_z_freq_hz'], 
               alpha=0.6, c='#E63946', s=30)
    
    # Add identity line
    max_freq = max(metrics_clean['gait_freq_hz'].max(), metrics_clean['dominant_z_freq_hz'].max())
    ax.plot([0, max_freq], [0, max_freq], 'k--', alpha=0.5, label='1:1 line')
    ax.plot([0, max_freq], [0, 2*max_freq], 'b--', alpha=0.3, label='2:1 line')
    
    r, p = pearsonr(metrics_clean['gait_freq_hz'], metrics_clean['dominant_z_freq_hz'])
    ax.set_xlabel('Gait Frequency (Hz)', fontsize=11)
    ax.set_ylabel('Dominant Z Frequency (Hz)', fontsize=11)
    ax.set_title(f'Gait vs Z Frequency\nPearson r = {r:.3f}, p = {p:.2e}', fontsize=12)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    
    # Panel 4: Histogram of Z oscillation amplitudes
    ax = axes[1, 0]
    ax.hist(metrics_clean['z_oscillation_std_um'], bins=30, color='#5C4D7D', alpha=0.7, edgecolor='white')
    ax.axvline(metrics_clean['z_oscillation_std_um'].median(), color='red', ls='--', lw=2, 
               label=f'Median: {metrics_clean["z_oscillation_std_um"].median():.1f} μm')
    ax.set_xlabel('Z Oscillation Amplitude (μm, std)', fontsize=11)
    ax.set_ylabel('Count', fontsize=11)
    ax.set_title('Distribution of Z Oscillation Amplitudes', fontsize=12)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    
    # Panel 5: Histogram of dominant frequencies
    ax = axes[1, 1]
    ax.hist(metrics_clean['dominant_z_freq_hz'], bins=30, color='#2D8F5C', alpha=0.7, edgecolor='white')
    ax.axvline(metrics_clean['dominant_z_freq_hz'].median(), color='red', ls='--', lw=2,
               label=f'Median: {metrics_clean["dominant_z_freq_hz"].median():.1f} Hz')
    ax.set_xlabel('Dominant Z Frequency (Hz)', fontsize=11)
    ax.set_ylabel('Count', fontsize=11)
    ax.set_title('Distribution of Z Oscillation Frequencies', fontsize=12)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    
    # Panel 6: Ratio of Z freq to gait freq
    ax = axes[1, 2]
    freq_ratio = metrics_clean['dominant_z_freq_hz'] / metrics_clean['gait_freq_hz']
    ax.hist(freq_ratio.dropna(), bins=30, color='#E63946', alpha=0.7, edgecolor='white')
    ax.axvline(1.0, color='blue', ls='--', lw=2, alpha=0.7, label='1:1')
    ax.axvline(2.0, color='green', ls='--', lw=2, alpha=0.7, label='2:1')
    ax.axvline(freq_ratio.median(), color='red', ls='-', lw=2, 
               label=f'Median: {freq_ratio.median():.2f}')
    ax.set_xlabel('Z Freq / Gait Freq Ratio', fontsize=11)
    ax.set_ylabel('Count', fontsize=11)
    ax.set_title('Z Oscillation to Gait Frequency Ratio', fontsize=12)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    
    fig.suptitle(f'Scutellum Z Oscillation Analysis Across {len(metrics_clean)} Walking Bouts', 
                 fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    
    # Save figure
    if output_dir:
        save_path = output_dir / "z_oscillation_speed_correlation.pdf"
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"\nSaved: {save_path}")
    
    plt.show()
    
    # Print summary statistics
    print("\n" + "=" * 70)
    print("SUMMARY STATISTICS")
    print("=" * 70)
    print(f"\nZ Oscillation Amplitude (μm, std):")
    print(f"  Mean: {metrics_clean['z_oscillation_std_um'].mean():.2f}")
    print(f"  Median: {metrics_clean['z_oscillation_std_um'].median():.2f}")
    print(f"  Range: {metrics_clean['z_oscillation_std_um'].min():.2f} - {metrics_clean['z_oscillation_std_um'].max():.2f}")
    
    print(f"\nDominant Z Frequency (Hz):")
    print(f"  Mean: {metrics_clean['dominant_z_freq_hz'].mean():.1f}")
    print(f"  Median: {metrics_clean['dominant_z_freq_hz'].median():.1f}")
    print(f"  Range: {metrics_clean['dominant_z_freq_hz'].min():.1f} - {metrics_clean['dominant_z_freq_hz'].max():.1f}")
    
    print(f"\nZ Freq / Gait Freq Ratio:")
    print(f"  Mean: {freq_ratio.mean():.2f}")
    print(f"  Median: {freq_ratio.median():.2f}")
    print(f"  This suggests Z oscillations occur at ~{freq_ratio.median():.1f}x the gait frequency")
    
    # Save metrics to CSV
    if output_dir:
        csv_path = output_dir / "z_oscillation_metrics.csv"
        metrics_df.to_csv(csv_path, index=False)
        print(f"\nSaved metrics: {csv_path}")
    
    return metrics_df


# Run the aggregate analysis
if len(valid_bouts) > 0:
    oscillation_metrics = analyze_all_bouts_oscillations(
        valid_bouts, leg_tip_data, scutellum_data, fps=FPS, output_dir=OUTPUT_DIR
    )
else:
    print("No valid bouts. Run bout detection first.")


In [ ]:
# ============================================================================
# PHASE-LOCKED AVERAGE: SCUTELLUM Z ACROSS GAIT CYCLE (ALL BOUTS)
# ============================================================================
# Analyze Scutellum Z oscillations aligned to BOTH tripod group phases.
# In a tripod gait the body oscillates at 2x the leg stepping frequency —
# each tripod pushes the body up in turn, producing two peaks per leg cycle.

def compute_phase_locked_average_all_bouts(valid_bouts, leg_tip_data, scutellum_data, 
                                            fps=FPS, n_bins=72, output_dir=OUTPUT_DIR):
    """
    Phase-locked average of Scutellum Z referenced to both tripod groups.
    """
    
    
    print(f"\nComputing phase-locked average across {len(valid_bouts)} bouts...")
    
    # Collect (phase, z_osc, speed) for BOTH tripod references
    t1_phases, t2_phases = [], []
    all_z_osc = []
    all_speeds = []
    
    for bout in valid_bouts:
        start, end = bout['start'], bout['end']
        
        tripod1_phase, tripod2_phase, all_leg_phases, _ = compute_composite_gait_phase(
            leg_tip_data, start, end, fps
        )
        _, _, _, _, z_oscillation = analyze_scutellum_z_periodicity(
            scutellum_data, start, end, fps
        )
        speed, _, _ = compute_instant_speed(scutellum_data, start, end, fps)
        
        valid_mask = (~np.isnan(tripod1_phase) & ~np.isnan(tripod2_phase) 
                      & ~np.isnan(z_oscillation) & ~np.isnan(speed))
        
        t1_phases.extend(tripod1_phase[valid_mask])
        t2_phases.extend(tripod2_phase[valid_mask])
        all_z_osc.extend(z_oscillation[valid_mask] * 1000)  # μm
        all_speeds.extend(speed[valid_mask])
    
    t1_phases = np.array(t1_phases)
    t2_phases = np.array(t2_phases)
    all_z_osc = np.array(all_z_osc)
    all_speeds = np.array(all_speeds)
    
    print(f"  Total data points: {len(t1_phases):,}")
    
    # =====================================================================
    # BIN BY PHASE
    # =====================================================================
    phase_bins = np.linspace(0, 2*np.pi, n_bins + 1)
    phase_centers = (phase_bins[:-1] + phase_bins[1:]) / 2
    speed_median = np.median(all_speeds)
    
    def bin_by_phase(phases, values, speeds):
        """Bin values by phase, return mean/sem and slow/fast split."""
        z_mean = np.full(n_bins, np.nan)
        z_sem_arr = np.full(n_bins, np.nan)
        z_std_arr = np.full(n_bins, np.nan)
        z_slow = np.full(n_bins, np.nan)
        z_fast = np.full(n_bins, np.nan)
        counts = np.zeros(n_bins)
        
        for i in range(n_bins):
            mask = (phases >= phase_bins[i]) & (phases < phase_bins[i+1])
            counts[i] = mask.sum()
            if counts[i] > 1:
                z_mean[i] = np.nanmean(values[mask])
                z_std_arr[i] = np.nanstd(values[mask])
                z_sem_arr[i] = sem(values[mask], nan_policy='omit')
            
            slow_m = mask & (speeds < 30)
            fast_m = mask & (speeds >= 30)
            if slow_m.sum() > 0:
                z_slow[i] = np.nanmean(values[slow_m])
            if fast_m.sum() > 0:
                z_fast[i] = np.nanmean(values[fast_m])
        
        return z_mean, z_sem_arr, z_std_arr, z_slow, z_fast, counts
    
    t1_mean, t1_sem, t1_std, t1_slow, t1_fast, t1_counts = bin_by_phase(
        t1_phases, all_z_osc, all_speeds
    )
    t2_mean, t2_sem, t2_std, t2_slow, t2_fast, t2_counts = bin_by_phase(
        t2_phases, all_z_osc, all_speeds
    )
    
    # =====================================================================
    # FIGURE: 3 rows x 2 cols
    # =====================================================================
    pi_ticks = [0, np.pi/2, np.pi, 3*np.pi/2, 2*np.pi]
    pi_labels = ['0', 'π/2', 'π', '3π/2', '2π']
    
    fig, axes = plt.subplots(3, 2, figsize=(16, 16))
    
    colors = {'t1': '#5C4D7D', 't2': '#D4763A'}
    
    def add_phase_regions(ax, label_tripod='1'):
        """Add shading for swing / stance / transition regions."""
        # Swing phase ~0 to ~π, Stance ~π to ~2π (approximate)
        ax.axvspan(0, np.pi, alpha=0.05, color='red', label='Swing half-cycle')
        ax.axvspan(np.pi, 2*np.pi, alpha=0.05, color='blue', label='Stance half-cycle')
        # Transition zones at 0 and π
        for x in [0, np.pi, 2*np.pi]:
            ax.axvline(x, color='gray', ls=':', alpha=0.4, lw=1)
        ax.axvline(np.pi/2, color='red', ls='--', alpha=0.25, lw=1)
        ax.axvline(3*np.pi/2, color='blue', ls='--', alpha=0.25, lw=1)
    
    # ----- Row 1: Tripod 1 reference & Tripod 2 reference -----
    for col, (z_m, z_s, label, color) in enumerate([
        (t1_mean, t1_sem, 'Tripod 1 (T1L, T2R, T3L)', colors['t1']),
        (t2_mean, t2_sem, 'Tripod 2 (T1R, T2L, T3R)', colors['t2']),
    ]):
        ax = axes[0, col]
        add_phase_regions(ax)
        ax.fill_between(phase_centers, z_m - z_s, z_m + z_s, alpha=0.3, color=color)
        ax.plot(phase_centers, z_m, 'o-', color=color, lw=2, ms=3, label='Mean ± SEM')
        ax.axhline(0, color='gray', ls=':', alpha=0.5)
        ax.set_xlabel('Gait Phase (rad)', fontsize=11)
        ax.set_ylabel('Scutellum ΔZ (μm)', fontsize=11)
        ax.set_xticks(pi_ticks)
        ax.set_xticklabels(pi_labels)
        ax.set_xlim(0, 2*np.pi)
        ax.set_title(f'Referenced to {label}', fontsize=12)
        ax.legend(fontsize=9, loc='upper right')
        ax.grid(True, alpha=0.2)
    
    # ----- Row 2 Left: Overlay both tripod references -----
    ax = axes[1, 0]
    add_phase_regions(ax)
    ax.fill_between(phase_centers, t1_mean - t1_sem, t1_mean + t1_sem, alpha=0.2, color=colors['t1'])
    ax.fill_between(phase_centers, t2_mean - t2_sem, t2_mean + t2_sem, alpha=0.2, color=colors['t2'])
    ax.plot(phase_centers, t1_mean, 'o-', color=colors['t1'], lw=2, ms=3, label='Ref: Tripod 1')
    ax.plot(phase_centers, t2_mean, 's-', color=colors['t2'], lw=2, ms=3, label='Ref: Tripod 2')
    ax.axhline(0, color='gray', ls=':', alpha=0.5)
    ax.set_xlabel('Gait Phase (rad)', fontsize=11)
    ax.set_ylabel('Scutellum ΔZ (μm)', fontsize=11)
    ax.set_xticks(pi_ticks)
    ax.set_xticklabels(pi_labels)
    ax.set_xlim(0, 2*np.pi)
    ax.set_title('Both Tripod References Overlaid', fontsize=12)
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.2)
    
    # ----- Row 2 Right: Speed split (using tripod 1 ref) -----
    ax = axes[1, 1]
    add_phase_regions(ax)
    ax.plot(phase_centers, t1_slow, 'o-', color='#457B9D', lw=2, ms=3,
            label=f'Slow (<{speed_median:.1f} mm/s)')
    ax.plot(phase_centers, t1_fast, 's-', color='#E63946', lw=2, ms=3,
            label=f'Fast (≥{speed_median:.1f} mm/s)')
    ax.axhline(0, color='gray', ls=':', alpha=0.5)
    ax.set_xlabel('Gait Phase — Tripod 1 ref (rad)', fontsize=11)
    ax.set_ylabel('Scutellum ΔZ (μm)', fontsize=11)
    ax.set_xticks(pi_ticks)
    ax.set_xticklabels(pi_labels)
    ax.set_xlim(0, 2*np.pi)
    ax.set_title('Slow vs Fast Walking (Tripod 1 ref)', fontsize=12)
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.2)
    
    # ----- Row 3 Left: Phase relationship between tripod 1 and tripod 2 -----
    ax = axes[2, 0]
    # Compute the phase difference between the two tripods
    phase_diff = (t2_phases - t1_phases) % (2 * np.pi)
    
    ax.hist(phase_diff, bins=72, color='#2D8F5C', alpha=0.7, edgecolor='white', density=True)
    mean_diff = circmean(phase_diff, high=2*np.pi, low=0)
    std_diff = circstd(phase_diff, high=2*np.pi, low=0)
    ax.axvline(mean_diff, color='red', lw=2, ls='--', 
               label=f'Mean = {mean_diff:.2f} rad ({np.degrees(mean_diff):.0f}°)')
    ax.axvline(np.pi, color='gray', lw=1.5, ls=':', alpha=0.6, label='π (perfect antiphase)')
    ax.set_xlabel('Tripod 2 − Tripod 1 Phase (rad)', fontsize=11)
    ax.set_ylabel('Density', fontsize=11)
    ax.set_xticks(pi_ticks)
    ax.set_xticklabels(pi_labels)
    ax.set_xlim(0, 2*np.pi)
    ax.set_title('Inter-Tripod Phase Difference', fontsize=12)
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.2)
    
    # ----- Row 3 Right: Dual-cycle view (2 full tripod 1 cycles) -----
    # This shows the Z pattern across two complete tripod 1 cycles so you
    # can see both peaks in context: one from each tripod's push-off.
    ax = axes[2, 1]
    # Tile the tripod 1 data twice
    pc2 = np.concatenate([phase_centers, phase_centers + 2*np.pi])
    zm2 = np.tile(t1_mean, 2)
    zs2 = np.tile(t1_sem, 2)
    
    ax.fill_between(pc2, zm2 - zs2, zm2 + zs2, alpha=0.2, color=colors['t1'])
    ax.plot(pc2, zm2, '-', color=colors['t1'], lw=2, label='Scutellum ΔZ')
    ax.axhline(0, color='gray', ls=':', alpha=0.5)
    
    # Mark tripod transitions 
    for x in [0, np.pi, 2*np.pi, 3*np.pi, 4*np.pi]:
        ax.axvline(x, color='gray', ls=':', alpha=0.4, lw=1)
    
    # Shade tripod 1 swing vs tripod 2 swing alternating
    for cycle in range(2):
        offset = cycle * 2 * np.pi
        ax.axvspan(offset, offset + np.pi, alpha=0.06, color='red')
        ax.axvspan(offset + np.pi, offset + 2*np.pi, alpha=0.06, color='blue')
        # Labels in first cycle only
        if cycle == 0:
            ax.text(offset + np.pi/2, ax.get_ylim()[0] if ax.get_ylim()[0] != 0 else -1, 
                    'T1 swing', ha='center', va='bottom', fontsize=9, color='red', alpha=0.7)
            ax.text(offset + 3*np.pi/2, ax.get_ylim()[0] if ax.get_ylim()[0] != 0 else -1, 
                    'T2 swing', ha='center', va='bottom', fontsize=9, color='blue', alpha=0.7)
    
    dual_ticks = [0, np.pi, 2*np.pi, 3*np.pi, 4*np.pi]
    dual_labels = ['0', 'π', '2π', '3π', '4π']
    ax.set_xticks(dual_ticks)
    ax.set_xticklabels(dual_labels)
    ax.set_xlim(0, 4*np.pi)
    ax.set_xlabel('Tripod 1 Phase (rad) — two full cycles', fontsize=11)
    ax.set_ylabel('Scutellum ΔZ (μm)', fontsize=11)
    ax.set_title('Two Full Gait Cycles (T1 ref)', fontsize=12)
    ax.legend(fontsize=10, loc='upper right')
    ax.grid(True, alpha=0.2)
    
    fig.suptitle(
        f'Scutellum Z Phase-Locked to Gait Cycle\n'
        f'N = {len(t1_phases):,} samples from {len(valid_bouts)} bouts',
        fontsize=14, fontweight='bold', y=1.01
    )
    plt.tight_layout()
    
    if output_dir:
        save_path = Path(output_dir) / "phase_locked_z_oscillation.pdf"
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"\nSaved: {save_path}")
    
    plt.show()
    
    # =====================================================================
    # QUANTITATIVE SUMMARY
    # =====================================================================
    t1_peak_phase = phase_centers[np.nanargmax(t1_mean)]
    t1_trough_phase = phase_centers[np.nanargmin(t1_mean)]
    t2_peak_phase = phase_centers[np.nanargmax(t2_mean)]
    t2_trough_phase = phase_centers[np.nanargmin(t2_mean)]
    t1_amp = np.nanmax(t1_mean) - np.nanmin(t1_mean)
    t2_amp = np.nanmax(t2_mean) - np.nanmin(t2_mean)
    
    print("\n" + "=" * 70)
    print("PHASE-LOCKED OSCILLATION SUMMARY")
    print("=" * 70)
    
    print(f"\n--- Referenced to Tripod 1 (T1L, T2R, T3L) ---")
    print(f"  Peak ΔZ at phase:   {t1_peak_phase:.2f} rad ({np.degrees(t1_peak_phase):.0f}°)")
    print(f"  Trough ΔZ at phase: {t1_trough_phase:.2f} rad ({np.degrees(t1_trough_phase):.0f}°)")
    print(f"  Amplitude (peak-to-trough): {t1_amp:.2f} μm")
    
    print(f"\n--- Referenced to Tripod 2 (T1R, T2L, T3R) ---")
    print(f"  Peak ΔZ at phase:   {t2_peak_phase:.2f} rad ({np.degrees(t2_peak_phase):.0f}°)")
    print(f"  Trough ΔZ at phase: {t2_trough_phase:.2f} rad ({np.degrees(t2_trough_phase):.0f}°)")
    print(f"  Amplitude (peak-to-trough): {t2_amp:.2f} μm")
    
    print(f"\n--- Inter-tripod phase relationship ---")
    print(f"  Mean T2-T1 phase diff: {mean_diff:.2f} rad ({np.degrees(mean_diff):.0f}°)")
    print(f"  Circular std: {std_diff:.2f} rad ({np.degrees(std_diff):.0f}°)")
    if abs(mean_diff - np.pi) < 0.3:
        print(f"  → Tripods are close to ANTIPHASE (π rad = 180°) as expected for tripod gait")
    
    print(f"\n--- Interpretation ---")
    # Check if trough is near transitions (0 or π)
    for ref_name, trough_ph in [('Tripod 1', t1_trough_phase), ('Tripod 2', t2_trough_phase)]:
        near_0 = min(trough_ph, 2*np.pi - trough_ph) < np.pi/4
        near_pi = abs(trough_ph - np.pi) < np.pi/4
        if near_0 or near_pi:
            trans = '0/2π (swing onset)' if near_0 else 'π (stance onset)'
            print(f"  {ref_name} ref: body is LOWEST near phase {trans}")
            print(f"    → Low point coincides with tripod TRANSITION")
        else:
            print(f"  {ref_name} ref: trough at {trough_ph:.2f} rad — mid-phase, not at a transition")
    
    slow_amp = np.nanmax(t1_slow) - np.nanmin(t1_slow)
    fast_amp = np.nanmax(t1_fast) - np.nanmin(t1_fast)
    print(f"\n--- Speed dependence (Tripod 1 ref) ---")
    print(f"  Slow walking amplitude: {slow_amp:.2f} μm")
    print(f"  Fast walking amplitude: {fast_amp:.2f} μm")
    if fast_amp > slow_amp:
        print("  → Faster walking shows LARGER vertical oscillations")
    else:
        print("  → Faster walking shows SMALLER vertical oscillations")
    
    return {
        'phase_centers': phase_centers,
        't1_mean': t1_mean, 't1_sem': t1_sem,
        't2_mean': t2_mean, 't2_sem': t2_sem,
        't1_slow': t1_slow, 't1_fast': t1_fast,
        't1_peak_phase_rad': t1_peak_phase,
        't1_trough_phase_rad': t1_trough_phase,
        't2_peak_phase_rad': t2_peak_phase,
        't2_trough_phase_rad': t2_trough_phase,
        'inter_tripod_phase_diff_rad': mean_diff,
    }


# Run the phase-locked analysis
if len(valid_bouts) > 0:
    phase_locked_results = compute_phase_locked_average_all_bouts(
        valid_bouts, leg_tip_data, scutellum_data, fps=FPS, output_dir=OUTPUT_DIR
    )
else:
    print("No valid bouts. Run bout detection first.")

In [ ]:
# ============================================================================
# GROOMING DETECTION CONFIGURATION
# ============================================================================
JARVIS_SCALE = 10.0
FPS = 800
SCUTELLUM = "Scutellum"
PDF_DPI = 300

mpl.rcParams.update({
    'pdf.fonttype': 42,
    'pdf.use14corefonts': True,
    'font.family': 'sans-serif',
    'axes.spines.right': False,
    'axes.spines.top': False,
})

# ============================================================================
# Grooming type: "anterior" or "posterior"
GROOM_TYPE = "anterior"

# --- STATIONARITY ---
GROOM_STATIONARY_SPEED = 7.0  # mm/s - scutellum speed must be below this

# --- CONFIDENCE ---
GROOM_CONF_THRESHOLD = 0.5
GROOM_CONF_GAP_BRIDGE = 30  # Bridge short confidence gaps (frames)

# --- PROXIMITY ---
GROOM_LEG_PROXIMITY_MM = 2.0     # mm - max 3D distance between L/R lifted tips
GROOM_TARGET_PROXIMITY_MM = 1.5  # mm - max distance from lifted leg to target centroid

# --- LEG-ON-LEG RUBBING ---
# During leg rubbing, L/R legs are close and share Y position.
# This is an ALTERNATIVE to target proximity (OR logic):
#   valid = (close to target) OR (close to each other AND shared Y)
GROOM_LEG_RUB_Y_TOLERANCE = 0.1  # mm - max |Y_L - Y_R| for shared-Y detection

# --- NON-LIFTED LEGS MUST STAY GROUNDED ---
# Prevents cross-type false positives (e.g., anterior grooming detected as posterior)
GROOM_NON_LIFTED_GROUNDED = True
GROOM_NON_LIFTED_MAX_Z = 0.7  # mm - non-lifted legs must have Z below this

# --- UPRIGHT POSTURE ---
GROOM_UPRIGHT_ENABLED = True  # Grounded leg tips must be below scutellum Z

# --- BOUT PARAMETERS ---
GROOM_MIN_BOUT_FRAMES = 80  # Minimum bout duration (100ms at 800fps)
GROOM_MAX_GAP = 20          # Bridge gaps shorter than this (frames)

# --- DIAGNOSTICS ---
GROOM_SHOW_BOUNDARY_DIAG = True
GROOM_BOUNDARY_DIAG_FRAMES = 50

# ============================================================================
# TYPE-DEPENDENT SETTINGS
# ============================================================================
if GROOM_TYPE == "anterior":
    GROOM_LIFTED_TIPS = ["T1L_TaTip", "T1R_TaTip"]
    GROOM_GROUNDED_TIPS = ["T2L_TaTip", "T2R_TaTip", "T3L_TaTip", "T3R_TaTip"]
    GROOM_TARGET_KPS = ["Antenna_Base", "EyeL", "EyeR"]
    GROOM_CONF_EXCLUDE_PREFIXES = ["T1L", "T1R"]
    GROOM_LIFTED_Z_THRESHOLD = 0.6   # mm - T1 legs lift high for head grooming
    GROOM_TARGET_PROXIMITY_MM = 1.5   # mm - head region is compact
else:  # posterior
    GROOM_LIFTED_TIPS = ["T3L_TaTip", "T3R_TaTip"]
    GROOM_GROUNDED_TIPS = ["T1L_TaTip", "T1R_TaTip", "T2L_TaTip", "T2R_TaTip"]
    GROOM_TARGET_KPS = ["Abd_tip", "WingL_V12", "WingR_V12"]
    GROOM_CONF_EXCLUDE_PREFIXES = ["T3L", "T3R"]
    GROOM_LIFTED_Z_THRESHOLD = 0.3   # mm - T3 legs barely lift
    GROOM_TARGET_PROXIMITY_MM = 2.0   # mm - posterior body region is larger

ALL_LEG_TIPS_GROOM = ["T1L_TaTip", "T1R_TaTip", "T2L_TaTip", "T2R_TaTip",
                       "T3L_TaTip", "T3R_TaTip"]

print(f"Grooming detection configured: {GROOM_TYPE}")
print(f"  Lifted legs: {GROOM_LIFTED_TIPS}")
print(f"  Grounded legs: {GROOM_GROUNDED_TIPS}")
print(f"  Target keypoints: {GROOM_TARGET_KPS}")
print(f"  Confidence exclude: {GROOM_CONF_EXCLUDE_PREFIXES}")
print(f"  Stationary speed: <{GROOM_STATIONARY_SPEED} mm/s")
print(f"  Lifted Z: >{GROOM_LIFTED_Z_THRESHOLD} mm")
print(f"  Non-lifted max Z: <{GROOM_NON_LIFTED_MAX_Z} mm (grounded={GROOM_NON_LIFTED_GROUNDED})")
print(f"  Leg proximity: <{GROOM_LEG_PROXIMITY_MM} mm")
print(f"  Target proximity: <{GROOM_TARGET_PROXIMITY_MM} mm")
print(f"  Leg rub Y tolerance: <{GROOM_LEG_RUB_Y_TOLERANCE} mm")
print(f"  Grooming activity: target_prox OR (leg_prox AND shared_Y)")

In [ ]:
# ============================================================================
# SHARED UTILITY FUNCTIONS
# (Defined here so grooming detection can run independently from walking)
# ============================================================================

def get_all_keypoint_names(df):
    """Get all unique keypoint names from dataframe columns."""
    cols = df.columns.tolist()
    kp_names = []
    seen = set()
    for col in cols:
        base_name = col.split('.')[0] if '.' in col else col
        if base_name not in seen:
            seen.add(base_name)
            kp_names.append(base_name)
    return kp_names


def extract_xyzc(df, kp_name, scale=JARVIS_SCALE):
    """Extract x, y, z coordinates and confidence for a keypoint."""
    cols = df.columns.tolist()
    start_idx = cols.index(kp_name)
    x = df.iloc[:, start_idx].values.astype(float) / scale
    y = df.iloc[:, start_idx + 1].values.astype(float) / scale
    z = df.iloc[:, start_idx + 2].values.astype(float) / scale
    conf = df.iloc[:, start_idx + 3].values.astype(float)
    return x, y, z, conf


def bridge_short_gaps(mask, max_gap):
    """Bridge short gaps (runs of False) in a boolean mask."""
    if max_gap <= 0:
        return mask
    bridged = mask.copy()
    in_gap = False
    gap_start = 0
    for i in range(len(bridged)):
        if not bridged[i]:
            if not in_gap:
                gap_start = i
                in_gap = True
        else:
            if in_gap:
                gap_len = i - gap_start
                if gap_len <= max_gap and gap_start > 0:
                    bridged[gap_start:i] = True
                in_gap = False
    return bridged


def compute_instant_speed(scutellum_data, start, end, fps=FPS):
    """Compute instantaneous speed at each frame (mm/s)."""
    x = scutellum_data['x'][start:end+1]
    y = scutellum_data['y'][start:end+1]
    dx = np.diff(x)
    dy = np.diff(y)
    displacement = np.sqrt(dx**2 + dy**2)
    speed = displacement * fps
    speed = np.concatenate([[speed[0] if len(speed) > 0 else 0], speed])
    mean_speed = np.nanmean(speed)
    max_speed = np.nanmax(speed)
    return speed, mean_speed, max_speed


def compute_total_distance(scutellum_data, start, end):
    """Compute total path distance and net displacement (mm)."""
    x = scutellum_data['x'][start:end+1]
    y = scutellum_data['y'][start:end+1]
    valid = ~np.isnan(x) & ~np.isnan(y)
    if valid.sum() < 2:
        return 0.0, 0.0
    x_valid = x[valid]
    y_valid = y[valid]
    dx = np.diff(x_valid)
    dy = np.diff(y_valid)
    total_distance = np.sum(np.sqrt(dx**2 + dy**2))
    net_displacement = np.sqrt((x_valid[-1] - x_valid[0])**2 + (y_valid[-1] - y_valid[0])**2)
    return total_distance, net_displacement


def find_contiguous_bouts_with_gap_bridging(valid_mask, min_frames, max_gap):
    """Find contiguous bouts, bridging small gaps."""
    bridged = valid_mask.copy()
    in_gap = False
    gap_start = 0
    for i in range(len(bridged)):
        if not bridged[i]:
            if not in_gap:
                gap_start = i
                in_gap = True
        else:
            if in_gap:
                gap_len = i - gap_start
                if gap_len <= max_gap and gap_start > 0 and bridged[gap_start - 1]:
                    bridged[gap_start:i] = True
                in_gap = False
    bouts = []
    in_bout = False
    start = 0
    for i, valid in enumerate(bridged):
        if valid and not in_bout:
            start = i
            in_bout = True
        elif not valid and in_bout:
            if i - start >= min_frames:
                bouts.append((start, i - 1))
            in_bout = False
    if in_bout and len(bridged) - start >= min_frames:
        bouts.append((start, len(bridged) - 1))
    return bouts


def _add_bout_boundaries(ax, actual_start, actual_end, add_legend=False):
    """Add bout boundary shading and vertical lines to an axis."""
    ax.axvspan(actual_start, actual_end, color='#2ca02c', alpha=0.08, zorder=0,
               label='Detected bout' if add_legend else None)
    ax.axvline(x=actual_start, color='#d62728', ls='--', lw=1.5, alpha=0.8, zorder=3,
               label='Bout start/end' if add_legend else None)
    ax.axvline(x=actual_end, color='#d62728', ls='--', lw=1.5, alpha=0.8, zorder=3)


# ============================================================================
# GROOMING DETECTION FUNCTIONS
# ============================================================================

GROOM_FILTER_NAMES = ['confidence', 'stationary', 'upright', 'lifted',
                      'non_lifted_grounded', 'grooming_activity']


def apply_grooming_filter(df, scale=JARVIS_SCALE, fps=FPS):
    """
    Apply frame-level filters for grooming bout detection.

    Filters:
    1. Confidence >threshold for all keypoints EXCEPT lifted leg keypoints
    2. Stationarity: scutellum speed < threshold
    3. Upright: grounded leg tips below scutellum Z
    4. Lifted legs: lifted leg tip Z above threshold
    5. Non-lifted grounded: non-lifted legs must stay below max Z
    6. Grooming activity (OR logic):
       (A) Target proximity: lifted legs close to target centroid
       (B) Leg rubbing: lifted legs close to each other AND sharing Y

    Returns:
        valid_mask, leg_data, scut_data, extra_kp_data, filter_masks, diagnostics
    """
    n_frames = len(df)

    # --- Extract leg tip data (all 6 legs) ---
    leg_data = {}
    for tip in ALL_LEG_TIPS_GROOM:
        x, y, z, conf = extract_xyzc(df, tip, scale)
        leg_data[tip] = {'x': x, 'y': y, 'z': z, 'conf': conf}

    # --- Extract scutellum ---
    scut_x, scut_y, scut_z, scut_conf = extract_xyzc(df, SCUTELLUM, scale)
    scut_data = {'x': scut_x, 'y': scut_y, 'z': scut_z, 'conf': scut_conf}

    # --- Extract target keypoints (antenna/eyes or abdomen/wings) ---
    extra_kp_data = {}
    for kp in GROOM_TARGET_KPS:
        x, y, z, conf = extract_xyzc(df, kp, scale)
        extra_kp_data[kp] = {'x': x, 'y': y, 'z': z, 'conf': conf}

    # --- Extract ALL keypoint confidences ---
    all_kp_names = get_all_keypoint_names(df)
    all_kp_conf = {}
    for kp in all_kp_names:
        try:
            _, _, _, conf = extract_xyzc(df, kp, scale)
            all_kp_conf[kp] = conf
        except (ValueError, IndexError):
            pass

    # =========================================================================
    # FILTER 1: Confidence (exclude lifted leg keypoints)
    # =========================================================================
    conf_mask = np.ones(n_frames, dtype=bool)
    n_checked = 0
    for kp_name, conf_arr in all_kp_conf.items():
        if any(kp_name.startswith(prefix) for prefix in GROOM_CONF_EXCLUDE_PREFIXES):
            continue
        conf_mask &= (conf_arr >= GROOM_CONF_THRESHOLD)
        n_checked += 1

    if GROOM_CONF_GAP_BRIDGE > 0:
        conf_mask = bridge_short_gaps(conf_mask, GROOM_CONF_GAP_BRIDGE)

    # =========================================================================
    # FILTER 2: Stationarity (scutellum speed < threshold)
    # =========================================================================
    speed_all, _, _ = compute_instant_speed(scut_data, 0, n_frames - 1, fps)
    stationary_mask = speed_all < GROOM_STATIONARY_SPEED

    # =========================================================================
    # FILTER 3: Upright posture (grounded legs below scutellum)
    # =========================================================================
    upright_mask = np.ones(n_frames, dtype=bool)
    if GROOM_UPRIGHT_ENABLED:
        for tip in GROOM_GROUNDED_TIPS:
            upright_mask &= (scut_z > leg_data[tip]['z'])

    # =========================================================================
    # FILTER 4: Lifted legs above ground
    # =========================================================================
    lifted_mask = np.ones(n_frames, dtype=bool)
    for tip in GROOM_LIFTED_TIPS:
        lifted_mask &= (leg_data[tip]['z'] > GROOM_LIFTED_Z_THRESHOLD)

    # =========================================================================
    # FILTER 5: Non-lifted legs must stay grounded
    # Prevents cross-type false positives (e.g., front legs lifted in posterior mode)
    # =========================================================================
    non_lifted_grounded_mask = np.ones(n_frames, dtype=bool)
    if GROOM_NON_LIFTED_GROUNDED:
        for tip in GROOM_GROUNDED_TIPS:
            non_lifted_grounded_mask &= (leg_data[tip]['z'] < GROOM_NON_LIFTED_MAX_Z)

    # =========================================================================
    # FILTER 6: Grooming activity (OR logic)
    # Either: (A) legs close to target (head/abdomen grooming)
    #     OR: (B) legs close to each other with shared Y (leg-on-leg rubbing)
    # =========================================================================

    # --- Sub-filter A: Target proximity ---
    target_x = np.mean([extra_kp_data[kp]['x'] for kp in GROOM_TARGET_KPS], axis=0)
    target_y = np.mean([extra_kp_data[kp]['y'] for kp in GROOM_TARGET_KPS], axis=0)
    target_z = np.mean([extra_kp_data[kp]['z'] for kp in GROOM_TARGET_KPS], axis=0)

    target_proximity_mask = np.ones(n_frames, dtype=bool)
    for tip in GROOM_LIFTED_TIPS:
        dist_to_target = np.sqrt(
            (leg_data[tip]['x'] - target_x)**2 +
            (leg_data[tip]['y'] - target_y)**2 +
            (leg_data[tip]['z'] - target_z)**2
        )
        target_proximity_mask &= (dist_to_target < GROOM_TARGET_PROXIMITY_MM)

    # --- Sub-filter B: Leg-on-leg rubbing (close + shared Y) ---
    tip_L, tip_R = GROOM_LIFTED_TIPS[0], GROOM_LIFTED_TIPS[1]
    inter_leg_dist = np.sqrt(
        (leg_data[tip_L]['x'] - leg_data[tip_R]['x'])**2 +
        (leg_data[tip_L]['y'] - leg_data[tip_R]['y'])**2 +
        (leg_data[tip_L]['z'] - leg_data[tip_R]['z'])**2
    )
    leg_proximity_mask = inter_leg_dist < GROOM_LEG_PROXIMITY_MM
    shared_y_mask = np.abs(leg_data[tip_L]['y'] - leg_data[tip_R]['y']) < GROOM_LEG_RUB_Y_TOLERANCE
    leg_rub_mask = leg_proximity_mask & shared_y_mask

    # --- Combined: target proximity OR leg rubbing ---
    grooming_activity_mask = target_proximity_mask | leg_rub_mask

    # =========================================================================
    # COMBINE ALL FILTERS
    # =========================================================================
    valid_mask = (
        conf_mask &
        stationary_mask &
        upright_mask &
        lifted_mask &
        non_lifted_grounded_mask &
        grooming_activity_mask
    )

    filter_masks = {
        'confidence': conf_mask,
        'stationary': stationary_mask,
        'upright': upright_mask,
        'lifted': lifted_mask,
        'non_lifted_grounded': non_lifted_grounded_mask,
        'grooming_activity': grooming_activity_mask,
    }

    diagnostics = {
        'total_frames': n_frames,
        'confidence_pass': int(conf_mask.sum()),
        'confidence_pass_pct': 100 * conf_mask.sum() / n_frames,
        'confidence_kps_checked': n_checked,
        'stationary_pass': int(stationary_mask.sum()),
        'stationary_pass_pct': 100 * stationary_mask.sum() / n_frames,
        'upright_pass': int(upright_mask.sum()),
        'upright_pass_pct': 100 * upright_mask.sum() / n_frames,
        'lifted_pass': int(lifted_mask.sum()),
        'lifted_pass_pct': 100 * lifted_mask.sum() / n_frames,
        'non_lifted_grounded_pass': int(non_lifted_grounded_mask.sum()),
        'non_lifted_grounded_pass_pct': 100 * non_lifted_grounded_mask.sum() / n_frames,
        'target_proximity_pass': int(target_proximity_mask.sum()),
        'target_proximity_pass_pct': 100 * target_proximity_mask.sum() / n_frames,
        'leg_rub_pass': int(leg_rub_mask.sum()),
        'leg_rub_pass_pct': 100 * leg_rub_mask.sum() / n_frames,
        'grooming_activity_pass': int(grooming_activity_mask.sum()),
        'grooming_activity_pass_pct': 100 * grooming_activity_mask.sum() / n_frames,
        'combined_pass': int(valid_mask.sum()),
        'combined_pass_pct': 100 * valid_mask.sum() / n_frames,
    }

    return valid_mask, leg_data, scut_data, extra_kp_data, filter_masks, diagnostics


def diagnose_grooming_boundaries(bout_start, bout_end, filter_masks,
                                  n_frames_context=GROOM_BOUNDARY_DIAG_FRAMES):
    """Analyze frames around grooming bout boundaries to show which filters fail."""
    total_frames = len(filter_masks['confidence'])

    pre_start = max(0, bout_start - n_frames_context)
    pre_range = slice(pre_start, bout_start)
    post_end = min(total_frames, bout_end + n_frames_context + 1)
    post_range = slice(bout_end + 1, post_end)

    pre_failures = {'n_frames_analyzed': bout_start - pre_start}
    post_failures = {'n_frames_analyzed': post_end - bout_end - 1}

    for name in GROOM_FILTER_NAMES:
        mask = filter_masks[name]
        pre_failures[name] = int((~mask[pre_range]).sum())
        post_failures[name] = int((~mask[post_range]).sum())

    pre_items = [(k, v) for k, v in pre_failures.items() if k != 'n_frames_analyzed']
    post_items = [(k, v) for k, v in post_failures.items() if k != 'n_frames_analyzed']
    pre_max = max(pre_items, key=lambda x: x[1]) if pre_items else ('none', 0)
    post_max = max(post_items, key=lambda x: x[1]) if post_items else ('none', 0)

    return {
        'pre_bout_failures': pre_failures,
        'post_bout_failures': post_failures,
        'primary_cause_before': pre_max[0] if pre_max[1] > 0 else None,
        'primary_cause_after': post_max[0] if post_max[1] > 0 else None,
    }


def detect_grooming_bouts(df, scale=JARVIS_SCALE, fps=FPS, verbose=True):
    """
    Complete grooming bout detection pipeline.

    Returns:
        grooming_bouts, leg_data, scut_data, extra_kp_data, filter_masks, diagnostics
    """
    if verbose:
        print("=" * 70)
        print(f"{GROOM_TYPE.upper()} GROOMING BOUT DETECTION")
        print("=" * 70)
        print(f"  Lifted legs: {GROOM_LIFTED_TIPS}")
        print(f"  Target keypoints: {GROOM_TARGET_KPS}")
        print(f"  Stationary speed: <{GROOM_STATIONARY_SPEED} mm/s")
        print(f"  Confidence: >{GROOM_CONF_THRESHOLD} (excluding {GROOM_CONF_EXCLUDE_PREFIXES})")
        print(f"  Lifted Z threshold: >{GROOM_LIFTED_Z_THRESHOLD} mm")
        print(f"  Non-lifted max Z: <{GROOM_NON_LIFTED_MAX_Z} mm")
        print(f"  Leg proximity: <{GROOM_LEG_PROXIMITY_MM} mm")
        print(f"  Target proximity: <{GROOM_TARGET_PROXIMITY_MM} mm")
        print(f"  Leg rub Y tolerance: <{GROOM_LEG_RUB_Y_TOLERANCE} mm")
        print(f"  Activity logic: target_prox OR (leg_prox AND shared_Y)")
        print(f"  Min bout frames: {GROOM_MIN_BOUT_FRAMES}")

    # Step 1: Frame-level filtering
    valid_mask, leg_data, scut_data, extra_kp_data, filter_masks, diagnostics = \
        apply_grooming_filter(df, scale, fps)

    if verbose:
        print(f"\n--- Frame-level filtering ---")
        print(f"  Total frames: {diagnostics['total_frames']}")
        print(f"  Confidence pass: {diagnostics['confidence_pass_pct']:.1f}% ({diagnostics['confidence_kps_checked']} kps checked)")
        print(f"  Stationary pass: {diagnostics['stationary_pass_pct']:.1f}%")
        print(f"  Upright pass: {diagnostics['upright_pass_pct']:.1f}%")
        print(f"  Lifted legs pass: {diagnostics['lifted_pass_pct']:.1f}%")
        print(f"  Non-lifted grounded pass: {diagnostics['non_lifted_grounded_pass_pct']:.1f}%")
        print(f"  Target proximity pass: {diagnostics['target_proximity_pass_pct']:.1f}%")
        print(f"  Leg rubbing pass: {diagnostics['leg_rub_pass_pct']:.1f}%")
        print(f"  Grooming activity pass: {diagnostics['grooming_activity_pass_pct']:.1f}% (target OR leg_rub)")
        print(f"  Combined valid: {diagnostics['combined_pass_pct']:.1f}%")

    # Step 2: Find contiguous bouts with gap bridging
    bouts_raw = find_contiguous_bouts_with_gap_bridging(
        valid_mask, GROOM_MIN_BOUT_FRAMES, GROOM_MAX_GAP
    )

    if verbose:
        print(f"\n--- Bout detection ---")
        print(f"  Found: {len(bouts_raw)} grooming bouts (min {GROOM_MIN_BOUT_FRAMES} frames)")

    # Step 3: Build bout info dicts + boundary diagnostics
    grooming_bouts = []
    for i, (start, end) in enumerate(bouts_raw):
        n = end - start + 1
        _, mean_speed, max_speed = compute_instant_speed(scut_data, start, end, fps)
        total_dist, net_disp = compute_total_distance(scut_data, start, end)

        bout_info = {
            'start': start,
            'end': end,
            'n_frames': n,
            'duration_s': n / fps,
            'mean_speed_mm_s': mean_speed,
            'max_speed_mm_s': max_speed,
            'total_distance_mm': total_dist,
            'bout_idx': i + 1,
        }

        if GROOM_SHOW_BOUNDARY_DIAG:
            bout_info['boundary_diagnostics'] = diagnose_grooming_boundaries(
                start, end, filter_masks, GROOM_BOUNDARY_DIAG_FRAMES
            )

        grooming_bouts.append(bout_info)

    # Print boundary diagnostics
    if verbose and GROOM_SHOW_BOUNDARY_DIAG and grooming_bouts:
        print(f"\n--- Boundary diagnostics (first 5 bouts) ---")
        for bout in grooming_bouts[:5]:
            bd = bout['boundary_diagnostics']
            pre = bd['pre_bout_failures']
            post = bd['post_bout_failures']
            print(f"\n  Bout {bout['bout_idx']} boundary analysis:")
            print(f"    Before bout ({pre['n_frames_analyzed']} frames):")
            for name in GROOM_FILTER_NAMES:
                if pre[name] > 0:
                    marker = " <-- PRIMARY" if bd['primary_cause_before'] == name else ""
                    print(f"      {name}: {pre[name]} failures{marker}")
            if post['n_frames_analyzed'] > 0:
                print(f"    After bout ({post['n_frames_analyzed']} frames):")
                for name in GROOM_FILTER_NAMES:
                    if post[name] > 0:
                        marker = " <-- PRIMARY" if bd['primary_cause_after'] == name else ""
                        print(f"      {name}: {post[name]} failures{marker}")

    if verbose and grooming_bouts:
        print(f"\n--- Grooming bout summary ---")
        for bout in grooming_bouts[:10]:
            print(f"  Bout {bout['bout_idx']}: frames {bout['start']}-{bout['end']} "
                  f"({bout['n_frames']} frames, {bout['duration_s']:.3f}s, "
                  f"speed={bout['mean_speed_mm_s']:.1f}mm/s)")
        if len(grooming_bouts) > 10:
            print(f"  ... and {len(grooming_bouts) - 10} more")

    diagnostics['n_grooming_bouts'] = len(grooming_bouts)

    return grooming_bouts, leg_data, scut_data, extra_kp_data, filter_masks, diagnostics


print("Grooming detection functions defined (with shared utilities).")

In [ ]:
# ============================================================================
# GROOMING BOUT VISUALIZATION
# ============================================================================

def plot_grooming_bout_figure(
    bout_info, leg_data, scut_data, extra_kp_data, filter_masks,
    frame_offset=0, fps=FPS, save_path=None, dpi=PDF_DPI,
    context_frames=200
):
    """
    Create grooming bout figure: leg Z trajectories + scutellum speed.
    Boundary diagnostics annotated at bout edges.
    """
    start = bout_info['start']
    end = bout_info['end']
    bout_idx = bout_info.get('bout_idx', '?')

    data_len = len(scut_data['x'])
    ctx_start = max(0, start - context_frames)
    ctx_end = min(data_len - 1, end + context_frames)

    actual_start = start + frame_offset
    actual_end = end + frame_offset
    frames = np.arange(ctx_start + frame_offset, ctx_end + frame_offset + 1)

    # Speed for extended range
    speed, _, _ = compute_instant_speed(scut_data, ctx_start, ctx_end, fps)
    _, mean_speed, max_speed = compute_instant_speed(scut_data, start, end, fps)

    # Boundary reasons from diagnostics
    bd = bout_info.get('boundary_diagnostics', None)
    start_reason = bd.get('primary_cause_before') if bd else None
    end_reason = bd.get('primary_cause_after') if bd else None

    fig, (ax1, ax2) = plt.subplots(
        2, 1, figsize=(14, 8), sharex=True,
        gridspec_kw={'height_ratios': [1.2, 0.8], 'hspace': 0.15}
    )

    # Color scheme for all 6 legs
    all_leg_colors = {
        'T1L_TaTip': '#E63946', 'T1R_TaTip': '#457B9D',
        'T2L_TaTip': '#F4A261', 'T2R_TaTip': '#2A9D8F',
        'T3L_TaTip': '#9B2226', 'T3R_TaTip': '#1D3557',
    }

    # ===== PANEL 1: Leg Tip Z Trajectories =====
    _add_bout_boundaries(ax1, actual_start, actual_end, add_legend=True)
    for tip in ALL_LEG_TIPS_GROOM:
        z = leg_data[tip]['z'][ctx_start:ctx_end+1]
        label = tip.replace('_TaTip', '')
        is_lifted = tip in GROOM_LIFTED_TIPS
        color = all_leg_colors.get(tip, 'gray')
        lw = 2.0 if is_lifted else 0.8
        alpha = 1.0 if is_lifted else 0.4
        ls = '-' if is_lifted else '--'
        ax1.plot(frames, z, label=f'{label} (lifted)' if is_lifted else label,
                 color=color, lw=lw, alpha=alpha, ls=ls)

    ax1.axhline(y=GROOM_LIFTED_Z_THRESHOLD, color='gray', ls=':', lw=1.5, alpha=0.7,
                label=f'Lifted threshold ({GROOM_LIFTED_Z_THRESHOLD}mm)')
    ax1.set_ylabel('Leg Tip Z (mm)', fontsize=12)
    ax1.set_title(f'{GROOM_TYPE.capitalize()} Grooming | Bout {bout_idx} | Leg Z Trajectories',
                  fontsize=13)
    ax1.legend(loc='upper left', bbox_to_anchor=(1.02, 1), fontsize=9, ncol=1)
    ax1.grid(True, alpha=0.3)

    if start_reason:
        ax1.annotate(
            start_reason, xy=(actual_start, 1), xycoords=('data', 'axes fraction'),
            xytext=(6, -8), textcoords='offset points', fontsize=8, color='#d62728',
            fontweight='bold', ha='left', va='top',
            bbox=dict(boxstyle='round,pad=0.3', fc='white', ec='#d62728', alpha=0.85))
    if end_reason:
        ax1.annotate(
            end_reason, xy=(actual_end, 1), xycoords=('data', 'axes fraction'),
            xytext=(-6, -8), textcoords='offset points', fontsize=8, color='#d62728',
            fontweight='bold', ha='right', va='top',
            bbox=dict(boxstyle='round,pad=0.3', fc='white', ec='#d62728', alpha=0.85))

    # ===== PANEL 2: Scutellum Speed =====
    _add_bout_boundaries(ax2, actual_start, actual_end)
    ax2.plot(frames, speed, color='#264653', lw=1.2)
    ax2.fill_between(frames, 0, speed, alpha=0.3, color='#264653')
    ax2.axhline(y=GROOM_STATIONARY_SPEED, color='#E76F51', ls='--', lw=1.5,
                label=f'Stationary threshold ({GROOM_STATIONARY_SPEED} mm/s)')
    ax2.axhline(y=mean_speed, color='#2A9D8F', ls=':', lw=1.5,
                label=f'Bout mean: {mean_speed:.1f} mm/s')
    ax2.set_ylabel('Speed (mm/s)', fontsize=12)
    ax2.set_xlabel('Frame', fontsize=11)
    ax2.set_title('Scutellum Speed', fontsize=13)
    ax2.legend(loc='upper right', fontsize=10)
    ax2.grid(True, alpha=0.3)
    ax2.set_ylim(bottom=0)

    duration_s = bout_info['n_frames'] / fps
    boundary_str = ""
    if start_reason or end_reason:
        parts = []
        if start_reason:
            parts.append(f"start: {start_reason}")
        if end_reason:
            parts.append(f"end: {end_reason}")
        boundary_str = f"\nBoundary causes: {' | '.join(parts)}"

    fig.suptitle(
        f'{GROOM_TYPE.capitalize()} Grooming Bout {bout_idx} | '
        f'Frames {actual_start}-{actual_end} | {duration_s:.3f}s'
        f'{boundary_str}',
        fontsize=13, fontweight='bold', y=1.0
    )
    plt.tight_layout()

    if save_path:
        fig.savefig(save_path, format='pdf', dpi=dpi, bbox_inches='tight')
        print(f"  Saved: {save_path}")

    return fig


print("Grooming visualization function defined.")

In [ ]:
# ============================================================================
# RUN GROOMING BOUT DETECTION
# ============================================================================

# Optional: specify frame range (set to None for full video)
GROOM_FRAME_START = None  # e.g., 0
GROOM_FRAME_END = None    # e.g., 100000

# --- Prepare data (standalone-compatible) ---
if GROOM_FRAME_START is not None or GROOM_FRAME_END is not None:
    _s = GROOM_FRAME_START if GROOM_FRAME_START is not None else 0
    _e = GROOM_FRAME_END if GROOM_FRAME_END is not None else len(df_full)
    groom_df = df_full.iloc[_s:_e].reset_index(drop=True)
    groom_frame_offset = _s
    print(f"Using frame range: {_s} to {_e} ({len(groom_df)} frames)")
elif 'df' in dir():
    groom_df = df
    groom_frame_offset = frame_offset if 'frame_offset' in dir() else 0
    print(f"Using existing data slice: {len(groom_df)} frames (offset={groom_frame_offset})")
else:
    groom_df = df_full
    groom_frame_offset = 0
    print(f"Using full data: {len(groom_df)} frames")

# --- Run detection ---
grooming_bouts, groom_leg_data, groom_scut_data, groom_extra_kp_data, \
    groom_filter_masks, groom_diagnostics = detect_grooming_bouts(groom_df, verbose=True)

print(f"\n{'='*70}")
print(f"DETECTION COMPLETE: {len(grooming_bouts)} {GROOM_TYPE} grooming bouts found")
print(f"{'='*70}")

# --- Save CSV summary ---
GROOM_OUTPUT_DIR = OUTPUT_DIR / f"grooming_{GROOM_TYPE}"
GROOM_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if grooming_bouts:
    groom_summary_data = []
    for bout in grooming_bouts:
        groom_summary_data.append({
            'fly_id': fly_id,
            'grooming_type': GROOM_TYPE,
            'bout_idx': bout['bout_idx'],
            'start_frame': bout['start'] + groom_frame_offset,
            'end_frame': bout['end'] + groom_frame_offset,
            'n_frames': bout['n_frames'],
            'duration_s': bout['duration_s'],
            'mean_speed_mm_s': bout['mean_speed_mm_s'],
            'max_speed_mm_s': bout['max_speed_mm_s'],
            'total_distance_mm': bout['total_distance_mm'],
        })
    groom_summary_df = pd.DataFrame(groom_summary_data)
    groom_summary_path = OUTPUT_DIR / f"grooming_{GROOM_TYPE}_bouts_summary.csv"
    groom_summary_df.to_csv(groom_summary_path, index=False)
    print(f"\nSummary saved: {groom_summary_path}")
    display(groom_summary_df)

# --- Generate and save ALL bout figures ---
if grooming_bouts:
    sorted_groom_bouts = sorted(grooming_bouts, key=lambda b: b['n_frames'], reverse=True)
    print(f"\nGenerating figures for {len(sorted_groom_bouts)} grooming bouts...")
    for bout in sorted_groom_bouts:
        save_path = GROOM_OUTPUT_DIR / f"grooming_{GROOM_TYPE}_bout_{bout['bout_idx']:03d}.pdf"
        fig = plot_grooming_bout_figure(
            bout, groom_leg_data, groom_scut_data, groom_extra_kp_data,
            groom_filter_masks, frame_offset=groom_frame_offset, fps=FPS,
            save_path=save_path, context_frames=200
        )
        plt.show()
else:
    print("No grooming bouts found. Try adjusting thresholds (proximity, lifted Z, speed).")

# Courtship Bout Detection

Detect **pulse** and **sine** courtship song bouts based on wing Z oscillations.

- **Pulse song**: Wing edge points (V12, V13) show rapid, high-amplitude Z oscillations. Fly is moving forward (speed > 5 mm/s). Upright posture. Confidence > 75% for all keypoints.
- **Sine song**: Wing points show lower-amplitude sinusoidal Z oscillations. Fly is stationary. Must follow a pulse bout within a configurable time window.

Detection uses windowed mean |dZ/dt| on wing tips as the oscillation metric. At least one wing (both V12 + V13 of that wing) must exceed the threshold. Pipeline runs pulse detection first, then sine detection using pulse bout locations as a temporal constraint.

In [ ]:
# ============================================================================
# COURTSHIP DETECTION CONFIGURATION
# ============================================================================

# --- SHARED CONSTANTS (allows standalone execution after Paths & Loading) ---
JARVIS_SCALE = 10.0
FPS = 800
SCUTELLUM = "Scutellum"
PDF_DPI = 300

mpl.rcParams.update({
    'pdf.fonttype': 42,
    'pdf.use14corefonts': True,
    'font.family': 'sans-serif',
    'axes.spines.right': False,
    'axes.spines.top': False,
})

# ============================================================================
# DETECTION — captures all courtship (pulse + sine) as unified bouts
# ============================================================================
COURT_CONF_THRESHOLD = 0.70
COURT_CONF_GAP_BRIDGE = 5
COURT_CONF_WINGS_ONLY = True       # True = confidence filter only checks wing keypoints
COURT_UPRIGHT_ENABLED = True

# Wing oscillation: at least one wing pair (V12+V13) above this threshold
COURT_WING_TIPS = ["WingL_V12", "WingL_V13", "WingR_V12", "WingR_V13"]
COURT_WING_ACTIVITY_WINDOW = 100   # frames (~31ms at 800fps)
COURT_WING_THRESHOLD = 15.0        # mm/s mean |dZ/dt| — low to catch both pulse and sine

# No speed filter for detection (pulse = moving, sine = slow → speed not discriminative)

# ============================================================================
# CLASSIFICATION — label frames within each bout as pulse or sine
# ============================================================================
COURT_PULSE_CLASSIFY_SPEED = 8.0   # mm/s — above = pulse, below = sine
                                    # (pulse median ~22, sine median ~4.7)
COURT_MIN_SEGMENT_FRAMES = 40     # Smooth out rapid switching (speed smoothing window)

# ============================================================================
# BOUT PARAMETERS
# ============================================================================
COURT_MIN_BOUT_FRAMES = 80
COURT_MAX_GAP = 200

# ============================================================================
# DIAGNOSTICS
# ============================================================================
COURT_SHOW_BOUNDARY_DIAG = True
COURT_BOUNDARY_DIAG_FRAMES = 50

# ============================================================================
# LEG TIPS (for upright check)
# ============================================================================
COURT_ALL_LEG_TIPS = ["T1L_TaTip", "T1R_TaTip", "T2L_TaTip", "T2R_TaTip",
                       "T3L_TaTip", "T3R_TaTip"]

# ============================================================================
# ABDOMEN TIP (for visualization)
# ============================================================================
COURT_ABD_TIP = "Abd_tip"

# ============================================================================
# ARENA DIMENSIONS (for XY trajectory panel)
# ============================================================================
COURT_ARENA_X_MM = 23.5
COURT_ARENA_Y_MM = 5.5

print("Courtship detection configured (unified pulse+sine).")
print(f"  Confidence: >{COURT_CONF_THRESHOLD} (gap bridge: {COURT_CONF_GAP_BRIDGE}, "
      f"wings_only={COURT_CONF_WINGS_ONLY})")
print(f"  Upright: {'ENABLED' if COURT_UPRIGHT_ENABLED else 'DISABLED'}")
print(f"  Wing oscillation: >{COURT_WING_THRESHOLD} mm/s (window: {COURT_WING_ACTIVITY_WINDOW})")
print(f"  Classification: speed >{COURT_PULSE_CLASSIFY_SPEED} mm/s → pulse, else → sine")
print(f"  Min bout: {COURT_MIN_BOUT_FRAMES} frames, gap bridge: {COURT_MAX_GAP}")

In [ ]:
# ============================================================================
# SHARED UTILITY FUNCTIONS
# (Defined here so courtship detection can run independently from walking)
# ============================================================================

def get_all_keypoint_names(df):
    """Get all unique keypoint names from dataframe columns."""
    cols = df.columns.tolist()
    kp_names = []
    seen = set()
    for col in cols:
        base_name = col.split('.')[0] if '.' in col else col
        if base_name not in seen:
            seen.add(base_name)
            kp_names.append(base_name)
    return kp_names


def extract_xyzc(df, kp_name, scale=JARVIS_SCALE):
    """Extract x, y, z coordinates and confidence for a keypoint."""
    cols = df.columns.tolist()
    start_idx = cols.index(kp_name)
    x = df.iloc[:, start_idx].values.astype(float) / scale
    y = df.iloc[:, start_idx + 1].values.astype(float) / scale
    z = df.iloc[:, start_idx + 2].values.astype(float) / scale
    conf = df.iloc[:, start_idx + 3].values.astype(float)
    return x, y, z, conf


def bridge_short_gaps(mask, max_gap):
    """Bridge short gaps (runs of False) in a boolean mask."""
    if max_gap <= 0:
        return mask
    bridged = mask.copy()
    in_gap = False
    gap_start = 0
    for i in range(len(bridged)):
        if not bridged[i]:
            if not in_gap:
                gap_start = i
                in_gap = True
        else:
            if in_gap:
                gap_len = i - gap_start
                if gap_len <= max_gap and gap_start > 0:
                    bridged[gap_start:i] = True
                in_gap = False
    return bridged


def compute_instant_speed(scutellum_data, start, end, fps=FPS):
    """Compute instantaneous speed at each frame (mm/s)."""
    x = scutellum_data['x'][start:end+1]
    y = scutellum_data['y'][start:end+1]
    dx = np.diff(x)
    dy = np.diff(y)
    displacement = np.sqrt(dx**2 + dy**2)
    speed = displacement * fps
    speed = np.concatenate([[speed[0] if len(speed) > 0 else 0], speed])
    mean_speed = np.nanmean(speed)
    max_speed = np.nanmax(speed)
    return speed, mean_speed, max_speed


def compute_total_distance(scutellum_data, start, end):
    """Compute total path distance and net displacement (mm)."""
    x = scutellum_data['x'][start:end+1]
    y = scutellum_data['y'][start:end+1]
    valid = ~np.isnan(x) & ~np.isnan(y)
    if valid.sum() < 2:
        return 0.0, 0.0
    x_valid = x[valid]
    y_valid = y[valid]
    dx = np.diff(x_valid)
    dy = np.diff(y_valid)
    total_distance = np.sum(np.sqrt(dx**2 + dy**2))
    net_displacement = np.sqrt((x_valid[-1] - x_valid[0])**2 + (y_valid[-1] - y_valid[0])**2)
    return total_distance, net_displacement


def find_contiguous_bouts_with_gap_bridging(valid_mask, min_frames, max_gap):
    """Find contiguous bouts, bridging small gaps."""
    bridged = valid_mask.copy()
    in_gap = False
    gap_start = 0
    for i in range(len(bridged)):
        if not bridged[i]:
            if not in_gap:
                gap_start = i
                in_gap = True
        else:
            if in_gap:
                gap_len = i - gap_start
                if gap_len <= max_gap and gap_start > 0 and bridged[gap_start - 1]:
                    bridged[gap_start:i] = True
                in_gap = False
    bouts = []
    in_bout = False
    start = 0
    for i, valid in enumerate(bridged):
        if valid and not in_bout:
            start = i
            in_bout = True
        elif not valid and in_bout:
            if i - start >= min_frames:
                bouts.append((start, i - 1))
            in_bout = False
    if in_bout and len(bridged) - start >= min_frames:
        bouts.append((start, len(bridged) - 1))
    return bouts


def _add_bout_boundaries(ax, actual_start, actual_end, add_legend=False):
    """Add bout boundary shading and vertical lines to an axis."""
    ax.axvspan(actual_start, actual_end, color='#2ca02c', alpha=0.08, zorder=0,
               label='Detected bout' if add_legend else None)
    ax.axvline(x=actual_start, color='#d62728', ls='--', lw=1.5, alpha=0.8, zorder=3,
               label='Bout start/end' if add_legend else None)
    ax.axvline(x=actual_end, color='#d62728', ls='--', lw=1.5, alpha=0.8, zorder=3)


# ============================================================================
# COURTSHIP DETECTION FUNCTIONS
# ============================================================================

COURT_FILTER_NAMES = ['confidence', 'upright', 'wing_oscillation']


def compute_wing_activity(df, scale=JARVIS_SCALE, fps=FPS,
                          wing_tips=None, window=None):
    """
    Compute windowed mean |dZ/dt| for each wing tip.

    NaN-safe: fills NaN gaps with 0 before smoothing, then corrects
    the mean to only count valid (non-NaN) frames in each window.

    Returns:
        activities: Dict mapping wing tip name -> smoothed |dZ/dt| array (mm/s)
        wing_data: Dict mapping wing tip name -> {'x','y','z','conf'}
    """
    if wing_tips is None:
        wing_tips = COURT_WING_TIPS
    if window is None:
        window = COURT_WING_ACTIVITY_WINDOW

    activities = {}
    wing_data = {}
    for tip in wing_tips:
        x, y, z, conf = extract_xyzc(df, tip, scale)
        wing_data[tip] = {'x': x, 'y': y, 'z': z, 'conf': conf}
        dz = np.abs(np.diff(z, prepend=z[0]) * fps)
        nan_mask = np.isnan(dz)
        # Fill NaN with 0 so uniform_filter1d doesn't propagate NaN
        filled = np.where(nan_mask, 0.0, dz)
        smoothed = uniform_filter1d(filled, size=window)
        # Correct: divide by fraction of valid frames in each window
        valid_frac = uniform_filter1d((~nan_mask).astype(float), size=window)
        activities[tip] = np.where(valid_frac > 0.01, smoothed / valid_frac, np.nan)

    return activities, wing_data


def apply_courtship_filter(df, scale=JARVIS_SCALE, fps=FPS):
    """
    Apply frame-level filters for unified courtship detection.

    Filters:
    1. Confidence > threshold (all keypoints, or wings-only if COURT_CONF_WINGS_ONLY)
    2. Upright: scutellum Z > all leg tip Z
    3. Wing oscillation: at least one wing pair (V12+V13) > threshold

    Returns:
        valid_mask, leg_data, scut_data, wing_data, activities, filter_masks,
        diagnostics, abd_data
    """
    n_frames = len(df)

    # --- Extract leg tip data ---
    leg_data = {}
    for tip in COURT_ALL_LEG_TIPS:
        x, y, z, conf = extract_xyzc(df, tip, scale)
        leg_data[tip] = {'x': x, 'y': y, 'z': z, 'conf': conf}

    # --- Extract scutellum ---
    scut_x, scut_y, scut_z, scut_conf = extract_xyzc(df, SCUTELLUM, scale)
    scut_data = {'x': scut_x, 'y': scut_y, 'z': scut_z, 'conf': scut_conf}

    # --- Extract abdomen tip ---
    abd_data = {}
    try:
        abd_x, abd_y, abd_z, abd_conf = extract_xyzc(df, COURT_ABD_TIP, scale)
        abd_data = {'x': abd_x, 'y': abd_y, 'z': abd_z, 'conf': abd_conf}
    except (ValueError, IndexError):
        pass

    # --- Wing activity ---
    activities, wing_data = compute_wing_activity(df, scale, fps)

    # =========================================================================
    # FILTER 1: Confidence
    # If COURT_CONF_WINGS_ONLY: only check wing keypoint confidences
    # Otherwise: check ALL keypoints
    # =========================================================================
    conf_mask = np.ones(n_frames, dtype=bool)
    n_checked = 0

    if COURT_CONF_WINGS_ONLY:
        # Only check wing keypoints
        for tip in COURT_WING_TIPS:
            conf_mask &= (wing_data[tip]['conf'] >= COURT_CONF_THRESHOLD)
            n_checked += 1
    else:
        # Check all keypoints
        all_kp_names = get_all_keypoint_names(df)
        for kp in all_kp_names:
            try:
                _, _, _, conf = extract_xyzc(df, kp, scale)
                conf_mask &= (conf >= COURT_CONF_THRESHOLD)
                n_checked += 1
            except (ValueError, IndexError):
                pass

    if COURT_CONF_GAP_BRIDGE > 0:
        conf_mask = bridge_short_gaps(conf_mask, COURT_CONF_GAP_BRIDGE)

    # =========================================================================
    # FILTER 2: Upright (scutellum Z > all leg tip Z)
    # =========================================================================
    upright_mask = np.ones(n_frames, dtype=bool)
    if COURT_UPRIGHT_ENABLED:
        for tip in COURT_ALL_LEG_TIPS:
            upright_mask &= (scut_z > leg_data[tip]['z'])

    # =========================================================================
    # FILTER 3: Wing oscillation
    # At least one wing (both V12 + V13) exceeds detection threshold
    # =========================================================================
    left_active = ((activities['WingL_V12'] > COURT_WING_THRESHOLD) &
                   (activities['WingL_V13'] > COURT_WING_THRESHOLD))
    right_active = ((activities['WingR_V12'] > COURT_WING_THRESHOLD) &
                    (activities['WingR_V13'] > COURT_WING_THRESHOLD))
    wing_mask = left_active | right_active

    # =========================================================================
    # COMBINE ALL FILTERS
    # =========================================================================
    valid_mask = conf_mask & upright_mask & wing_mask

    filter_masks = {
        'confidence': conf_mask,
        'upright': upright_mask,
        'wing_oscillation': wing_mask,
    }

    # Confidence is a hard boundary: bouts cannot cross a confidence failure.
    # Attach conf_mask to filter_masks so detect_courtship_bouts can use it.
    filter_masks['_conf_mask_for_detection'] = conf_mask

    conf_mode = "wings only" if COURT_CONF_WINGS_ONLY else "all keypoints"
    diagnostics = {
        'total_frames': n_frames,
        'confidence_pass': int(conf_mask.sum()),
        'confidence_pass_pct': 100 * conf_mask.sum() / n_frames,
        'confidence_kps_checked': n_checked,
        'confidence_mode': conf_mode,
        'upright_pass': int(upright_mask.sum()),
        'upright_pass_pct': 100 * upright_mask.sum() / n_frames,
        'wing_oscillation_pass': int(wing_mask.sum()),
        'wing_oscillation_pass_pct': 100 * wing_mask.sum() / n_frames,
        'combined_pass': int(valid_mask.sum()),
        'combined_pass_pct': 100 * valid_mask.sum() / n_frames,
    }

    return valid_mask, leg_data, scut_data, wing_data, activities, filter_masks, diagnostics, abd_data


def classify_courtship_segments(bout_info, scut_data, fps=FPS):
    """
    Classify frames within a courtship bout as pulse or sine.

    Uses smoothed scutellum speed:
      speed > COURT_PULSE_CLASSIFY_SPEED → pulse
      speed <= COURT_PULSE_CLASSIFY_SPEED → sine

    Adds to bout_info: segments, n_pulse_frames, n_sine_frames, pct_pulse, pct_sine
    """
    start, end = bout_info['start'], bout_info['end']
    speed, _, _ = compute_instant_speed(scut_data, start, end, fps)

    # Smooth speed to avoid rapid pulse/sine switching
    smooth_window = min(COURT_MIN_SEGMENT_FRAMES, len(speed))
    if smooth_window > 1:
        smoothed = uniform_filter1d(speed, size=smooth_window)
    else:
        smoothed = speed

    is_pulse = smoothed > COURT_PULSE_CLASSIFY_SPEED

    # Find contiguous segments
    segments = []
    current_type = 'pulse' if is_pulse[0] else 'sine'
    seg_start_local = 0

    for i in range(1, len(is_pulse)):
        frame_type = 'pulse' if is_pulse[i] else 'sine'
        if frame_type != current_type:
            segments.append({
                'start': start + seg_start_local,
                'end': start + i - 1,
                'type': current_type,
                'n_frames': i - seg_start_local,
                'duration_s': (i - seg_start_local) / fps,
            })
            current_type = frame_type
            seg_start_local = i

    # Last segment
    segments.append({
        'start': start + seg_start_local,
        'end': end,
        'type': current_type,
        'n_frames': end - start - seg_start_local + 1,
        'duration_s': (end - start - seg_start_local + 1) / fps,
    })

    n_pulse = int(is_pulse.sum())
    n_sine = len(is_pulse) - n_pulse

    bout_info['segments'] = segments
    bout_info['n_pulse_frames'] = n_pulse
    bout_info['n_sine_frames'] = n_sine
    bout_info['pct_pulse'] = 100 * n_pulse / len(is_pulse)
    bout_info['pct_sine'] = 100 * n_sine / len(is_pulse)


def diagnose_courtship_boundaries(bout_start, bout_end, filter_masks,
                                   n_frames_context=COURT_BOUNDARY_DIAG_FRAMES):
    """Analyze frames around courtship bout boundaries to show which filters fail."""
    total_frames = len(filter_masks['confidence'])

    pre_start = max(0, bout_start - n_frames_context)
    pre_range = slice(pre_start, bout_start)
    post_end = min(total_frames, bout_end + n_frames_context + 1)
    post_range = slice(bout_end + 1, post_end)

    pre_failures = {'n_frames_analyzed': bout_start - pre_start}
    post_failures = {'n_frames_analyzed': post_end - bout_end - 1}

    for name in COURT_FILTER_NAMES:
        mask = filter_masks[name]
        pre_failures[name] = int((~mask[pre_range]).sum())
        post_failures[name] = int((~mask[post_range]).sum())

    pre_items = [(k, v) for k, v in pre_failures.items() if k != 'n_frames_analyzed']
    post_items = [(k, v) for k, v in post_failures.items() if k != 'n_frames_analyzed']
    pre_max = max(pre_items, key=lambda x: x[1]) if pre_items else ('none', 0)
    post_max = max(post_items, key=lambda x: x[1]) if post_items else ('none', 0)

    return {
        'pre_bout_failures': pre_failures,
        'post_bout_failures': post_failures,
        'primary_cause_before': pre_max[0] if pre_max[1] > 0 else None,
        'primary_cause_after': post_max[0] if post_max[1] > 0 else None,
    }


def detect_courtship_bouts(df, scale=JARVIS_SCALE, fps=FPS, verbose=True):
    """
    Unified courtship bout detection with pulse/sine classification.

    Step 1: Detect courtship frames (conf + upright + wing oscillation)
    Step 2: Find contiguous bouts
    Step 3: Classify frames within each bout as pulse or sine (by speed)

    Returns:
        bouts, leg_data, scut_data, wing_data, activities, filter_masks,
        diagnostics, abd_data
    """
    if verbose:
        print("\n" + "=" * 70)
        print("COURTSHIP BOUT DETECTION (unified pulse + sine)")
        print("=" * 70)
        conf_mode = "wings only" if COURT_CONF_WINGS_ONLY else "all keypoints"
        print(f"  Confidence: >{COURT_CONF_THRESHOLD} ({conf_mode}, "
              f"gap bridge: {COURT_CONF_GAP_BRIDGE})")
        print(f"  Upright: {'ENABLED' if COURT_UPRIGHT_ENABLED else 'DISABLED'}")
        print(f"  Wing oscillation: >{COURT_WING_THRESHOLD} mm/s (window: {COURT_WING_ACTIVITY_WINDOW})")
        print(f"  Classification: speed >{COURT_PULSE_CLASSIFY_SPEED} → pulse, else → sine")
        print(f"  Min bout: {COURT_MIN_BOUT_FRAMES} frames, gap bridge: {COURT_MAX_GAP}")

    # Step 1: Frame-level filtering
    valid_mask, leg_data, scut_data, wing_data, activities, filter_masks, diagnostics, abd_data = \
        apply_courtship_filter(df, scale, fps)

    if verbose:
        print(f"\n--- Frame-level filtering ---")
        print(f"  Total frames: {diagnostics['total_frames']}")
        print(f"  Confidence pass: {diagnostics['confidence_pass_pct']:.1f}% "
              f"({diagnostics['confidence_kps_checked']} kps, {diagnostics['confidence_mode']})")
        print(f"  Upright pass: {diagnostics['upright_pass_pct']:.1f}%")
        print(f"  Wing oscillation pass: {diagnostics['wing_oscillation_pass_pct']:.1f}%")
        print(f"  Combined valid: {diagnostics['combined_pass_pct']:.1f}%")

    # Step 2: Find contiguous bouts with gap bridging, but ONLY within
    # contiguous confidence-valid regions.  Confidence is a hard boundary —
    # a bout must not cross a frame where all wing keypoints drop below
    # COURT_CONF_THRESHOLD (after gap bridging).  Upright / wing-oscillation
    # failures may still be bridged up to COURT_MAX_GAP within a region.
    conf_mask_det = filter_masks['_conf_mask_for_detection']
    # Find contiguous confidence-valid spans (no additional bridging here —
    # COURT_CONF_GAP_BRIDGE was already applied inside apply_courtship_filter)
    conf_regions = []
    in_conf = False
    cr_start = 0
    for i, v in enumerate(conf_mask_det):
        if v and not in_conf:
            cr_start = i
            in_conf = True
        elif not v and in_conf:
            conf_regions.append((cr_start, i - 1))
            in_conf = False
    if in_conf:
        conf_regions.append((cr_start, len(conf_mask_det) - 1))

    bouts_raw = []
    for cr_s, cr_e in conf_regions:
        region_valid = valid_mask[cr_s:cr_e+1]
        region_bouts = find_contiguous_bouts_with_gap_bridging(
            region_valid, COURT_MIN_BOUT_FRAMES, COURT_MAX_GAP
        )
        for bs, be in region_bouts:
            bouts_raw.append((cr_s + bs, cr_s + be))

    if verbose:
        print(f"\n--- Bout detection ---")
        print(f"  Found: {len(bouts_raw)} courtship bouts (min {COURT_MIN_BOUT_FRAMES} frames)")

    # Step 3: Build bout info + classify pulse/sine + boundary diagnostics
    court_bouts = []
    for i, (start, end) in enumerate(bouts_raw):
        n = end - start + 1
        _, mean_speed, max_speed = compute_instant_speed(scut_data, start, end, fps)
        total_dist, net_disp = compute_total_distance(scut_data, start, end)

        bout_info = {
            'start': start,
            'end': end,
            'n_frames': n,
            'duration_s': n / fps,
            'mean_speed_mm_s': mean_speed,
            'max_speed_mm_s': max_speed,
            'total_distance_mm': total_dist,
            'bout_idx': i + 1,
        }

        # Classify pulse/sine segments
        classify_courtship_segments(bout_info, scut_data, fps)

        # Boundary diagnostics
        if COURT_SHOW_BOUNDARY_DIAG:
            bout_info['boundary_diagnostics'] = diagnose_courtship_boundaries(
                start, end, filter_masks, COURT_BOUNDARY_DIAG_FRAMES
            )

        court_bouts.append(bout_info)

    # Print boundary diagnostics
    if verbose and COURT_SHOW_BOUNDARY_DIAG and court_bouts:
        print(f"\n--- Boundary diagnostics (first 5 bouts) ---")
        for bout in court_bouts[:5]:
            bd = bout['boundary_diagnostics']
            pre = bd['pre_bout_failures']
            post = bd['post_bout_failures']
            print(f"\n  Bout {bout['bout_idx']} boundary analysis:")
            print(f"    Before bout ({pre['n_frames_analyzed']} frames):")
            for name in COURT_FILTER_NAMES:
                if pre[name] > 0:
                    marker = " <-- PRIMARY" if bd['primary_cause_before'] == name else ""
                    print(f"      {name}: {pre[name]} failures{marker}")
            if post['n_frames_analyzed'] > 0:
                print(f"    After bout ({post['n_frames_analyzed']} frames):")
                for name in COURT_FILTER_NAMES:
                    if post[name] > 0:
                        marker = " <-- PRIMARY" if bd['primary_cause_after'] == name else ""
                        print(f"      {name}: {post[name]} failures{marker}")

    if verbose and court_bouts:
        print(f"\n--- Courtship bout summary ---")
        for bout in court_bouts[:10]:
            seg_str = " -> ".join(f"{s['type']}({s['n_frames']})" for s in bout['segments'])
            print(f"  Bout {bout['bout_idx']}: frames {bout['start']}-{bout['end']} "
                  f"({bout['n_frames']} frames, {bout['duration_s']:.3f}s, "
                  f"pulse={bout['pct_pulse']:.0f}% sine={bout['pct_sine']:.0f}%)")
            print(f"    Segments: {seg_str}")
        if len(court_bouts) > 10:
            print(f"  ... and {len(court_bouts) - 10} more")

    diagnostics['n_courtship_bouts'] = len(court_bouts)

    return court_bouts, leg_data, scut_data, wing_data, activities, filter_masks, diagnostics, abd_data


print("Courtship detection functions defined (unified with pulse/sine classification).")

In [ ]:
# ============================================================================
# COURTSHIP BOUT VISUALIZATION
# ============================================================================
# Segment shading colors
_SEG_COLORS = {
    'pulse': '#E76F5133',   # red-orange, semi-transparent
    'sine': '#2A9D8F33',    # teal, semi-transparent
    'waggle': '#E9C46A33',  # gold, semi-transparent
}
_SEG_EDGE_COLORS = {
    'pulse': '#E76F51',
    'sine': '#2A9D8F',
    'waggle': '#E9C46A',
}

# Z axis limits for wing/abd panel
Z_PLOT_MIN = 0.0
Z_PLOT_MAX = 5.0


def _add_segment_shading(ax, segments, frame_offset=0):
    """Add colored shading for pulse/sine/waggle segments inside a bout."""
    added_labels = set()
    for seg in segments:
        stype = seg['type']
        if stype == 'quiet':
            continue
        s = seg['start'] + frame_offset
        e = seg['end'] + frame_offset
        color = _SEG_COLORS.get(stype, '#cccccc33')
        edge = _SEG_EDGE_COLORS.get(stype, '#999999')
        label = stype.capitalize() if stype not in added_labels else None
        added_labels.add(stype)
        ax.axvspan(s, e, color=color, zorder=1, label=label)


def _add_segment_freq_annotations(ax, segments, window_features, frame_offset=0):
    """Add compact peak frequency annotations at the bottom of each segment."""
    if window_features is None:
        return
    wc = window_features.get('window_centers', None)
    pf = window_features.get('peak_freq', None)
    if wc is None or pf is None:
        return

    for seg in segments:
        stype = seg['type']
        if stype == 'quiet':
            continue
        s, e = seg['start'], seg['end']
        in_seg = (wc >= s) & (wc <= e)
        if in_seg.sum() == 0:
            continue
        seg_freqs = pf[in_seg]
        median_f = np.nanmedian(seg_freqs)
        min_f = np.nanmin(seg_freqs)
        max_f = np.nanmax(seg_freqs)

        mid_frame = (s + e) / 2 + frame_offset
        edge_color = _SEG_EDGE_COLORS.get(stype, '#999999')

        if min_f == max_f:
            freq_str = f"{median_f:.0f}Hz"
        else:
            freq_str = f"{min_f:.0f}-{max_f:.0f}Hz"

        ax.annotate(
            freq_str,
            xy=(mid_frame, 0.02), xycoords=('data', 'axes fraction'),
            fontsize=7, color=edge_color, fontweight='bold',
            ha='center', va='bottom',
            bbox=dict(boxstyle='round,pad=0.15', fc='white', ec=edge_color,
                      alpha=0.85, lw=0.7))


def plot_courtship_bout_figure(
    bout_info, leg_data, scut_data, wing_data, wing_activities, filter_masks,
    frame_offset=0, fps=FPS, save_path=None, dpi=PDF_DPI,
    context_frames=200, abd_data=None, window_features=None
):
    """
    Create courtship bout figure with 5 panels:
    1. Wing tip Z + Abd_tip Z, with pulse/sine/waggle segment shading & freq labels
    2. Leg tip Z trajectories
    3. Scutellum speed
    4. Wing confidence (per tip + threshold line + low-conf shading)
    5. Wing XY trajectory in arena
    """
    start = bout_info['start']
    end = bout_info['end']
    bout_idx = bout_info.get('bout_idx', '?')
    pct_pulse = bout_info.get('pct_pulse', 0)
    pct_sine = bout_info.get('pct_sine', 0)
    pct_waggle = bout_info.get('pct_waggle', 0)
    segments = bout_info.get('segments', [])
    dom_wing = bout_info.get('dominant_wing', '?')

    data_len = len(scut_data['x'])
    ctx_start = max(0, start - context_frames)
    ctx_end = min(data_len - 1, end + context_frames)

    actual_start = start + frame_offset
    actual_end = end + frame_offset
    frames = np.arange(ctx_start + frame_offset, ctx_end + frame_offset + 1)

    speed, _, _ = compute_instant_speed(scut_data, ctx_start, ctx_end, fps)
    _, mean_speed, max_speed = compute_instant_speed(scut_data, start, end, fps)

    bd = bout_info.get('boundary_diagnostics', None)
    start_reason = bd.get('primary_cause_before') if bd else None
    end_reason = bd.get('primary_cause_after') if bd else None

    mode_parts = []
    if pct_pulse > 0:
        mode_parts.append(f"pulse={pct_pulse:.0f}%")
    if pct_sine > 0:
        mode_parts.append(f"sine={pct_sine:.0f}%")
    if pct_waggle > 0:
        mode_parts.append(f"waggle={pct_waggle:.0f}%")
    mode_str = " ".join(mode_parts) if mode_parts else "no song"

    duration_s = bout_info['n_frames'] / fps

    fig, axes = plt.subplots(
        5, 1, figsize=(14, 18),
        gridspec_kw={'height_ratios': [1.2, 1, 0.7, 0.7, 1], 'hspace': 0.28}
    )
    ax_wing, ax_leg, ax_speed, ax_conf, ax_xy = axes

    boundary_str = ""
    if start_reason or end_reason:
        parts = []
        if start_reason:
            parts.append(f"start: {start_reason}")
        if end_reason:
            parts.append(f"end: {end_reason}")
        boundary_str = f"  |  {' | '.join(parts)}"

    fig.suptitle(
        f'Courtship Bout {bout_idx}  |  Frames {actual_start}-{actual_end}  |  '
        f'{duration_s:.3f}s  |  Wing: {dom_wing}  |  {mode_str}{boundary_str}',
        fontsize=12, fontweight='bold', y=0.995
    )

    # ===== PANEL 1: Wing Tip Z + Abd Tip Z =====
    wing_colors = {
        'WingL_V12': '#7B2D8E', 'WingL_V13': '#A855F7',
        'WingR_V12': '#0369A1', 'WingR_V13': '#38BDF8',
    }
    _add_bout_boundaries(ax_wing, actual_start, actual_end, add_legend=True)
    _add_segment_shading(ax_wing, segments, frame_offset)

    for tip in COURT_WING_TIPS:
        z = wing_data[tip]['z'][ctx_start:ctx_end+1].copy()
        z = np.where((z >= Z_PLOT_MIN) & (z <= Z_PLOT_MAX), z, np.nan)
        label = tip.replace('Wing', 'W')
        color = wing_colors.get(tip, 'gray')
        ax_wing.plot(frames, z, label=label, color=color, lw=1.5, alpha=0.9)

    if abd_data and 'z' in abd_data:
        abd_z = abd_data['z'][ctx_start:ctx_end+1].copy()
        abd_z = np.where((abd_z >= Z_PLOT_MIN) & (abd_z <= Z_PLOT_MAX), abd_z, np.nan)
        ax_wing.plot(frames, abd_z, label='Abd_tip', color='#D4A017',
                     lw=1.8, alpha=0.85, ls='-')

    _add_segment_freq_annotations(ax_wing, segments, window_features, frame_offset)

    ax_wing.set_ylabel('Z (mm)', fontsize=12)
    ax_wing.set_title('Wing + Abdomen Z', fontsize=11)
    ax_wing.set_ylim(Z_PLOT_MIN, Z_PLOT_MAX)
    ax_wing.legend(loc='upper left', bbox_to_anchor=(1.02, 1), fontsize=9, ncol=1)
    ax_wing.grid(True, alpha=0.3)

    if start_reason:
        ax_wing.annotate(
            start_reason, xy=(actual_start, 0.95), xycoords=('data', 'axes fraction'),
            xytext=(6, 0), textcoords='offset points', fontsize=7, color='#d62728',
            fontweight='bold', ha='left', va='top',
            bbox=dict(boxstyle='round,pad=0.2', fc='white', ec='#d62728', alpha=0.85))
    if end_reason:
        ax_wing.annotate(
            end_reason, xy=(actual_end, 0.95), xycoords=('data', 'axes fraction'),
            xytext=(-6, 0), textcoords='offset points', fontsize=7, color='#d62728',
            fontweight='bold', ha='right', va='top',
            bbox=dict(boxstyle='round,pad=0.2', fc='white', ec='#d62728', alpha=0.85))

    # ===== PANEL 2: Leg Tip Z Trajectories =====
    leg_colors = {
        'T1L_TaTip': '#E63946', 'T1R_TaTip': '#457B9D',
        'T2L_TaTip': '#F4A261', 'T2R_TaTip': '#2A9D8F',
        'T3L_TaTip': '#9B2226', 'T3R_TaTip': '#1D3557',
    }
    _add_bout_boundaries(ax_leg, actual_start, actual_end)
    _add_segment_shading(ax_leg, segments, frame_offset)
    for tip in COURT_ALL_LEG_TIPS:
        z = leg_data[tip]['z'][ctx_start:ctx_end+1].copy()
        z = np.where((z >= Z_PLOT_MIN) & (z <= Z_PLOT_MAX), z, np.nan)
        label = tip.replace('_TaTip', '')
        color = leg_colors.get(tip, 'gray')
        ax_leg.plot(frames, z, label=label, color=color, lw=1.0, alpha=0.8)

    ax_leg.set_ylabel('Leg Tip Z (mm)', fontsize=12)
    ax_leg.set_title('Leg Z Trajectories', fontsize=11)
    ax_leg.set_ylim(Z_PLOT_MIN, Z_PLOT_MAX)
    ax_leg.legend(loc='upper left', bbox_to_anchor=(1.02, 1), fontsize=9, ncol=1)
    ax_leg.grid(True, alpha=0.3)
    ax_leg.sharex(ax_wing)

    # ===== PANEL 3: Scutellum Speed =====
    _add_bout_boundaries(ax_speed, actual_start, actual_end)
    _add_segment_shading(ax_speed, segments, frame_offset)
    ax_speed.plot(frames, speed, color='#264653', lw=1.2)
    ax_speed.fill_between(frames, 0, speed, alpha=0.3, color='#264653')
    ax_speed.axhline(y=COURT_PULSE_CLASSIFY_SPEED, color='#E76F51', ls='--', lw=1.5,
                     label=f'Pulse/sine threshold ({COURT_PULSE_CLASSIFY_SPEED} mm/s)')
    ax_speed.axhline(y=mean_speed, color='#2A9D8F', ls=':', lw=1.5,
                     label=f'Bout mean: {mean_speed:.1f} mm/s')
    ax_speed.set_ylabel('Speed (mm/s)', fontsize=12)
    ax_speed.set_title('Scutellum Speed', fontsize=11)
    ax_speed.legend(loc='upper right', fontsize=9)
    ax_speed.grid(True, alpha=0.3)
    ax_speed.set_ylim(bottom=0)
    ax_speed.sharex(ax_wing)

    # ===== PANEL 4: Wing Confidence =====
    _add_bout_boundaries(ax_conf, actual_start, actual_end)

    for tip in COURT_WING_TIPS:
        c = wing_data[tip]['conf'][ctx_start:ctx_end+1].astype(float)
        color = wing_colors.get(tip, 'gray')
        label = tip.replace('Wing', 'W')
        ax_conf.plot(frames, c, label=label, color=color, lw=1.0, alpha=0.85)

    ax_conf.axhline(y=COURT_CONF_THRESHOLD, color='red', ls='--', lw=1.2,
                    label=f'Threshold ({COURT_CONF_THRESHOLD})')

    # Shade frames that fail the confidence filter
    conf_mask_ctx = filter_masks['confidence'][ctx_start:ctx_end+1]
    fail_transitions = np.diff(np.concatenate([[False], ~conf_mask_ctx, [False]]).astype(int))
    fail_starts = np.where(fail_transitions == 1)[0]
    fail_ends   = np.where(fail_transitions == -1)[0]
    for fs, fe in zip(fail_starts, fail_ends):
        ax_conf.axvspan(frames[fs], frames[min(fe, len(frames)-1)],
                        color='red', alpha=0.12, zorder=0)

    ax_conf.set_ylabel('Confidence', fontsize=11)
    ax_conf.set_title('Wing Keypoint Confidence  (red shading = filter fails)', fontsize=11)
    ax_conf.set_ylim(-0.05, 1.05)
    ax_conf.legend(loc='upper left', bbox_to_anchor=(1.02, 1), fontsize=9, ncol=1)
    ax_conf.grid(True, alpha=0.3)
    ax_conf.sharex(ax_wing)

    # ===== PANEL 5: Wing XY Trajectory in Arena =====
    arena_x = COURT_ARENA_X_MM
    arena_y = COURT_ARENA_Y_MM

    rect_x = [0, arena_x, arena_x, 0, 0]
    rect_y = [0, 0, arena_y, arena_y, 0]
    ax_xy.plot(rect_x, rect_y, 'k-', lw=2, label=f'Arena ({arena_x}x{arena_y}mm)')

    if dom_wing == 'L':
        xy_tips = [('WingL_V12', '#7B2D8E'), ('WingL_V13', '#A855F7')]
    elif dom_wing == 'R':
        xy_tips = [('WingR_V12', '#0369A1'), ('WingR_V13', '#38BDF8')]
    else:
        xy_tips = [('WingR_V12', '#0369A1'), ('WingL_V12', '#7B2D8E')]

    for tip, color in xy_tips:
        if start - ctx_start > 1:
            pre_x = wing_data[tip]['x'][ctx_start:start+1]
            pre_y = wing_data[tip]['y'][ctx_start:start+1]
            v = ~np.isnan(pre_x) & ~np.isnan(pre_y)
            if v.sum() > 1:
                ax_xy.plot(pre_x[v], pre_y[v], color='gray', ls='--', lw=0.8, alpha=0.4)

        if ctx_end - end > 1:
            post_x = wing_data[tip]['x'][end:ctx_end+1]
            post_y = wing_data[tip]['y'][end:ctx_end+1]
            v = ~np.isnan(post_x) & ~np.isnan(post_y)
            if v.sum() > 1:
                ax_xy.plot(post_x[v], post_y[v], color='gray', ls=':', lw=0.8, alpha=0.4)

        bx = wing_data[tip]['x'][start:end+1]
        by = wing_data[tip]['y'][start:end+1]
        v = ~np.isnan(bx) & ~np.isnan(by)
        xv, yv = bx[v], by[v]

        if len(xv) > 1:
            pts = np.array([xv, yv]).T.reshape(-1, 1, 2)
            segs = np.concatenate([pts[:-1], pts[1:]], axis=1)
            t = np.linspace(0, 1, len(segs))
            lc = LineCollection(segs, cmap='viridis', norm=plt.Normalize(0, 1))
            lc.set_array(t)
            lc.set_linewidth(2.0)
            ax_xy.add_collection(lc)

        tip_label = tip.replace('Wing', 'W')
        if len(xv) > 0:
            ax_xy.plot(xv[0], yv[0], 'o', color=color, ms=8, zorder=5,
                       label=f'{tip_label} start')
            ax_xy.plot(xv[-1], yv[-1], '^', color=color, ms=8, zorder=5,
                       label=f'{tip_label} end')

    padding_x = arena_x * 0.05
    padding_y = arena_y * 0.15
    ax_xy.set_xlim(-padding_x, arena_x + padding_x)
    ax_xy.set_ylim(-padding_y, arena_y + padding_y)
    ax_xy.set_aspect('equal')
    ax_xy.set_xlabel('X (mm)', fontsize=12)
    ax_xy.set_ylabel('Y (mm)', fontsize=12)
    total_dist = bout_info.get('total_distance_mm', 0)
    ax_xy.set_title(f'Wing XY Trajectory (dominant: {dom_wing}) | Scut dist: {total_dist:.2f} mm',
                    fontsize=11)
    ax_xy.legend(loc='upper right', fontsize=8, ncol=2)
    ax_xy.grid(True, alpha=0.3)

    sm = plt.cm.ScalarMappable(cmap='viridis', norm=plt.Normalize(0, 1))
    sm.set_array([])
    cbar = plt.colorbar(sm, ax=ax_xy, shrink=0.6, pad=0.02, orientation='vertical')
    cbar.set_label('Time (normalized)', fontsize=10)

    plt.tight_layout()

    if save_path:
        fig.savefig(save_path, format='pdf', dpi=dpi, bbox_inches='tight')
        print(f"  Saved: {save_path}")

    return fig


print("Courtship visualization function defined (5 panels: wing Z, leg Z, speed, confidence, XY).")


In [ ]:
# ============================================================================
# FFT-BASED COURTSHIP SONG MODE CLASSIFICATION
# ============================================================================
# Uses spectral analysis of wing tip Z trajectories to classify song modes:
#   - pulse: carrier 150-300 Hz, pulsatile (1-3 cycle bursts, ~35ms IPI)
#   - sine:  carrier 100-200 Hz, continuous hum
#   - waggle: 5-25 Hz anti-phase bilateral wing oscillation
#
# Reference: Li, Thieringer, Gao & Murthy (2025) bioRxiv 10.1101/2025.08.21.671456
# ============================================================================

from scipy.signal import spectrogram as sp_spectrogram, csd, butter, sosfiltfilt, hilbert

# --- FFT configuration ---
FFT_NPERSEG = 128            # frames (160ms at 800fps), freq resolution = 6.25 Hz
FFT_HOP = 32                 # frames (40ms), time step per window
FFT_SONG_BAND = (80, 350)    # Hz — broad singing band (covers both pulse and sine)
FFT_SINE_BAND = (80, 200)    # Hz — sine carrier range
FFT_PULSE_BAND = (150, 350)  # Hz — pulse carrier range
FFT_WAGGLE_BAND = (5, 25)    # Hz — waggling oscillation range

FFT_SONG_POWER_THRESHOLD = 5e-7   # min song-band power to classify as singing
FFT_PULSE_PEAK_FREQ_MIN = 137.5   # min peak freq (Hz) for pulse — sine tops out ~125
FFT_PULSE_FREQ_RATIO_MIN = 0.30   # min pulse_band/song_band ratio for pulse
FFT_WAGGLE_PHASE_MIN = 2.0        # min |L-R phase diff| (radians) for waggling (~pi)
FFT_WAGGLE_BILATERAL_MIN = 0.4    # min(L,R)/max(L,R) power ratio for bilateral
FFT_MIN_SEGMENT_FRAMES = 40       # merge segments shorter than this into neighbors


def _interp_nan(z):
    """Fill NaN gaps with nearest-neighbor interpolation."""
    nans = np.isnan(z)
    if not nans.any():
        return z.copy()
    out = z.copy()
    idx = np.arange(len(z))
    valid = ~nans
    if valid.sum() < 2:
        return np.zeros_like(z)
    out[nans] = np.interp(idx[nans], idx[valid], z[valid])
    return out


def _band_power(Sxx, f, fmin, fmax):
    """Mean power in frequency band for each time window."""
    mask = (f >= fmin) & (f <= fmax)
    if mask.sum() == 0:
        return np.zeros(Sxx.shape[1])
    return np.mean(Sxx[mask, :], axis=0)


def _peak_freq(Sxx, f, fmin, fmax):
    """Peak frequency in band for each time window."""
    mask = (f >= fmin) & (f <= fmax)
    if mask.sum() == 0:
        return np.zeros(Sxx.shape[1])
    f_band = f[mask]
    Sxx_band = Sxx[mask, :]
    peak_idx = np.argmax(Sxx_band, axis=0)
    return f_band[peak_idx]


def _merge_short_segments(labels, min_len):
    """Merge segments shorter than min_len into their longest neighbor."""
    if len(labels) == 0:
        return labels
    out = labels.copy()
    # Find contiguous runs via element-wise comparison (not np.diff, which fails on strings)
    changes = np.where(out[1:] != out[:-1])[0] + 1
    starts = np.concatenate([[0], changes])
    ends = np.concatenate([changes, [len(out)]])

    for s_i, e_i in zip(starts, ends):
        seg_len = e_i - s_i
        if seg_len >= min_len:
            continue
        # Find left and right neighbor labels
        left_label = out[s_i - 1] if s_i > 0 else None
        right_label = out[e_i] if e_i < len(out) else None
        # Merge into the neighbor; prefer the one that matches on both sides
        if left_label is not None and right_label is not None:
            # Prefer left (maintains temporal continuity)
            out[s_i:e_i] = left_label
        elif left_label is not None:
            out[s_i:e_i] = left_label
        elif right_label is not None:
            out[s_i:e_i] = right_label

    return out


def classify_song_modes_fft(wing_data, start, end, fps=FPS):
    """
    FFT-based classification of courtship wing behaviors.

    Computes spectrograms of wing tip Z and classifies each time window as:
      pulse  — high-freq carrier (150-350 Hz), pulsatile envelope, unilateral
      sine   — lower-freq carrier (80-200 Hz), continuous, unilateral
      waggle — low-freq (5-25 Hz), anti-phase bilateral oscillation
      quiet  — no significant spectral activity

    Pulse requires BOTH: peak_freq >= 137.5 Hz AND pulse/song ratio >= 0.30
    Otherwise singing windows default to sine.

    Args:
        wing_data: dict mapping tip name -> {'z': array, ...}
        start, end: frame indices (inclusive) into wing_data arrays
        fps: sampling rate

    Returns:
        frame_labels: array of str labels for frames start..end
        window_features: dict of per-window feature arrays
    """
    n_frames = end - start + 1
    nperseg = min(FFT_NPERSEG, n_frames)
    noverlap = nperseg - FFT_HOP
    if noverlap < 0:
        noverlap = 0

    # --- Compute spectrograms for each wing tip ---
    tips_L = ['WingL_V12', 'WingL_V13']
    tips_R = ['WingR_V12', 'WingR_V13']
    all_tips = tips_L + tips_R

    spectrograms = {}
    f_axis = None
    t_axis = None
    for tip in all_tips:
        z = _interp_nan(wing_data[tip]['z'][start:end+1])
        f, t, Sxx = sp_spectrogram(z, fs=fps, nperseg=nperseg,
                                    noverlap=noverlap, detrend='linear')
        spectrograms[tip] = Sxx
        if f_axis is None:
            f_axis = f
            t_axis = t

    n_windows = len(t_axis)

    # --- Per-side features (average V12 + V13) ---
    Sxx_L = (spectrograms['WingL_V12'] + spectrograms['WingL_V13']) / 2
    Sxx_R = (spectrograms['WingR_V12'] + spectrograms['WingR_V13']) / 2

    song_L = _band_power(Sxx_L, f_axis, *FFT_SONG_BAND)
    song_R = _band_power(Sxx_R, f_axis, *FFT_SONG_BAND)
    sine_L = _band_power(Sxx_L, f_axis, *FFT_SINE_BAND)
    sine_R = _band_power(Sxx_R, f_axis, *FFT_SINE_BAND)
    pulse_L = _band_power(Sxx_L, f_axis, *FFT_PULSE_BAND)
    pulse_R = _band_power(Sxx_R, f_axis, *FFT_PULSE_BAND)
    waggle_L = _band_power(Sxx_L, f_axis, *FFT_WAGGLE_BAND)
    waggle_R = _band_power(Sxx_R, f_axis, *FFT_WAGGLE_BAND)

    peak_f_L = _peak_freq(Sxx_L, f_axis, *FFT_SONG_BAND)
    peak_f_R = _peak_freq(Sxx_R, f_axis, *FFT_SONG_BAND)

    # --- Derived per-window features ---
    dominant_song = np.maximum(song_L, song_R)
    dominant_side = np.where(song_R >= song_L, 'R', 'L')
    dominant_pulse = np.where(song_R >= song_L, pulse_R, pulse_L)
    dominant_peak_f = np.where(song_R >= song_L, peak_f_R, peak_f_L)

    # Pulse frequency ratio
    with np.errstate(divide='ignore', invalid='ignore'):
        pulse_freq_ratio = np.where(dominant_song > 0,
                                     dominant_pulse / dominant_song, 0.0)

    # Waggle: bilateral ratio and cross-spectral phase
    max_waggle = np.maximum(waggle_L, waggle_R)
    min_waggle = np.minimum(waggle_L, waggle_R)
    with np.errstate(divide='ignore', invalid='ignore'):
        bilateral_ratio = np.where(max_waggle > 0, min_waggle / max_waggle, 0.0)

    # Cross-spectral phase for waggling detection
    z_L_mean = (_interp_nan(wing_data['WingL_V12']['z'][start:end+1]) +
                _interp_nan(wing_data['WingL_V13']['z'][start:end+1])) / 2
    z_R_mean = (_interp_nan(wing_data['WingR_V12']['z'][start:end+1]) +
                _interp_nan(wing_data['WingR_V13']['z'][start:end+1])) / 2

    f_csd, Pxy = csd(z_L_mean, z_R_mean, fs=fps, nperseg=nperseg,
                      noverlap=noverlap)
    csd_phase = np.abs(np.angle(Pxy))
    waggle_f_mask = (f_csd >= FFT_WAGGLE_BAND[0]) & (f_csd <= FFT_WAGGLE_BAND[1])
    if waggle_f_mask.sum() > 0:
        mean_waggle_phase = np.mean(csd_phase[waggle_f_mask])
    else:
        mean_waggle_phase = 0.0

    # --- Per-window classification ---
    # Pulse requires BOTH high peak frequency AND high pulse-band ratio.
    # This prevents low-frequency sine with slightly elevated ratio from
    # being misclassified as pulse.
    window_labels = np.full(n_windows, 'quiet', dtype=object)

    for i in range(n_windows):
        is_singing = dominant_song[i] >= FFT_SONG_POWER_THRESHOLD
        is_waggle = (max_waggle[i] > dominant_song[i] and
                     bilateral_ratio[i] >= FFT_WAGGLE_BILATERAL_MIN and
                     mean_waggle_phase >= FFT_WAGGLE_PHASE_MIN)

        if is_waggle and not is_singing:
            window_labels[i] = 'waggle'
        elif is_singing:
            if (dominant_peak_f[i] >= FFT_PULSE_PEAK_FREQ_MIN and
                    pulse_freq_ratio[i] >= FFT_PULSE_FREQ_RATIO_MIN):
                window_labels[i] = 'pulse'
            else:
                window_labels[i] = 'sine'
        # else: stays 'quiet'

    # --- Map window labels to frame labels ---
    frame_labels = np.full(n_frames, 'quiet', dtype=object)
    window_centers = (t_axis * fps).astype(int)  # in local frame coords

    for i in range(n_windows):
        # Each window covers [center - hop/2, center + hop/2)
        w_start = max(0, window_centers[i] - FFT_HOP // 2)
        w_end = min(n_frames, window_centers[i] + FFT_HOP // 2)
        frame_labels[w_start:w_end] = window_labels[i]

    # Fill any gaps at edges
    if n_windows > 0:
        frame_labels[:max(0, window_centers[0])] = window_labels[0]
        frame_labels[min(n_frames, window_centers[-1]):] = window_labels[-1]

    # Merge short segments
    frame_labels = _merge_short_segments(frame_labels, FFT_MIN_SEGMENT_FRAMES)

    window_features = {
        'window_centers': window_centers + start,  # absolute frame coords
        'song_power_L': song_L, 'song_power_R': song_R,
        'dominant_song_power': dominant_song,
        'dominant_side': dominant_side,
        'peak_freq': dominant_peak_f,
        'pulse_freq_ratio': pulse_freq_ratio,
        'waggle_power': max_waggle,
        'bilateral_ratio': bilateral_ratio,
    }

    return frame_labels, window_features


def reclassify_bouts_with_fft(bouts, wing_data, fps=FPS, verbose=True):
    """
    Re-classify segments within each courtship bout using FFT spectral analysis.

    Replaces the speed-based pulse/sine classification with frequency-domain
    classification that detects pulse, sine, and waggle modes from wing Z spectra.

    Args:
        bouts: list of bout_info dicts from detect_courtship_bouts()
        wing_data: dict mapping tip name -> {'z': array, ...}
        fps: sampling rate
        verbose: print classification summary

    Returns:
        bouts: same list, with updated segments/pct_pulse/pct_sine/pct_waggle
    """
    if verbose:
        print(f"\n{'='*70}")
        print("FFT-BASED SONG MODE RECLASSIFICATION")
        print(f"{'='*70}")
        print(f"  Song band: {FFT_SONG_BAND[0]}-{FFT_SONG_BAND[1]} Hz")
        print(f"  Pulse band: {FFT_PULSE_BAND[0]}-{FFT_PULSE_BAND[1]} Hz")
        print(f"  Sine band: {FFT_SINE_BAND[0]}-{FFT_SINE_BAND[1]} Hz")
        print(f"  Waggle band: {FFT_WAGGLE_BAND[0]}-{FFT_WAGGLE_BAND[1]} Hz")
        print(f"  Power threshold: {FFT_SONG_POWER_THRESHOLD:.1e}")
        print(f"  Pulse requires: peak_freq >= {FFT_PULSE_PEAK_FREQ_MIN} Hz "
              f"AND ratio >= {FFT_PULSE_FREQ_RATIO_MIN}")
        print(f"  Window: {FFT_NPERSEG} frames ({FFT_NPERSEG/fps*1000:.0f}ms), "
              f"hop: {FFT_HOP} frames ({FFT_HOP/fps*1000:.0f}ms)")

    for bout in bouts:
        start, end = bout['start'], bout['end']
        n_frames = end - start + 1

        if n_frames < FFT_NPERSEG // 2:
            # Too short for FFT — keep existing classification
            bout.setdefault('pct_waggle', 0.0)
            bout.setdefault('fft_classified', False)
            continue

        frame_labels, wf = classify_song_modes_fft(wing_data, start, end, fps)

        # Build segments from contiguous label runs
        segments = []
        current_type = frame_labels[0]
        seg_start_local = 0

        for i in range(1, len(frame_labels)):
            if frame_labels[i] != current_type:
                seg_n = i - seg_start_local
                segments.append({
                    'start': start + seg_start_local,
                    'end': start + i - 1,
                    'type': current_type,
                    'n_frames': seg_n,
                    'duration_s': seg_n / fps,
                })
                current_type = frame_labels[i]
                seg_start_local = i

        # Last segment
        seg_n = n_frames - seg_start_local
        segments.append({
            'start': start + seg_start_local,
            'end': end,
            'type': current_type,
            'n_frames': seg_n,
            'duration_s': seg_n / fps,
        })

        # Filter out quiet segments (keep only song modes)
        song_segments = [s for s in segments if s['type'] != 'quiet']

        # If FFT found no song activity, keep the full bout as a single segment
        # with the dominant spectral mode
        if not song_segments:
            dom_power = np.max(wf['dominant_song_power']) if len(wf['dominant_song_power']) > 0 else 0
            fallback_type = 'sine'  # default if power is ambiguous
            if dom_power >= FFT_SONG_POWER_THRESHOLD:
                peak_ratio = np.median(wf['pulse_freq_ratio'])
                peak_f = np.median(wf['peak_freq'])
                if (peak_f >= FFT_PULSE_PEAK_FREQ_MIN and
                        peak_ratio >= FFT_PULSE_FREQ_RATIO_MIN):
                    fallback_type = 'pulse'
            song_segments = [{
                'start': start, 'end': end, 'type': fallback_type,
                'n_frames': n_frames, 'duration_s': n_frames / fps,
            }]

        bout['segments'] = song_segments
        bout['fft_classified'] = True

        # Update percentages
        total = sum(s['n_frames'] for s in song_segments)
        n_pulse = sum(s['n_frames'] for s in song_segments if s['type'] == 'pulse')
        n_sine = sum(s['n_frames'] for s in song_segments if s['type'] == 'sine')
        n_waggle = sum(s['n_frames'] for s in song_segments if s['type'] == 'waggle')

        bout['n_pulse_frames'] = n_pulse
        bout['n_sine_frames'] = n_sine
        bout['n_waggle_frames'] = n_waggle
        bout['pct_pulse'] = 100 * n_pulse / total if total > 0 else 0
        bout['pct_sine'] = 100 * n_sine / total if total > 0 else 0
        bout['pct_waggle'] = 100 * n_waggle / total if total > 0 else 0

        # Store dominant side
        if len(wf['dominant_side']) > 0:
            from collections import Counter
            side_counts = Counter(wf['dominant_side'])
            bout['dominant_wing'] = side_counts.most_common(1)[0][0]

    if verbose:
        print(f"\n--- FFT reclassification summary ---")
        for bout in bouts[:10]:
            if not bout.get('fft_classified', False):
                print(f"  Bout {bout['bout_idx']}: too short for FFT")
                continue
            seg_str = " -> ".join(
                f"{s['type']}({s['n_frames']})" for s in bout['segments'])
            dom = bout.get('dominant_wing', '?')
            print(f"  Bout {bout['bout_idx']}: frames {bout['start']}-{bout['end']} "
                  f"({bout['n_frames']} frames, {bout['duration_s']:.3f}s)")
            print(f"    Dominant wing: {dom} | "
                  f"pulse={bout['pct_pulse']:.0f}% sine={bout['pct_sine']:.0f}% "
                  f"waggle={bout.get('pct_waggle', 0):.0f}%")
            print(f"    Segments: {seg_str}")
        if len(bouts) > 10:
            print(f"  ... and {len(bouts) - 10} more")

    return bouts


print("FFT-based song mode classification defined.")
print(f"  Song: {FFT_SONG_BAND[0]}-{FFT_SONG_BAND[1]} Hz (threshold {FFT_SONG_POWER_THRESHOLD:.1e})")
print(f"  Pulse: peak >= {FFT_PULSE_PEAK_FREQ_MIN} Hz AND ratio >= {FFT_PULSE_FREQ_RATIO_MIN}")
print(f"  Sine: {FFT_SINE_BAND[0]}-{FFT_SINE_BAND[1]} Hz | "
      f"Waggle: {FFT_WAGGLE_BAND[0]}-{FFT_WAGGLE_BAND[1]} Hz")

In [ ]:
# ============================================================================
# RUN COURTSHIP BOUT DETECTION
# ============================================================================

# ── Config ───────────────────────────────────────────────────────────────────
FRAME_OFFSET      = 0       # add to all frame indices for absolute frame labels
MAX_BOUTS_TO_PLOT = None    # None = plot all detected bouts
SAVE_FIGS         = True   # save each bout as a PDF
SAVE_SUMMARY_CSV  = True    # save courtship_bouts_summary.csv
SAVE_DIR          = JARVIS_PREDICT_DIR  # where to save PDFs/CSV

# ── Step 1: Detect courtship bouts ───────────────────────────────────────────
(court_bouts,
 court_leg_data, court_scut_data, court_wing_data,
 court_activities, court_filter_masks,
 court_diagnostics, court_abd_data) = detect_courtship_bouts(df_full)

if not court_bouts:
    print("\nNo courtship bouts detected. Try lowering COURT_WING_THRESHOLD "
          "or COURT_CONF_THRESHOLD in the config cell above.")
else:
    # ── Step 2: Reclassify with FFT ───────────────────────────────────────────
    court_bouts = reclassify_bouts_with_fft(court_bouts, court_wing_data)

    # ── Summary table ─────────────────────────────────────────────────────────
    print(f"\n{'─'*70}")
    print(f"  {'#':>3}  {'Start':>8}  {'End':>8}  {'Dur(s)':>7}  "
          f"{'Pulse%':>7}  {'Sine%':>6}  {'Waggle%':>8}  {'Wing':>5}  {'Speed':>8}")
    print(f"{'─'*70}")
    for b in court_bouts:
        print(f"  {b['bout_idx']:>3}  {b['start']+FRAME_OFFSET:>8}  "
              f"{b['end']+FRAME_OFFSET:>8}  {b['duration_s']:>7.3f}  "
              f"{b['pct_pulse']:>7.1f}  {b['pct_sine']:>6.1f}  "
              f"{b.get('pct_waggle',0):>8.1f}  "
              f"{b.get('dominant_wing','?'):>5}  "
              f"{b['mean_speed_mm_s']:>7.1f}")
    print(f"{'─'*70}")
    print(f"  Total: {len(court_bouts)} bouts")

    # ── Save CSV summary ──────────────────────────────────────────────────────
    if SAVE_SUMMARY_CSV:
        import pandas as _pd
        rows = []
        for b in court_bouts:
            rows.append({
                'fly_id':            fly_id,
                'bout_idx':          b['bout_idx'],
                'start_frame':       b['start'] + FRAME_OFFSET,
                'end_frame':         b['end']   + FRAME_OFFSET,
                'n_frames':          b['n_frames'],
                'duration_s':        b['duration_s'],
                'mean_speed_mm_s':   b['mean_speed_mm_s'],
                'total_distance_mm': b.get('total_distance_mm', float('nan')),
                'pct_pulse':         b['pct_pulse'],
                'pct_sine':          b['pct_sine'],
                'pct_waggle':        b.get('pct_waggle', 0.0),
                'dominant_wing':     b.get('dominant_wing', ''),
                'start_reason':      (b.get('boundary_diagnostics') or {}).get('primary_cause_before', ''),
                'end_reason':        (b.get('boundary_diagnostics') or {}).get('primary_cause_after', ''),
            })
        court_summary_df = _pd.DataFrame(rows)
        csv_path = SAVE_DIR / 'courtship_bouts_summary.csv'
        court_summary_df.to_csv(csv_path, index=False)
        print(f"\nSummary saved: {csv_path}")

    # ── Step 3: Plot each bout ─────────────────────────────────────────────────
    bouts_to_show = court_bouts if MAX_BOUTS_TO_PLOT is None else court_bouts[:MAX_BOUTS_TO_PLOT]
    for b in bouts_to_show:
        save_path = None
        if SAVE_FIGS:
            save_path = SAVE_DIR / f"courtship_bout_{b['bout_idx']:03d}.pdf"
        plot_courtship_bout_figure(
            b,
            court_leg_data, court_scut_data, court_wing_data, court_activities,
            court_filter_masks,
            frame_offset=FRAME_OFFSET,
            fps=FPS,
            save_path=save_path,
            abd_data=court_abd_data,
            window_features=b.get('window_features'),
        )
        plt.show()


# Grooming Bout Detection

Detect anterior and posterior grooming bouts based on:
- **Anterior**: Both T1L and T1R lifted, close to each other and to antenna/eyes. T2+T3 grounded. Stationary. Upright.
- **Posterior**: Both T3L and T3R lifted, close to each other and to abdomen tip. T1+T2 grounded. Stationary. Upright.

Confidence filter excludes the lifted legs (poorly tracked during grooming). Set `GROOM_TYPE` to `"anterior"` or `"posterior"` in the config cell.

# PCA Things

environment settings

In [ ]:
import sys
from pathlib import Path
import seaborn as sns


# Add project root to Python path FIRST to ensure our modules take priority
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
    print(f"Added {project_root} to Python path")
elif sys.path.index(str(project_root)) != 0:
    # Move to front if it exists but isn't first
    sys.path.remove(str(project_root))
    sys.path.insert(0, str(project_root))
    print(f"Moved {project_root} to front of Python path")

import os
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ['MUJOCO_GL'] = 'egl'
os.environ['PYOPENGL_PLATFORM'] = 'egl'
os.environ["XLA_FLAGS"] = "--xla_gpu_triton_gemm_any=True"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # Use GPU 0

# JAX setup
import jax
jax.print_environment_info()

jax.config.update("jax_compilation_cache_dir", "/tmp/jax_cache")
jax.config.update("jax_persistent_cache_min_entry_size_bytes", -1)
jax.config.update("jax_persistent_cache_min_compile_time_secs", 0)
# Note: jax_persistent_cache_enable_xla_caches may not be available in all JAX versions
try:
    jax.config.update("jax_persistent_cache_enable_xla_caches", "xla_gpu_per_fusion_autotune_cache_dir")
except AttributeError:
    pass  # Skip if not available in this JAX version
jax.config.update("jax_default_matmul_precision", "high")
# Matplotlib setups
import matplotlib as mpl
mpl.rcParams.update({
    'font.size': 10,
    'axes.linewidth': 2,
    'xtick.major.size': 5,
    'ytick.major.size': 5,
    'xtick.major.width': 2,
    'ytick.major.width': 2,
    'axes.spines.right': False,
    'axes.spines.top': False,
    'pdf.fonttype': 42,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'figure.facecolor': 'white',
    'pdf.use14corefonts': True,
    'svg.fonttype': 'none',
    'font.family': 'sans-serif',
    'font.serif': 'Arial',
})

import numpy as np
import matplotlib.pyplot as plt
from typing import Dict, List, Union 
import jax.numpy as jnp

tool to print structure of bout dictionaries


In [ ]:

def print_bout_dict_structure(
    bout_dict: Dict[str, Dict[str, Union[np.ndarray, jnp.ndarray, List[str]]]],
    max_depth: int = 2,
    show_values: bool = False
) -> None:
    """
    Print the structure, shapes, and types of data in a nested bout dictionary.
    
    Displays a hierarchical view of the dictionary structure with array shapes,
    data types, and optionally list/string values for metadata.
    
    Args:
        bout_dict: Nested dictionary with bout-based structure (output of reorganize_stac_by_bouts).
        max_depth: Maximum nesting depth to print (default: 2).
        show_values: If True, show first few elements of lists and strings (default: False).
    
    Examples:
        >>> bout_dict = reorganize_stac_by_bouts(stac_data, [100, 150])
        >>> print_bout_dict_structure(bout_dict)
        
        >>> # Show with metadata values
        >>> print_bout_dict_structure(bout_dict, show_values=True)
    """
    def _format_value(value, indent=""):
        """Format value based on type."""
        if isinstance(value, (np.ndarray, jnp.ndarray)):
            array_type = "jax" if isinstance(value, jnp.ndarray) else "numpy"
            return f"<{array_type} array: shape={value.shape}, dtype={value.dtype}>"
        elif isinstance(value, list):
            if len(value) == 0:
                return "[]"
            elif show_values:
                if len(value) <= 3:
                    return f"{value}"
                else:
                    return f"[{value[0]}, {value[1]}, ..., {value[-1]}] (len={len(value)})"
            else:
                return f"<list: len={len(value)}>"
        elif isinstance(value, str):
            if show_values:
                return f"'{value}'"
            else:
                return f"<str: len={len(value)}>"
        elif isinstance(value, dict):
            return f"<dict: {len(value)} keys>"
        elif isinstance(value, (int, float)):
            return f"{value}"
        else:
            return f"<{type(value).__name__}>"
    
    def _print_dict(d, prefix="", depth=0):
        """Recursively print dictionary structure."""
        if depth > max_depth:
            return
        
        keys = list(d.keys())
        for i, key in enumerate(keys):
            is_last = (i == len(keys) - 1)
            connector = "└── " if is_last else "├── "
            extension = "    " if is_last else "│   "
            
            value = d[key]
            value_str = _format_value(value)
            
            print(f"{prefix}{connector}{key}: {value_str}")
            
            # Recursively print nested dicts
            if isinstance(value, dict) and depth < max_depth:
                _print_dict(value, prefix + extension, depth + 1)
    
    # Print header
    print("=" * 80)
    print("BOUT DICTIONARY STRUCTURE")
    print("=" * 80)
    
    # Count bouts and info
    n_bouts = len([k for k in bout_dict.keys() if k != 'info'])
    has_info = 'info' in bout_dict
    
    print(f"Total keys: {len(bout_dict)} ({'info' + ' + ' if has_info else ''}{n_bouts} bouts)")
    print()
    
    # Print structure
    _print_dict(bout_dict)
    
    # Print summary statistics
    print()
    print("=" * 80)
    print("SUMMARY")
    print("=" * 80)
    
    # Info summary
    if 'info' in bout_dict:
        print(f"📋 Metadata ('info' key): {len(bout_dict['info'])} items")
        for key in bout_dict['info'].keys():
            value_str = _format_value(bout_dict['info'][key])
            print(f"   - {key}: {value_str}")
        print()
    
    # Bout summaries
    bout_keys = sorted([k for k in bout_dict.keys() if k != 'info'])
    if bout_keys:
        print(f"🎬 Bouts: {len(bout_keys)} total")
        for bout_name in bout_keys:
            bout_data = bout_dict[bout_name]
            
            # Get frame count from first array
            n_frames = None
            for value in bout_data.values():
                if isinstance(value, (np.ndarray, jnp.ndarray)) and len(value.shape) > 0:
                    n_frames = value.shape[0]
                    break
            
            # Count data types
            array_keys = [k for k, v in bout_data.items() 
                         if isinstance(v, (np.ndarray, jnp.ndarray))]
            other_keys = [k for k in bout_data.keys() if k not in array_keys]
            
            frame_str = f"{n_frames} frames" if n_frames else "no frames"
            print(f"   - {bout_name}: {frame_str}, {len(array_keys)} arrays, {len(other_keys)} other")
    
    print("=" * 80)

In [ ]:

import utils.io_dict_to_hdf5 as ioh5

df_ik = ioh5.load('/home/user/src/JARVIS-HybridNet/projects/fly50_V5/predictions/predictions3D/Data_analysis/Testing/ik_output_combined_v1.h5')

In [ ]:
df_ik['info']['names_xpos']


## Phase space


In [ ]:
##extract out z components of the end effectors for bout 047 (a long one)

def find_idx(substring:List[str], key, info):
    """"Find indices and names in info[key] that contain any of the substrings."""

    idxs = np.asarray([idx for idx, name in enumerate(info[key]) if any(sub in name for sub in substring)])
    names = np.asarray([name for idx, name in enumerate(info[key]) if any(sub in name for sub in substring)])

    return idxs, names

# df_ik['info']['site_names_egocentric']
N = 5
substring =  ['Tip']
end_eff_idx, end_eff_names = find_idx(substring,'site_names_egocentric', df_ik['info'])
bout_047 = df_ik['bout_047']['xpos_egocentric'][:,end_eff_idx,2]

bouts_EFZ = {key: df_ik[key]['xpos_egocentric'][:,end_eff_idx,2] for key in df_ik if key != 'info'} #only end effectors Z values for every bout, made into a dictionary
bouts_vel = {key: np.diff(bouts_EFZ[key], axis=0, prepend=bouts_EFZ[key][0:1]) for key in bouts_EFZ} #only end effectors Z velocity for every bout, made into a dictionary
bouts_smooth = {key: np.apply_along_axis(lambda m: np.convolve(m, np.ones(N), mode='same'), axis=0, arr=bouts_EFZ[key]) for key in bouts_EFZ} #end effector Z smoothed 
bouts_phase1 = {key: np.angle(scipy.signal.hilbert(bouts_smooth[key],axis=0)) for key in bouts_smooth} #phase for end effector Z 

bouts_crossings = {key: neg_to_pos_crossings(bouts_phase1[key][:,0]) for key in bouts_phase1} #crossings for T1L


In [ ]:
end_eff_names

In [ ]:
plt.plot(bouts_phase1['bout_047'][:,0])
plt.plot(bouts_smooth['bout_047'][:,0])
def neg_to_pos_crossings(phase):
    """Find indices where phase crosses from negative to positive."""
    crossings = np.where((phase[:-1] < 0) & (phase[1:] >= 0))[0]
    return crossings
bouts_crossings = {key: neg_to_pos_crossings(bouts_phase1[key][:,0]) for key in bouts_phase1}
plt.plot(bouts_crossings['bout_047'], bouts_phase1['bout_047'][bouts_crossings['bout_047'],0], 'ro')

plt.figure()
# resample data to uniform phase grid
phase_samp = np.linspace(0, 2*np.pi, 100)
for n in range(len(bouts_crossings['bout_047'])-1):
    start = bouts_crossings['bout_047'][n]
    end = bouts_crossings['bout_047'][n+1]
    phase2 =  np.linspace(0, 2*np.pi, end-start)
    data = bouts_smooth['bout_047'][start:end,0]
    curr_phase = phase2#[clip_idx,start:end,endeff_idxs_all[endeff_idx]]
    resamped_trace = np.interp(phase_samp, curr_phase, data, period=2*np.pi)
    plt.plot(phase_samp,resamped_trace)

bouts_resampled = {}
for bout in bouts_smooth:
    crossings = bouts_crossings[bout]
    n_strides = len(crossings) - 1
    n_tips = bouts_smooth[bout].shape[1]  # adjust if 3D array
    bout_resampled = np.zeros((n_strides, len(phase_samp), n_tips))
    for n in range(n_strides):
        start = crossings[n]
        end = crossings[n + 1]
        phase = np.linspace(0, 2 * np.pi, end - start)
        for tip_idx in range(n_tips):
            data = bouts_smooth[bout][start:end, tip_idx]
            bout_resampled[n, :, tip_idx] = np.interp(
                phase_samp, phase, data, period=2 * np.pi
            )
    bouts_resampled[bout] = bout_resampled

bout_key  = "bout_045"
bout_data = bouts_resampled[bout_key]   # (n_strides, 100, n_tips)
n_strides, n_phase, n_tips = bout_data.shape
fig, axes = plt.subplots(n_tips, 1, figsize=(8, 2 * n_tips), sharex=True)
if n_tips == 1:
    axes = [axes]
for tip_idx in range(n_tips):
    ax = axes[tip_idx]
    # All strides overlaid (grey, thin) — exactly what the original loop does
    for stride in range(n_strides):
        ax.plot(phase_samp, bout_data[stride, :, tip_idx],
                color='gray', lw=0.8, alpha=0.35)
    # Mean across strides (black, thick)
    ax.plot(phase_samp, bout_data[:, :, tip_idx].mean(axis=0),
            color='black', lw=2)
    ax.set_ylabel(f'Leg {tip_idx}')
    ax.set_xticks([0, np.pi/2, np.pi, 3*np.pi/2, 2*np.pi])
    ax.set_xticklabels(['0', 'π/2', 'π', '3π/2', '2π'])
axes[-1].set_xlabel('Stride phase (rad)')
fig.suptitle(f'Phase-aligned Z — {bout_key}')
plt.tight_layout()

In [ ]:
N = 5
end_eff_vel = np.diff(bout_047, axis=0, prepend=bout_047[0:1])
end_eff_smooth= np.apply_along_axis(lambda m: np.convolve(m, np.ones(N), mode='same'), axis=0, arr=bout_047) # Smoothing with window size N
phase1 = np.angle(scipy.signal.hilbert(end_eff_smooth,axis=0))

plt.plot(phase1[:,0])
plt.plot(end_eff_smooth[:,0])

def neg_to_pos_crossings(phase):
    """Find indices where phase crosses from negative to positive."""
    crossings = np.where((phase[:-1] < 0) & (phase[1:] >= 0))[0]
    return crossings
crossings = neg_to_pos_crossings(phase1[:,0])
plt.plot(crossings, phase1[crossings,0], 'ro')

plt.figure()
# resample data to uniform phase grid
phase_samp = np.linspace(0, 2*np.pi, 100)
for n in range(len(crossings)-1):
    start = crossings[n]
    end = crossings[n+1]
    phase2 =  np.linspace(0, 2*np.pi, end-start)
    data = end_eff_smooth[start:end,0]
    curr_phase = phase2#[clip_idx,start:end,endeff_idxs_all[endeff_idx]]
    resamped_trace = np.interp(phase_samp, curr_phase, data, period=2*np.pi)
    plt.plot(phase_samp,resamped_trace)

plt.figure()
feti_idx, feti_names = find_idx(['Tip'],'site_names_egocentric', df_ik['info'])
# feti_idx, feti_names = find_idx(['tibia'],'names_qpos', df_ik['info'])

feti_data = df_ik['bout_047']['xpos_egocentric'][:,feti_idx]
# plt.plot(feti_data[:,0,2])
feti_vel =np.diff(feti_data[:,0], axis=0, prepend=feti_data[0:1,0])
feti_vel_smooth = np.apply_along_axis(lambda m: np.convolve(m, np.ones(N), mode='same'), axis=0, arr=feti_vel) # Smoothing with window size N
fig, axs = plt.subplots(1,1)
ax = axs
ax.plot(feti_vel_smooth)
ax2 = ax.twinx()
ax2.plot(bout_047[:,0], color='orange')


In [ ]:

hilbert_transform = np.imag(scipy.signal.hilbert(bout_047[:,0]))
plt.figure(figsize=(12, 6))
plt.plot(bout_047[:,0], label='Z position of claw 1')
plt.plot(hilbert_transform, label='Hilbert Transform (envelope)', alpha=0.7)
plt.title('Z Position of Claw 1 and its Hilbert Transform')

In [ ]:
qpos_all = np.concatenate([df_ik[key]['qpos'] for key in df_ik.keys() if key != 'info'], axis=0)
qvel_smooth = np.concatenate([np.convolve(np.abs(df_ik[key]['qvel'][:,0]), np.ones(10)/10, mode='same') for key in df_ik.keys() if key != 'info'], axis=0)

# Create bout index array (same length as qpos_all)                                                                                                                                                  
bout_keys = [key for key in df_ik.keys() if key != 'info']                                                                                                                                           
bout_idx = np.concatenate([np.full(df_ik[key]['qpos'].shape[0], i) for i, key in enumerate(bout_keys)], axis=0)   
qpos_all.shape

In [ ]:
PCA_substring =  ['coxa', 'femur', 'tibia']
PCA_idxs = np.asarray([idx for idx, name in enumerate(df_ik['info']['names_qpos']) if any(sub in name for sub in PCA_substring)])
# T1R_idxs.shape
# T1R_idxs = np.asarray([idx for idx, name in enumerate(df_ik['names_qpos']) if ('tarsus1' in name) or ('coxa_T1_left' in name)])


In [ ]:
from sklearn.decomposition import PCA
pca = PCA(n_components=3)
qpos_PCA = pca.fit_transform(qpos_all[:, PCA_idxs])
fig = plt.figure(figsize=(20,15))
ax = fig.add_subplot(111, projection='3d')
sc = ax.scatter(qpos_PCA[1:1000,0], qpos_PCA[1:1000,1], qpos_PCA[1:1000,2], c=qvel_smooth[1:1000], cmap='turbo', s=5
                , alpha=0.5)
plt.colorbar(sc, label='Smoothed |qvel|')
ax.set_xlabel('PCA 1')
ax.set_ylabel('PCA 2')
ax.set_zlabel('PCA 3')
ax.set_title('PCA of Leg Joint Angles Colored by Smoothed Velocity') 
ax.view_init(elev=28, azim=200)

In [ ]:
plt.figure(figsize=(5,4))
plt.scatter(qpos_PCA[0:2000,0], qpos_PCA[0:2000,1], c=bout_idx[0:2000], cmap='turbo', s=10)
#plt.plot(pca_df['fnum'].iloc[:205]/0.3, pca_df['PC74'].iloc[:205]
sns.despine()

In [ ]:
n_bouts = min(10, len(bout_keys))  # Show up to 10 bouts                                                                                                                                             
n_cols = 5                                                                                                                                                                                           
n_rows = (n_bouts + n_cols - 1) // n_cols  # Ceiling division                                                                                                                                        
                                                                                                                                                                                                     
fig, axes = plt.subplots(n_rows, n_cols, figsize=(4*n_cols, 4*n_rows))                                                                                                                               
axes = axes.flatten()                                                                                                                                                                                
                                                                                                                                                                                                     
# Track cumulative frame index                                                                                                                                                                       
frame_start = 0                                                                                                                                                                                      
for i, key in enumerate(bout_keys[:n_bouts]):                                                                                                                                                        
    n_frames = df_ik[key]['qpos'].shape[0]                                                                                                                                                           
    frame_end = frame_start + n_frames                                                                                                                                                               
                                                                                                                                                                                                     
    axes[i].scatter(qpos_PCA[frame_start:frame_end, 0],                                                                                                                                              
                    qpos_PCA[frame_start:frame_end, 1],                                                                                                                                              
                    c=np.arange(n_frames), cmap='turbo', s=10, alpha=0.7)                                                                                                                            
    axes[i].set_title(f'{key}')                                                                                                                                                                      
    axes[i].set_xlabel('PC1')                                                                                                                                                                        
    axes[i].set_ylabel('PC2')                                                                                                                                                                        
    sns.despine(ax=axes[i])                                                                                                                                                                          
                                                                                                                                                                                                     
    frame_start = frame_end                                                                                                                                                                          
                                                                                                                                                                                                     
# Hide unused subplots                                                                                                                                                                               
for j in range(n_bouts, len(axes)):                                                                                                                                                                  
    axes[j].axis('off')                                                                                                                                                                              
                                                                                                                                                                                                     
plt.tight_layout()     

In [ ]:
import cupy
import cuml

from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(qpos_all[:, PCA_idxs])
umap = cuml.manifold.UMAP(n_components=6, n_neighbors=15, min_dist=0.01, metric='euclidean', random_state=42)
reduced_data = umap.fit_transform(X_scaled)  # Example with random data

In [ ]:
fig = plt.figure(figsize=(20,25))
ax = fig.add_subplot(111, projection='3d')
sc = ax.scatter(reduced_data[0:1000,0], reduced_data[0:1000,1], reduced_data[0:1000,2], c=qvel_smooth[0:1000], cmap='turbo', s=3, alpha=0.5)
plt.colorbar(sc, label='Smoothed |qvel|')
ax.set_xlabel('UMAP 1')
ax.set_ylabel('UMAP 2')
ax.set_zlabel('UMAP 3')
ax.set_title('UMAP of Leg Joint Angles Colored by Smoothed Velocity') 
ax.view_init(elev=5, azim=30)
plt.show()